## Etapa 01 — Definição do problema preditivo

Treinar um modelo de regressão para estimar o preço anunciado (`price`) de cada anúncio a partir de seus atributos.

- **Base:** `data/raw/listings_details.csv`, snapshot de 06–07/12/2018 (`scrape_id=20181206172549`). A pasta de 16/09/2026 foi removida.
- **Métrica principal:** MAPE em preços válidos e positivos. MAE, RMSE, viés e desempenho por segmento serão métricas de apoio.
- **Preditores candidatos:** tipo e capacidade do imóvel, localização, anfitrião e reputação. A lista final e as regras contra vazamento serão definidas nas etapas seguintes.
- **Limite:** o modelo descreve preços anunciados no snapshot de 2018. Não prevê preços atuais ou de 2027, nem identifica preço ótimo.
- **Pendência:** confirmar a URL de origem e a licença do arquivo histórico.

In [1]:
# Importa conversão de datas para registrar a última modificação dos arquivos.
from datetime import datetime
# Importa Path para montar e validar caminhos do projeto.
from pathlib import Path
# Importa o leitor de CSV para consultar os cabeçalhos das bases.
import csv

# Usa o diretório atual como raiz inicial do projeto.
project_root = Path.cwd()
# Sobe um nível se o diretório atual não for a raiz do projeto.
if not (project_root / "data" / "raw").is_dir():
    # Define a pasta pai como raiz do projeto.
    project_root = project_root.parent

# Monta o caminho da pasta de dados brutos.
raw_dir = project_root / "data" / "raw"
# Inicializa a lista de arquivos e metadados encontrados.
file_inventory = []
# Percorre os itens da pasta em ordem alfabética.
for file_path in sorted(raw_dir.iterdir()):
    # Mantém apenas arquivos e ignora o marcador .gitkeep.
    if file_path.is_file() and file_path.name != ".gitkeep":
        # Converte a data de modificação do arquivo para texto legível.
        modified_at = datetime.fromtimestamp(file_path.stat().st_mtime).isoformat(timespec="seconds")
        # Guarda o nome, o tamanho em bytes e a data de modificação.
        file_inventory.append((file_path.name, file_path.stat().st_size, modified_at))

# Exibe a raiz de projeto identificada.
print("Diretório do projeto:", project_root)
# Informa que a próxima listagem contém os arquivos brutos.
print("Arquivos encontrados em data/raw:")
# Exibe cada arquivo com tamanho e data de modificação.
for file_name, file_size, modified_at in file_inventory:
    # Formata os metadados de um arquivo por linha.
    print(f"{file_name} | {file_size} bytes | modificado em {modified_at}")

# Registra variantes de nomes de arquivos CSV que serão verificadas.
schema_candidates = [
    "listings_details.csv",
    "calendar.csv",
    "reviews.csv",
    "reviews_details.csv",
    "neighbourhoods.csv",
    "neighbouhood.csv"
]
# Confere cada arquivo candidato antes de ler seu esquema.
for file_name in schema_candidates:
    # Monta o caminho completo do arquivo candidato.
    file_path = raw_dir / file_name
    # Continua apenas se o arquivo existir e tiver extensão CSV.
    if file_path.exists() and file_path.suffix.lower() == ".csv":
        # Abre o CSV preservando a leitura correta do cabeçalho.
        with file_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
            # Lê a primeira linha como nomes de colunas.
            column_names = next(csv.reader(csv_file), [])
        # Mostra o nome do arquivo e a quantidade de colunas.
        print(f"\n{file_name}: {len(column_names)} colunas")
        # Exibe os nomes das colunas encontrados.
        print(column_names)

Diretório do projeto: D:\repos\projetos\airbnb_price
Arquivos encontrados em data/raw:
calendar.csv | 173834328 bytes | modificado em 2019-10-15T03:09:38
listings.csv | 3219978 bytes | modificado em 2019-10-15T03:09:52
listings_details.csv | 78150593 bytes | modificado em 2019-10-15T03:09:52
neighbourhoods.csv | 420 bytes | modificado em 2019-10-15T03:10:02
neighbourhoods.geojson | 253697 bytes | modificado em 2019-10-15T03:10:02
reviews.csv | 8320428 bytes | modificado em 2019-10-15T03:10:02
reviews_details.csv | 151547718 bytes | modificado em 2019-10-15T03:10:02

listings_details.csv: 96 colunas
['id', 'listing_url', 'scrape_id', 'last_scraped', 'name', 'summary', 'space', 'description', 'experiences_offered', 'neighborhood_overview', 'notes', 'transit', 'access', 'interaction', 'house_rules', 'thumbnail_url', 'medium_url', 'picture_url', 'xl_picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'ho

#### Interpretação da auditoria inicial

`listings_details.csv` é a base principal: 20.030 anúncios, 96 colunas e IDs únicos. `listings.csv` é resumido e não será usado. O snapshot tem `scrape_id=20181206172549` e datas entre 06 e 07/12/2018.

Os campos de preço e os candidatos a preditores foram confirmados no arquivo. Atributos de reputação, disponibilidade e datas ainda precisam passar por verificação de vazamento. `availability_365` indica noites disponíveis, não noites vendidas.

As análises refletem o padrão histórico de 2018. A URL e a licença do arquivo ainda precisam ser confirmadas.

In [2]:
# Importa contador para resumir as frequências dos campos.
from collections import Counter
# Importa date para validar e comparar datas de coleta.
from datetime import date
# Importa o leitor de CSV para percorrer os anúncios.
import csv

# Define o caminho do arquivo principal de anúncios.
listings_path = raw_dir / "listings_details.csv"
# Guarda IDs já vistos para contar identificadores únicos e repetidos.
listing_ids = set()
# Prepara a contagem das versões de coleta.
scrape_id_counts = Counter()
# Prepara a contagem por tipo de quarto.
room_type_counts = Counter()
# Prepara a contagem por tipo de propriedade.
property_type_counts = Counter()
# Armazena datas válidas de coleta para obter o intervalo observado.
last_scraped_dates = []
# Inicializa o total de linhas lidas.
listing_rows = 0
# Inicializa o total de IDs repetidos.
duplicate_listing_ids = 0

# Abre o arquivo principal para leitura linha a linha.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Usa os nomes do cabeçalho como chaves de cada registro.
    reader = csv.DictReader(csv_file)
    # Percorre cada anúncio do arquivo.
    for row in reader:
        # Atualiza o total de anúncios lidos.
        listing_rows += 1
        # Lê o ID e remove espaços em branco das extremidades.
        listing_id = (row.get("id") or "").strip()
        # Conta o ID quando ele já apareceu em uma linha anterior.
        if listing_id in listing_ids:
            # Soma uma ocorrência duplicada.
            duplicate_listing_ids += 1
        # Adiciona apenas IDs não vazios ao conjunto de identificadores.
        if listing_id:
            # Registra o ID para detectar repetições futuras.
            listing_ids.add(listing_id)
        # Conta o scrape_id, identificando campos vazios explicitamente.
        scrape_id_counts[(row.get("scrape_id") or "").strip() or "(vazio)"] += 1
        # Conta o tipo de quarto, incluindo valores vazios.
        room_type_counts[(row.get("room_type") or "").strip() or "(vazio)"] += 1
        # Conta o tipo de propriedade, incluindo valores vazios.
        property_type_counts[(row.get("property_type") or "").strip() or "(vazio)"] += 1
        # Lê a data de coleta e remove espaços extras.
        scraped_value = (row.get("last_scraped") or "").strip()
        # Tenta converter somente datas não vazias.
        if scraped_value:
            # Interpreta os primeiros dez caracteres como data ISO.
            try:
                # Guarda a data validada para calcular mínimo e máximo.
                last_scraped_dates.append(date.fromisoformat(scraped_value[:10]))
            # Ignora valores que não podem ser interpretados como data.
            except ValueError:
                # Continua a leitura sem interromper a auditoria.
                pass

# Informa o arquivo usado como base da verificação.
print("Arquivo principal:", listings_path)
# Compara o total de linhas com IDs únicos e repetidos.
print("Anúncios:", listing_rows, "| IDs únicos:", len(listing_ids), "| IDs repetidos:", duplicate_listing_ids)
# Exibe as frequências de cada identificador de coleta.
print("scrape_id:", scrape_id_counts.most_common())
# Mostra o intervalo das datas de coleta válidas, se houver datas.
print("last_scraped mínimo/máximo:", min(last_scraped_dates) if last_scraped_dates else None, "/", max(last_scraped_dates) if last_scraped_dates else None)
# Exibe as frequências por tipo de quarto.
print("room_type:", room_type_counts.most_common())
# Exibe os dez tipos de propriedade mais frequentes.
print("property_type (10 mais frequentes):", property_type_counts.most_common(10))

Arquivo principal: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv
Anúncios: 20030 | IDs únicos: 20030 | IDs repetidos: 0
scrape_id: [('20181206172549', 20030)]
last_scraped mínimo/máximo: 2018-12-06 / 2018-12-07
room_type: [('Entire home/apt', 15889), ('Private room', 4076), ('Shared room', 65)]
property_type (10 mais frequentes): [('Apartment', 15582), ('House', 1523), ('Townhouse', 649), ('Bed and breakfast', 455), ('Loft', 384), ('Boat', 372), ('Condominium', 323), ('Houseboat', 225), ('Guest suite', 152), ('Aparthotel', 73)]


### 01.1 — Formulação do problema

`X` reúne atributos disponíveis de cada anúncio; `y = price` é o preço anunciado da diária. Cada linha representa um anúncio do snapshot de dezembro de 2018.

O modelo fará uma regressão transversal. Ele estima preços do snapshot, não a evolução do mercado para 2027 nem preços atuais.

O MAPE será calculado apenas para preços válidos e maiores que zero. Ele mede o erro frente ao preço anunciado; não indica preço ótimo ou efeito causal.

A lista final de atributos e as regras contra vazamento serão definidas nas etapas seguintes. Preços futuros de `calendar.csv` não serão usados.

In [3]:
# Importa Counter para contar frequências de símbolos e categorias.
from collections import Counter
# Importa Decimal para validar preços sem arredondamento binário.
from decimal import Decimal, InvalidOperation
# Importa Path para montar caminhos do projeto.
from pathlib import Path
# Importa csv para percorrer os registros do arquivo.
import csv

# Usa o diretório atual como ponto inicial para localizar o projeto.
project_root = Path.cwd()
# Sobe para a pasta pai quando data/raw não existe no diretório atual.
if not (project_root / "data" / "raw").is_dir():
    # Atualiza a raiz para a pasta pai do diretório atual.
    project_root = project_root.parent
# Monta o caminho da pasta de dados brutos.
raw_dir = project_root / "data" / "raw"
# Monta o caminho do arquivo principal de anúncios.
listings_path = raw_dir / "listings_details.csv"

# Lista os atributos candidatos e permite verificar se existem no esquema.
candidate_features = [
    "room_type", "property_type", "accommodates", "bedrooms", "bathrooms",
    "neighbourhood_cleansed", "latitude", "longitude", "host_is_superhost",
    "host_since", "host_listings_count", "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "minimum_nights", "availability_365", "instant_bookable"
]
# Prepara a contagem dos símbolos encontrados nos preços.
price_symbols = Counter()
# Guarda até cinco exemplos textuais do campo price.
price_examples = []
# Armazena preços positivos que puderam ser convertidos.
valid_prices = []
# Guarda IDs para identificar anúncios únicos e duplicados.
listing_ids = set()
# Inicializa o contador de linhas lidas.
listing_rows = 0
# Inicializa o contador de IDs repetidos.
duplicate_ids = 0
# Inicializa o contador de preços ausentes.
missing_price_rows = 0
# Inicializa o contador de preços que não puderam ser convertidos.
invalid_price_rows = 0
# Inicializa o contador de preços iguais a zero.
zero_price_rows = 0
# Inicializa o contador de preços negativos.
negative_price_rows = 0

# Abre a base principal em modo texto, preservando a codificação e as quebras de linha.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada linha como um dicionário indexado pelo nome da coluna.
    reader = csv.DictReader(csv_file)
    # Obtém os nomes das colunas do cabeçalho, ou uma lista vazia se não houver cabeçalho.
    listing_columns = reader.fieldnames or []
    # Percorre todos os anúncios da base.
    for row in reader:
        # Atualiza a quantidade de linhas processadas.
        listing_rows += 1
        # Lê o ID do anúncio e remove espaços das extremidades.
        listing_id = (row.get("id") or "").strip()
        # Verifica se o ID já apareceu em uma linha anterior.
        if listing_id in listing_ids:
            # Conta o ID como repetido.
            duplicate_ids += 1
        # Confere se o ID está preenchido.
        if listing_id:
            # Registra o ID para as próximas verificações de duplicidade.
            listing_ids.add(listing_id)
        # Lê o preço original e remove espaços das extremidades.
        raw_price = (row.get("price") or "").strip()
        # Identifica preços vazios antes de tentar convertê-los.
        if not raw_price:
            # Conta a linha como preço ausente.
            missing_price_rows += 1
            # Pula para o próximo anúncio, pois não há preço para validar.
            continue
        # Guarda apenas os cinco primeiros exemplos para inspeção.
        if len(price_examples) < 5:
            # Adiciona o texto original do preço à lista de exemplos.
            price_examples.append(raw_price)
        # Identifica valores que contêm o símbolo de dólar.
        if "$" in raw_price:
            # Incrementa a frequência do símbolo de dólar.
            price_symbols["$"] += 1
        # Identifica valores que contêm o símbolo de euro.
        elif "€" in raw_price:
            # Incrementa a frequência do símbolo de euro.
            price_symbols["€"] += 1
        # Identifica valores que contêm o símbolo de libra.
        elif "£" in raw_price:
            # Incrementa a frequência do símbolo de libra.
            price_symbols["£"] += 1
        # Classifica valores que não contêm esses símbolos.
        else:
            # Conta o valor como sem símbolo monetário reconhecido.
            price_symbols["sem símbolo"] += 1
        # Remove símbolos e separadores de milhar antes da conversão numérica.
        normalized_price = raw_price.replace("$", "").replace("€", "").replace("£", "").replace(",", "").strip()
        # Tenta converter o texto normalizado para número decimal.
        try:
            # Converte o preço para Decimal para validar o valor.
            parsed_price = Decimal(normalized_price)
        # Trata valores que não representam um número decimal válido.
        except InvalidOperation:
            # Conta a falha de conversão.
            invalid_price_rows += 1
            # Pula o registro inválido e continua com o próximo.
            continue
        # Identifica preços exatamente iguais a zero.
        if parsed_price == 0:
            # Incrementa o contador de preços zero.
            zero_price_rows += 1
        # Identifica preços abaixo de zero.
        elif parsed_price < 0:
            # Incrementa o contador de preços negativos.
            negative_price_rows += 1
        # Armazena preços positivos que passaram na conversão.
        else:
            # Adiciona o preço válido à lista de valores.
            valid_prices.append(parsed_price)

# Informa o caminho do arquivo inspecionado.
print("Arquivo usado:", listings_path)
# Resume linhas, IDs únicos e IDs repetidos.
print("Linhas:", listing_rows, "| IDs únicos:", len(listing_ids), "| IDs repetidos:", duplicate_ids)
# Verifica a presença de cada atributo candidato no cabeçalho.
print("Campos candidatos presentes:", {name: name in listing_columns for name in candidate_features})
# Exibe os exemplos originais de preço.
print("Exemplos brutos de price:", price_examples)
# Exibe a contagem por símbolo monetário identificado.
print("Símbolos em price:", price_symbols)
# Resume preços ausentes, inválidos, zero e negativos.
print("Preço ausente:", missing_price_rows, "| parsing inválido:", invalid_price_rows, "| zero:", zero_price_rows, "| negativo:", negative_price_rows)
# Mostra quantidade, mínimo e máximo dos preços positivos convertidos.
print("Preços positivos parseados:", len(valid_prices), "| mínimo:", min(valid_prices) if valid_prices else None, "| máximo:", max(valid_prices) if valid_prices else None)

Arquivo usado: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv
Linhas: 20030 | IDs únicos: 20030 | IDs repetidos: 0
Campos candidatos presentes: {'room_type': True, 'property_type': True, 'accommodates': True, 'bedrooms': True, 'bathrooms': True, 'neighbourhood_cleansed': True, 'latitude': True, 'longitude': True, 'host_is_superhost': True, 'host_since': True, 'host_listings_count': True, 'number_of_reviews': True, 'reviews_per_month': True, 'review_scores_rating': True, 'minimum_nights': True, 'availability_365': True, 'instant_bookable': True}
Exemplos brutos de price: ['$59.00', '$160.00', '$80.00', '$125.00', '$150.00']
Símbolos em price: Counter({'$': 20030})
Preço ausente: 0 | parsing inválido: 0 | zero: 2 | negativo: 0
Preços positivos parseados: 20028 | mínimo: 8.00 | máximo: 8500.00


#### Resultado da validação do alvo

Os 17 campos candidatos existem no arquivo, e os 20.030 IDs são únicos. `price` foi parseado em todas as linhas: 20.028 valores são positivos e 2 são zero. Os valores positivos vão de €8 a €8.500. Os dois zeros ficam fora do MAPE; o tratamento será decidido na Etapa 02.

O campo aparece com `$` no arquivo. O [dicionário oficial do Inside Airbnb](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing), versão 4.3 (agosto de 2022), define `price` como preço diário na moeda local e descreve o símbolo como artefato de exportação. O contrato do projeto usa euros. A URL e a licença do arquivo de 2018 seguem pendentes.

A faixa de preços será auditada na Etapa 02 para definir o tratamento de outliers.

### 01.2 — Métrica de avaliação

A métrica principal é o MAPE: `média(|preço real − preço previsto| / preço real) × 100`, calculada somente para preços reais válidos e positivos. O cálculo será feito em dados separados do treino, com a partição definida antes da seleção supervisionada de atributos.

Também serão avaliados MAE, RMSE, viés e resultados por segmento. Preços baixos e valores extremos merecem atenção porque o MAPE dá mais peso relativo aos alvos menores.

O alvo é o preço anunciado, não o preço pago nem a receita. MAPE menor indica previsões mais próximas dos anúncios do snapshot; não comprova preço ótimo ou ganho econômico.

### 01.3 — Critério de aceite

A definição está pronta para aprovação quando registrar alvo, snapshot, preditores candidatos, MAPE e limites de interpretação. A Etapa 01 não treina nem compara modelos. A limpeza começa na Etapa 02.

### 01.4 — Limites da análise

O modelo usa anúncios capturados em dezembro de 2018; os resultados descrevem esse snapshot, não os preços atuais ou de 2027.

O alvo é o preço anunciado. As previsões não demonstram causalidade, preço ótimo, reservas ou receita efetiva. Esta etapa não calcula ocupação, GMV, EV ou ROI.

A auditoria de qualidade, a escolha final de atributos, a partição dos dados e o treinamento serão tratados após seus respectivos gates.

### 01.5 — Resumo e gate

#### Evidências
- Snapshot: 20.030 anúncios com IDs únicos, captura de 06–07/12/2018 (`scrape_id=20181206172549`).
- Os 17 campos candidatos existem. `price` foi parseado em todas as linhas: 20.028 positivos e 2 zeros, com valores positivos de €8 a €8.500.
- O arquivo usa `$` como símbolo. O dicionário consultado o descreve como artefato; URL e licença do snapshot histórico seguem pendentes.

#### Decisões
- Tarefa: regressão supervisionada do preço anunciado, usando atributos do anúncio como preditores.
- MAPE é a métrica principal para preços positivos; MAE, RMSE, viés e segmentos são métricas de apoio.
- Usar o snapshot de 2018. Não tratar o problema como previsão para 2027 nem calcular métricas financeiras nesta etapa.

#### Pendências
- Confirmar a URL de origem e a licença do snapshot.
- Auditar qualidade, outliers e os dois preços zero na Etapa 02. Definir atributos e partição antes da modelagem.

#### Status do projeto
- **Status:** Etapa 01 aprovada pelo usuário; Etapa 02 em andamento.
- **Última etapa aprovada:** Etapa 01.
- **Decisões:** regressão supervisionada do preço anunciado; MAPE principal; snapshot de 2018.
- **Pendências:** procedência do arquivo; auditoria de qualidade e aprovação do gate da Etapa 02.

## Etapa 02 — Data Quality

Auditar a estrutura e a qualidade das bases do snapshot, identificar inconsistências e propor o tratamento. Nesta etapa, os dados brutos permanecem intactos: não haverá exclusão nem imputação antes da aprovação do diagnóstico e do plano.

### 02.0 — Partição inicial e preservação do teste bruto

Dividir todos os anúncios do snapshot por `id` antes das auditorias e da limpeza: 70% para treino, 15% para validação e 15% para teste, com semente 42. Salvar a partição de teste com todas as colunas e valores originais em `data/processed/df_test_raw.csv`. O teste fica fora da limpeza e da tabela de desenvolvimento desta etapa; será preparado somente durante a avaliação, depois de congeladas as decisões do modelo.

In [4]:
# 1. Importa as ferramentas para ler os IDs e preservar os registros brutos.
# Importa csv para validar as chaves antes da partição.
import csv
# Importa random para obter uma divisão reprodutível.
import random
# Importa pandas para guardar o teste bruto em memória e em disco.
import pandas as pd

# 2. Define o arquivo principal e inicializa os controles de leitura dos IDs.
listings_path = raw_dir / "listings_details.csv"
all_listing_ids = []
seen_listing_ids = set()
missing_split_id_rows = 0
duplicate_split_id_rows = 0

# 3. Lê todos os IDs do snapshot antes de qualquer auditoria ou limpeza.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Usa o cabeçalho para localizar a coluna id pelo nome.
    split_reader = csv.DictReader(csv_file)
    # Percorre as linhas originais da base detalhada.
    for split_row in split_reader:
        # Lê o ID como texto e remove espaços das extremidades.
        split_listing_id = (split_row.get("id") or "").strip()
        # Conta e ignora IDs ausentes para não criar uma partição ambígua.
        if not split_listing_id:
            missing_split_id_rows += 1
            continue
        # Conta repetições e mantém apenas a primeira ocorrência para a checagem.
        if split_listing_id in seen_listing_ids:
            duplicate_split_id_rows += 1
            continue
        # Guarda cada ID único encontrado no snapshot.
        seen_listing_ids.add(split_listing_id)
        # Mantém os IDs para ordenar e embaralhar de forma reprodutível.
        all_listing_ids.append(split_listing_id)

# 4. Interrompe a partição se a chave primária não for íntegra.
if missing_split_id_rows or duplicate_split_id_rows:
    # Informa a inconsistência que precisa ser resolvida antes da divisão.
    raise ValueError(
        f"IDs ausentes: {missing_split_id_rows}; "
        f"IDs duplicados: {duplicate_split_id_rows}"
    )

# 5. Embaralha os IDs ordenados com a semente fixa e calcula os tamanhos.
split_seed = 42
all_listing_ids = sorted(all_listing_ids)
split_rng = random.Random(split_seed)
split_rng.shuffle(all_listing_ids)
n_total_original = len(all_listing_ids)
n_train = int(n_total_original * 0.70)
n_validation = int(n_total_original * 0.15)
n_test = n_total_original - n_train - n_validation

# 6. Separa os IDs nos três conjuntos antes de aplicar qualquer filtro.
train_listing_ids = set(all_listing_ids[:n_train])
validation_listing_ids = set(
    all_listing_ids[n_train:n_train + n_validation]
)
test_listing_ids = set(all_listing_ids[n_train + n_validation:])

# 7. Associa cada ID à sua partição para reutilizar os rótulos depois.
partition_by_id = {}
for split_listing_id in train_listing_ids:
    # Registra a partição de treino do anúncio.
    partition_by_id[split_listing_id] = "treino"
for split_listing_id in validation_listing_ids:
    # Registra a partição de validação do anúncio.
    partition_by_id[split_listing_id] = "validação"
for split_listing_id in test_listing_ids:
    # Registra a partição de teste do anúncio.
    partition_by_id[split_listing_id] = "teste"

# 8. Confere que as partições são disjuntas e cobrem todos os IDs originais.
split_overlap = (
    bool(train_listing_ids & validation_listing_ids)
    or bool(train_listing_ids & test_listing_ids)
    or bool(validation_listing_ids & test_listing_ids)
)
split_coverage_ok = (
    train_listing_ids | validation_listing_ids | test_listing_ids
) == set(all_listing_ids)
if split_overlap or not split_coverage_ok:
    # Interrompe se algum ID estiver em mais de um conjunto ou sem partição.
    raise ValueError("A divisão dos IDs não é disjunta ou não cobre o snapshot.")

# 9. Carrega os valores originais como texto para não converter ausências ou preço.
df_snapshot_raw = pd.read_csv(
    listings_path,
    dtype="string",
    keep_default_na=False
)
# Registra o esquema original para conferir que todas as colunas foram preservadas.
df_test_raw_columns = df_snapshot_raw.columns.tolist()
# Extrai apenas os anúncios alocados ao teste, sem aplicar filtros ou conversões.
df_test = df_snapshot_raw.loc[
    df_snapshot_raw["id"].isin(test_listing_ids)
].copy()

# 10. Salva o teste bruto em uma pasta própria de dados processados.
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
df_test_raw_path = processed_dir / "df_test_raw.csv"
df_test.to_csv(df_test_raw_path, index=False, encoding="utf-8")

# 11. Libera a cópia integral do snapshot; mantém df_test sem transformação.
del df_snapshot_raw

# 12. Exibe os tamanhos, a integridade e o caminho do teste bruto preservado.
print("Arquivo particionado:", listings_path)
print("Semente:", split_seed, "| proporções: 70% / 15% / 15%")
print(
    "IDs originais | treino:", n_train,
    "| validação:", n_validation,
    "| teste:", n_test,
    "| total:", n_total_original
)
print(
    "IDs ausentes:", missing_split_id_rows,
    "| IDs duplicados:", duplicate_split_id_rows,
    "| sobreposição:", split_overlap,
    "| cobertura completa:", split_coverage_ok
)
print(
    "df_test bruto:", df_test.shape,
    "| colunas preservadas:", len(df_test_raw_columns),
    "| IDs somente do teste:",
    set(df_test["id"].tolist()) == test_listing_ids
)
print(
    "Arquivo salvo:", df_test_raw_path,
    "| existe:", df_test_raw_path.exists(),
    "| tamanho em bytes:", df_test_raw_path.stat().st_size
)

Arquivo particionado: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv
Semente: 42 | proporções: 70% / 15% / 15%
IDs originais | treino: 14021 | validação: 3004 | teste: 3005 | total: 20030
IDs ausentes: 0 | IDs duplicados: 0 | sobreposição: False | cobertura completa: True
df_test bruto: (3005, 96) | colunas preservadas: 96 | IDs somente do teste: True
Arquivo salvo: D:\repos\projetos\airbnb_price\data\processed\df_test_raw.csv | existe: True | tamanho em bytes: 11634013


### 02.1 — Estrutura, tipos e ausências

Verificar dimensões, colunas, tipos inferidos, chaves, duplicatas e valores ausentes em `listings_details.csv`. Esta auditoria define a base para as verificações de faixas e consistência entre arquivos.

In [5]:
# Importa csv para ler os registros das bases.
import csv
# Importa sys para registrar a versão do Python usada.
import sys
# Importa Counter para resumir contagens e frequências.
from collections import Counter
# Importa date para validar campos de data.
from datetime import date
# Importa Decimal e InvalidOperation para testar conversões numéricas.
from decimal import Decimal, InvalidOperation
# Importa Path para localizar os arquivos do projeto.
from pathlib import Path

# Define o diretório atual como raiz inicial do projeto.
project_root = Path.cwd()
# Usa a pasta pai caso data/raw não esteja no diretório atual.
if not (project_root / "data" / "raw").is_dir():
    # Atualiza a raiz para o diretório pai.
    project_root = project_root.parent
# Monta o caminho da pasta de dados brutos.
raw_dir = project_root / "data" / "raw"
# Monta o caminho da base principal de anúncios.
listings_path = raw_dir / "listings_details.csv"

# Declara atributos cuja existência e qualidade serão auditadas.
candidate_features = [
    "room_type", "property_type", "accommodates", "bedrooms", "bathrooms",
    "neighbourhood_cleansed", "latitude", "longitude", "host_is_superhost",
    "host_since", "host_listings_count", "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "minimum_nights", "availability_365", "instant_bookable"
]
# Identifica colunas que devem ser interpretadas como datas.
date_fields = {"last_scraped", "host_since", "first_review", "last_review", "calendar_last_scraped"}
# Identifica colunas booleanas e os campos com valores booleanos esperados.
boolean_fields = {
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "is_location_exact", "has_availability", "instant_bookable",
    "is_business_travel_ready", "require_guest_profile_picture",
    "require_guest_phone_verification"
}
# Identifica colunas de chave que não serão tratadas como medidas.
identifier_fields = {"id", "host_id"}
# Identifica variáveis categóricas para contar valores distintos.
categorical_fields = {
    "room_type", "property_type", "neighbourhood_cleansed",
    "host_is_superhost", "instant_bookable"
}
# Identifica variáveis numéricas para resumir os intervalos observados.
numeric_candidates = {
    "price", "accommodates", "bedrooms", "bathrooms", "latitude", "longitude",
    "host_listings_count", "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "minimum_nights", "availability_365"
}

# Inicializa o conjunto de IDs de anúncios encontrados.
listing_ids = set()
# Prepara a estrutura que armazenará contagens por tipo inferido.
type_counts = {}
# Inicializa contagens de valores ausentes por coluna.
missing_counts = Counter()
# Prepara uma contagem de categorias para cada variável categórica.
category_counts = {name: Counter() for name in categorical_fields}
# Prepara uma lista de valores numéricos válidos para cada variável numérica.
numeric_values = {name: [] for name in numeric_candidates}
# Inicializa o total de registros lidos.
row_count = 0
# Inicializa a contagem de IDs repetidos.
duplicate_ids = 0
# Inicializa a contagem de IDs vazios.
null_ids = 0

# Abre o arquivo principal para leitura registro a registro.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Interpreta cada registro com acesso às colunas pelo nome.
    reader = csv.DictReader(csv_file)
    # Guarda os nomes de colunas identificados no cabeçalho.
    column_names = reader.fieldnames or []
    # Cria uma contagem de tipos observados para cada coluna.
    type_counts = {name: Counter() for name in column_names}
    # Percorre todos os anúncios do arquivo.
    for row in reader:
        # Atualiza o número de registros processados.
        row_count += 1
        # Lê o ID do anúncio e remove espaços externos.
        listing_id = (row.get("id") or "").strip()
        # Verifica se o identificador está ausente.
        if not listing_id:
            # Conta o registro sem ID.
            null_ids += 1
        # Verifica se o identificador já foi visto.
        elif listing_id in listing_ids:
            # Conta uma ocorrência de ID duplicado.
            duplicate_ids += 1
        # Trata identificadores preenchidos e ainda não vistos.
        else:
            # Registra o ID único no conjunto.
            listing_ids.add(listing_id)

        # Percorre todas as colunas para classificar seus valores.
        for name in column_names:
            # Lê o valor como texto e remove espaços das extremidades.
            value = (row.get(name) or "").strip()
            # Verifica se a célula está vazia.
            if not value:
                # Incrementa a contagem de ausências da coluna.
                missing_counts[name] += 1
                # Registra a categoria de valor ausente na tipagem inferida.
                type_counts[name]["ausente"] += 1
                # Passa à próxima coluna do registro.
                continue

            # Identifica valores em campos que representam chaves.
            if name in identifier_fields:
                # Classifica o valor como identificador.
                type_counts[name]["identificador"] += 1
            # Valida os valores de colunas conhecidas como datas.
            elif name in date_fields:
                # Tenta interpretar o início do texto como data ISO.
                try:
                    # Registra o valor como data quando a conversão funciona.
                    date.fromisoformat(value[:10])
                    type_counts[name]["data"] += 1
                # Captura datas malformadas ou valores textuais inesperados.
                except ValueError:
                    # Classifica o conteúdo como texto ou data inválida.
                    type_counts[name]["texto/data inválida"] += 1
            # Verifica os valores de colunas booleanas.
            elif name in boolean_fields:
                # Aceita apenas representações booleanas conhecidas.
                if value.lower() in {"t", "f", "true", "false", "yes", "no"}:
                    # Classifica o valor como booleano.
                    type_counts[name]["booleano"] += 1
                # Registra valores que não correspondem ao domínio esperado.
                else:
                    # Classifica o conteúdo como texto ou valor inesperado.
                    type_counts[name]["texto/valor inesperado"] += 1
            # Tenta tratar as demais colunas como numéricas ou textuais.
            else:
                # Começa a conversão usando o valor original.
                numeric_text = value
                # Remove símbolo monetário da coluna de preço, se aplicável.
                if name == "price":
                    # Mantém apenas o conteúdo numérico do preço.
                    numeric_text = value.replace("$", "").replace("€", "").replace("£", "").strip()
                # Remove separadores de milhar antes da conversão.
                numeric_text = numeric_text.replace(",", "").strip()
                # Tenta interpretar o valor limpo como número.
                try:
                    # Converte o conteúdo para Decimal.
                    parsed_value = Decimal(numeric_text)
                    # Conta o valor como numérico.
                    type_counts[name]["numérico"] += 1
                    # Guarda o valor quando a coluna está entre os candidatos numéricos.
                    if name in numeric_values:
                        # Adiciona o valor validado à lista da coluna.
                        numeric_values[name].append(parsed_value)
                # Trata conteúdos que não podem ser convertidos para número.
                except InvalidOperation:
                    # Classifica o valor não numérico como texto.
                    type_counts[name]["texto"] += 1

            # Atualiza frequências para os campos categóricos selecionados.
            if name in category_counts:
                # Conta a ocorrência do valor da categoria.
                category_counts[name][value] += 1

# Ordena as colunas pela quantidade de ausências, da maior para a menor.
missing_columns = [(name, count) for name, count in missing_counts.most_common()]

# Registra a versão do interpretador Python.
print("Python:", sys.version.split()[0])
# Informa o arquivo submetido à auditoria.
print("Arquivo:", listings_path)
# Resume as dimensões da base.
print("Dimensões:", row_count, "linhas x", len(column_names), "colunas")
# Resume IDs únicos, ausentes e duplicados.
print("IDs únicos:", len(listing_ids), "| IDs nulos:", null_ids, "| IDs repetidos:", duplicate_ids)
# Lista os candidatos que não aparecem no esquema.
print("Campos candidatos ausentes no esquema:", [name for name in candidate_features if name not in column_names])
# Introduz a listagem de tipos inferidos dos candidatos.
print("\nTipos inferidos dos campos candidatos:")
# Percorre os atributos candidatos para mostrar suas classificações.
for name in candidate_features:
    # Exibe as contagens de tipos para o campo, ou um dicionário vazio se não houver.
    print(name, "|", dict(type_counts.get(name, {})))
# Introduz a listagem de cardinalidade das variáveis categóricas.
print("\nCardinalidade dos campos categóricos candidatos:")
# Percorre as categorias em ordem alfabética para facilitar a leitura.
for name in sorted(category_counts):
    # Mostra quantidade de valores distintos e as dez frequências mais altas.
    print(name, "| distintos:", len(category_counts[name]), "| frequências:", category_counts[name].most_common(10))
# Introduz a listagem de intervalos das variáveis numéricas.
print("\nFaixas observadas dos campos numéricos candidatos:")
# Percorre as variáveis numéricas em ordem alfabética.
for name in sorted(numeric_values):
    # Obtém os valores numéricos válidos da variável atual.
    values = numeric_values[name]
    # Mostra quantidade, mínimo e máximo, tratando listas vazias.
    print(name, "| valores válidos:", len(values), "| mínimo:", min(values) if values else None, "| máximo:", max(values) if values else None)
# Mostra quantas colunas apresentam valores ausentes.
print("\nColunas com ausências:", len(missing_columns))
# Percorre as colunas ordenadas pela quantidade de ausências.
for name, count in missing_columns:
    # Exibe contagem e percentual ausente em relação ao total de registros.
    print(name, "| ausências:", count, "| percentual:", round(count / row_count * 100, 2))

Python: 3.10.4
Arquivo: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv
Dimensões: 20030 linhas x 96 colunas
IDs únicos: 20030 | IDs nulos: 0 | IDs repetidos: 0
Campos candidatos ausentes no esquema: []

Tipos inferidos dos campos candidatos:
room_type | {'texto': 20030}
property_type | {'texto': 20030}
accommodates | {'numérico': 20030}
bedrooms | {'numérico': 20022, 'ausente': 8}
bathrooms | {'numérico': 20020, 'ausente': 10}
neighbourhood_cleansed | {'texto': 20030}
latitude | {'numérico': 20030}
longitude | {'numérico': 20030}
host_is_superhost | {'booleano': 20026, 'ausente': 4}
host_since | {'data': 20026, 'ausente': 4}
host_listings_count | {'numérico': 20026, 'ausente': 4}
number_of_reviews | {'numérico': 20030}
reviews_per_month | {'numérico': 17624, 'ausente': 2406}
review_scores_rating | {'numérico': 17391, 'ausente': 2639}
minimum_nights | {'numérico': 20030}
availability_365 | {'numérico': 20030}
instant_bookable | {'booleano': 20030}

Cardinalidade dos campos

#### Interpretação da verificação 02.1

A base tem 20.030 anúncios e 96 colunas, sem IDs nulos ou repetidos. Todos os 17 campos candidatos existem. Há ausências em 54 colunas; entre as candidatas, faltam 8 valores de `bedrooms`, 10 de `bathrooms`, 4 em campos do anfitrião, 2.406 de `reviews_per_month` e 2.639 de `review_scores_rating`.

`review_scores_rating` vai de 20 a 100. `bathrooms` chega a 100,5 e `minimum_nights` a 1.001; esses valores serão avaliados junto às distribuições antes de propor tratamento.

#### Impressão das ausências por coluna

A célula abaixo lista, em ordem decrescente, as colunas com campos vazios em `listings_details.csv`. O percentual usa todas as linhas da base como denominador; valores de texto como `N/A` não são contados como células vazias.

In [6]:
# Importa csv para ler os registros da base.
import csv
# Importa Counter para contar valores ausentes por coluna.
from collections import Counter
# Importa Path para montar caminhos do sistema de arquivos.
from pathlib import Path

# Usa o diretório atual como ponto inicial para encontrar o projeto.
project_root_print = Path.cwd()
# Sobe para a pasta pai quando a pasta de dados não está no diretório atual.
if not (project_root_print / "data" / "raw").is_dir():
    # Atualiza o caminho para a pasta pai.
    project_root_print = project_root_print.parent
# Monta o caminho do arquivo principal para a contagem de células vazias.
missing_data_path = project_root_print / "data" / "raw" / "listings_details.csv"

# Inicializa contadores de células vazias por coluna.
empty_value_counts = Counter()
# Inicializa a quantidade total de anúncios lidos.
missing_rows_total = 0

# Abre a base principal para percorrer os valores sem carregá-la inteira na memória.
with missing_data_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê os registros como dicionários associados ao cabeçalho.
    missing_reader = csv.DictReader(csv_file)
    # Guarda os nomes das colunas disponíveis no arquivo.
    missing_columns = missing_reader.fieldnames or []
    # Percorre cada anúncio da base.
    for missing_row in missing_reader:
        # Atualiza o total de linhas usado no denominador dos percentuais.
        missing_rows_total += 1
        # Verifica individualmente cada coluna do anúncio.
        for column_name in missing_columns:
            # Conta a célula quando está vazia ou contém apenas espaços.
            if not (missing_row.get(column_name) or "").strip():
                # Soma uma ausência para a coluna correspondente.
                empty_value_counts[column_name] += 1

# Identifica o arquivo usado na contagem.
print("Arquivo:", missing_data_path)
# Exibe a quantidade de anúncios considerada.
print("Anúncios:", missing_rows_total)
# Exibe quantas colunas têm ao menos uma célula vazia.
print("Colunas com células vazias:", len(empty_value_counts))
# Indica que a próxima listagem mostra percentuais e contagens.
print("Percentual de ausências por coluna (contagem / total):")
# Percorre as colunas da mais ausente para a menos ausente.
for column_name, missing_count in empty_value_counts.most_common():
    # Calcula o percentual e troca ponto decimal por vírgula na exibição.
    missing_percent_text = f"{missing_count / missing_rows_total * 100:.2f}".replace(".", ",")
    # Exibe nome, percentual e razão de valores ausentes.
    print(f"{column_name}: {missing_percent_text}% ({missing_count}/{missing_rows_total})")

Arquivo: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv
Anúncios: 20030
Colunas com células vazias: 54
Percentual de ausências por coluna (contagem / total):
thumbnail_url: 100,00% (20030/20030)
medium_url: 100,00% (20030/20030)
xl_picture_url: 100,00% (20030/20030)
neighbourhood_group_cleansed: 100,00% (20030/20030)
license: 99,96% (20021/20030)
square_feet: 97,97% (19624/20030)
monthly_price: 92,21% (18469/20030)
weekly_price: 85,81% (17187/20030)
notes: 54,91% (10999/20030)
host_about: 41,18% (8248/20030)
interaction: 40,22% (8056/20030)
access: 38,96% (7803/20030)
house_rules: 37,24% (7459/20030)
neighborhood_overview: 33,81% (6773/20030)
transit: 31,93% (6395/20030)
security_deposit: 30,78% (6166/20030)
host_neighbourhood: 29,00% (5808/20030)
space: 27,21% (5451/20030)
cleaning_fee: 18,12% (3629/20030)
review_scores_checkin: 13,29% (2661/20030)
review_scores_location: 13,28% (2660/20030)
review_scores_value: 13,28% (2659/20030)
review_scores_communication: 13,24% (26

A saída lista as 54 colunas com campos vazios, do maior para o menor percentual, usando 20.030 anúncios no denominador. Marcadores textuais `N/A` não entram nessa contagem.

### 02.2 — Faixas, notas, datas e booleanos

Calcular faixas e quantis dos campos numéricos, verificar notas, datas e domínios booleanos. A regra de IQR servirá para sinalizar valores extremos para revisão, sem removê-los.

In [7]:
# Importa csv para ler os anúncios.
import csv
# Importa Counter para contar valores e falhas de validação.
from collections import Counter
# Importa date para converter campos de data.
from datetime import date
# Importa Decimal e InvalidOperation para auditar valores numéricos.
from decimal import Decimal, InvalidOperation

# Seleciona no esquema todos os campos de notas de avaliação.
score_fields = [name for name in column_names if name.startswith("review_scores_")]
# Identifica campos de taxas percentuais do anfitrião.
rate_fields = {"host_response_rate", "host_acceptance_rate"}
# Combina os candidatos numéricos, notas e taxas para a auditoria numérica.
numeric_audit_fields = sorted(set(numeric_candidates) | set(score_fields) | rate_fields)
# Prepara listas para datas válidas que existam no arquivo.
date_values = {name: [] for name in date_fields if name in column_names}
# Prepara contadores para datas que não puderem ser interpretadas.
date_invalid = Counter()
# Prepara contadores dos valores booleanos existentes no arquivo.
boolean_values = {name: Counter() for name in boolean_fields if name in column_names}
# Prepara contadores de valores que não pertencem ao domínio booleano esperado.
boolean_unexpected = Counter()
# Prepara listas de valores numéricos válidos para cada campo auditado.
audit_values = {name: [] for name in numeric_audit_fields}
# Prepara contadores de conversões numéricas inválidas por campo.
numeric_invalid = Counter()
# Inicializa a lista de anúncios cujo preço é zero.
zero_price_cases = []

# Abre a base principal para percorrer os campos de auditoria.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio como um dicionário de valores.
    reader = csv.DictReader(csv_file)
    # Percorre os anúncios da base.
    for row in reader:
        # Audita cada campo numérico selecionado.
        for name in numeric_audit_fields:
            # Lê o valor bruto e remove espaços das extremidades.
            raw_value = (row.get(name) or "").strip()
            # Ignora a conversão quando a célula está vazia.
            if not raw_value:
                # Passa ao próximo campo numérico.
                continue
            # Remove símbolos monetários e de percentual antes da conversão.
            numeric_text = raw_value.replace("$", "").replace("€", "").replace("£", "").replace("%", "")
            # Remove separadores de milhar e espaços restantes.
            numeric_text = numeric_text.replace(",", "").strip()
            # Tenta converter o valor limpo para Decimal.
            try:
                # Converte e armazena o valor numérico válido.
                parsed_value = Decimal(numeric_text)
                audit_values[name].append(parsed_value)
                # Guarda dados de identificação apenas para casos com preço zero.
                if name == "price" and parsed_value == 0:
                    # Registra atributos do anúncio para inspecionar esses casos.
                    zero_price_cases.append({
                        "id": (row.get("id") or "").strip(),
                        "room_type": (row.get("room_type") or "").strip(),
                        "property_type": (row.get("property_type") or "").strip(),
                        "accommodates": (row.get("accommodates") or "").strip()
                    })
            # Captura valores que não podem ser interpretados como Decimal.
            except InvalidOperation:
                # Conta a conversão inválida para o campo.
                numeric_invalid[name] += 1

        # Percorre as colunas de data selecionadas.
        for name in date_values:
            # Lê o conteúdo bruto e remove espaços externos.
            raw_value = (row.get(name) or "").strip()
            # Tenta converter somente datas preenchidas.
            if raw_value:
                # Interpreta os primeiros dez caracteres como uma data ISO.
                try:
                    # Guarda a data validada na lista do respectivo campo.
                    date_values[name].append(date.fromisoformat(raw_value[:10]))
                # Registra datas não interpretáveis sem interromper a auditoria.
                except ValueError:
                    # Conta a data inválida para o campo.
                    date_invalid[name] += 1

        # Percorre as colunas booleanas disponíveis.
        for name in boolean_values:
            # Lê o valor em minúsculas para padronizar a comparação.
            raw_value = (row.get(name) or "").strip().lower()
            # Conta os valores booleanos que não estão vazios.
            if raw_value:
                # Atualiza a frequência do valor observado.
                boolean_values[name][raw_value] += 1
                # Verifica se o conteúdo corresponde a uma representação booleana aceita.
                if raw_value not in {"t", "f", "true", "false", "yes", "no"}:
                    # Conta o valor fora do domínio esperado.
                    boolean_unexpected[name] += 1

# Introduz a apresentação das faixas observadas nas notas.
print("Notas observadas:")
# Percorre cada campo de nota.
for name in score_fields:
    # Obtém os valores válidos do campo atual.
    values = audit_values[name]
    # Exibe quantidade, mínimo e máximo, tratando uma lista sem valores.
    print(name, "| válidos:", len(values), "| mínimo:", min(values) if values else None, "| máximo:", max(values) if values else None)

# Introduz os quantis e a contagem de pontos fora dos limites de 1,5 × IQR.
print("\nFaixas, quantis e valores fora de 1,5 × IQR:")
# Percorre os campos numéricos incluídos na auditoria.
for name in numeric_audit_fields:
    # Obtém os valores válidos do campo atual.
    values = audit_values[name]
    # Trata campos sem valores numéricos.
    if not values:
        # Informa que não há números válidos para resumir.
        print(name, "| sem valores numéricos")
        # Passa para o próximo campo auditado.
        continue
    # Ordena os valores para calcular posições de quantis e extremos.
    ordered = sorted(values)
    # Armazena a quantidade de observações numéricas válidas.
    n_values = len(ordered)
    # Seleciona o valor na posição de 25% como primeiro quartil.
    q1 = ordered[int((n_values - 1) * 0.25)]
    # Seleciona o valor na posição de 50% como mediana.
    median = ordered[int((n_values - 1) * 0.50)]
    # Seleciona o valor na posição de 75% como terceiro quartil.
    q3 = ordered[int((n_values - 1) * 0.75)]
    # Calcula o limite inferior pela regra de 1,5 vezes o intervalo interquartil.
    lower_limit = q1 - Decimal("1.5") * (q3 - q1)
    # Calcula o limite superior pela mesma regra de 1,5 × IQR.
    upper_limit = q3 + Decimal("1.5") * (q3 - q1)
    # Conta observações abaixo do limite inferior.
    lower_outliers = sum(value < lower_limit for value in ordered)
    # Conta observações acima do limite superior.
    upper_outliers = sum(value > upper_limit for value in ordered)
    # Exibe o resumo estatístico e a quantidade fora dos limites de IQR.
    print(name, "| n:", n_values, "| min:", ordered[0], "| Q1:", q1, "| mediana:", median, "| Q3:", q3, "| max:", ordered[-1], "| abaixo/acima IQR:", lower_outliers, "/", upper_outliers)

# Introduz o resumo de datas.
print("\nDatas:")
# Percorre datas em ordem alfabética pelo nome da coluna.
for name, values in sorted(date_values.items()):
    # Mostra válidas, inválidas e intervalo de datas, se houver valores.
    print(name, "| válidas:", len(values), "| inválidas:", date_invalid[name], "| mínimo/máximo:", min(values) if values else None, "/", max(values) if values else None)

# Introduz as frequências de valores booleanos.
print("\nValores booleanos:")
# Percorre colunas booleanas em ordem alfabética.
for name, values in sorted(boolean_values.items()):
    # Exibe frequências e contagem de valores inesperados.
    print(name, "|", dict(values), "| valores inesperados:", boolean_unexpected[name])

# Exibe os atributos dos anúncios com preço zero para inspeção.
print("\nAnúncios com price zero:", zero_price_cases)
# Resume os valores numéricos que não puderam ser convertidos.
print("Parsing numérico inválido:", dict(numeric_invalid))

Notas observadas:
review_scores_rating | válidos: 17391 | mínimo: 20 | máximo: 100
review_scores_accuracy | válidos: 17381 | mínimo: 2 | máximo: 10
review_scores_cleanliness | válidos: 17383 | mínimo: 2 | máximo: 10
review_scores_checkin | válidos: 17369 | mínimo: 2 | máximo: 10
review_scores_communication | válidos: 17378 | mínimo: 2 | máximo: 10
review_scores_location | válidos: 17370 | mínimo: 2 | máximo: 10
review_scores_value | válidos: 17371 | mínimo: 2 | máximo: 10

Faixas, quantis e valores fora de 1,5 × IQR:
accommodates | n: 20030 | min: 1 | Q1: 2 | mediana: 2 | Q3: 4 | max: 17 | abaixo/acima IQR: 0 / 169
availability_365 | n: 20030 | min: 0 | Q1: 0 | mediana: 3 | Q3: 67 | max: 365 | abaixo/acima IQR: 0 / 2963
bathrooms | n: 20020 | min: 0.0 | Q1: 1.0 | mediana: 1.0 | Q3: 1.0 | max: 100.5 | abaixo/acima IQR: 101 / 3604
bedrooms | n: 20022 | min: 0 | Q1: 1 | mediana: 1 | Q3: 2 | max: 12 | abaixo/acima IQR: 0 / 582
host_acceptance_rate | sem valores numéricos
host_listings_coun

#### Estatísticas descritivas numéricas e categóricas

Gerar duas tabelas para as colunas da base: estatísticas numéricas dos valores parseáveis e resumo das categorias (contagem, ausências, valores distintos, moda e frequência). IDs e datas ficam fora dessas tabelas; percentuais entram como números. Células vazias e marcadores `N/A` são contados como ausências. A estatística descreve a base e não aplica o corte de features.

In [8]:
# 1. Importa ferramentas para leitura dos dados, cálculos e exibição.
# Importa csv para percorrer registros da base principal.
import csv
# Importa statistics para calcular média, desvio padrão e quantis.
import statistics
# Importa Counter para contar categorias, ausências e falhas.
from collections import Counter
# Importa Decimal e InvalidOperation para converter valores sem imprecisão binária.
from decimal import Decimal, InvalidOperation
# Importa display e Markdown para exibir tabelas formatadas no notebook.
from IPython.display import display, Markdown

# 2. Define a base e identifica campos que não devem ser tratados como medidas numéricas.
# Constrói o caminho do arquivo detalhado de anúncios.
stats_path = raw_dir / "listings_details.csv"
# Lista identificadores que não são medidas numéricas.
stats_id_fields = {"id", "host_id", "scrape_id"}
# Lista colunas que representam datas e serão categorizadas separadamente.
stats_date_fields = {
    "last_scraped", "calendar_last_scraped", "host_since",
    "first_review", "last_review"
}
# Lista campos cujo domínio esperado é booleano.
stats_boolean_fields = {
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "is_location_exact", "has_availability", "instant_bookable",
    "is_business_travel_ready", "require_guest_profile_picture",
    "require_guest_phone_verification"
}
# Lista taxas percentuais que serão tratadas como variáveis numéricas.
stats_rate_fields = {"host_response_rate", "host_acceptance_rate"}
# Força campos ambíguos a serem categóricos.
stats_forced_categorical = {"zipcode", "license"}
# Lista variáveis categóricas permitidas para exibir moda.
stats_safe_mode_fields = {
    "room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost",
    "instant_bookable", "host_has_profile_pic", "host_identity_verified",
    "is_location_exact", "has_availability", "is_business_travel_ready",
    "require_guest_profile_picture", "require_guest_phone_verification",
    "bed_type", "cancellation_policy", "host_response_time",
    "market", "city", "state", "country"
}

# 3. Prepara contadores e listas para valores, categorias, ausências e falhas de conversão.
# Inicializa as listas de valores numéricos válidos por coluna.
stats_numeric_values = {}
# Inicializa os contadores de categorias por coluna.
stats_category_counts = {}
# Prepara contagens de células vazias por coluna.
stats_blank_counts = Counter()
# Prepara contagens de marcadores textuais de ausência.
stats_marker_counts = Counter()
# Prepara contagens de falhas na conversão numérica.
stats_parse_failures = Counter()
# Inicializa o total de anúncios lidos.
stats_row_count = 0

# 4. Lê o snapshot e separa valores vazios, marcadores, categorias e números válidos.
# Abre o arquivo principal em modo texto com codificação UTF-8.
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada linha como um dicionário indexado pelos nomes das colunas.
    stats_reader = csv.DictReader(csv_file)
    # Captura os nomes das colunas do cabeçalho.
    stats_columns = stats_reader.fieldnames or []
    # Cria uma lista para os valores numéricos de cada coluna.
    stats_numeric_values = {name: [] for name in stats_columns}
    # Cria um contador de categorias para cada coluna.
    stats_category_counts = {name: Counter() for name in stats_columns}

    # Percorre cada anúncio do snapshot.
    for stats_row in stats_reader:
        # Atualiza o total de registros processados.
        stats_row_count += 1
        # Analisa todas as colunas do registro atual.
        for stats_name in stats_columns:
            # Lê o conteúdo da célula e remove espaços externos.
            stats_value = (stats_row.get(stats_name) or "").strip()
            # Verifica se a célula está vazia.
            if not stats_value:
                # Conta a ausência para a coluna atual.
                stats_blank_counts[stats_name] += 1
                # Pula a classificação desse valor vazio.
                continue
            # Identifica marcadores textuais comuns de valor ausente.
            if stats_value.casefold() in {"n/a", "nan", "null"}:
                # Conta o marcador textual para a coluna.
                stats_marker_counts[stats_name] += 1
                # Pula a conversão do marcador de ausência.
                continue
            # Mantém identificadores, datas, booleanos e campos forçados como categorias.
            if stats_name in stats_id_fields or stats_name in stats_date_fields or stats_name in stats_boolean_fields or stats_name in stats_forced_categorical:
                # Conta o valor como categoria observada.
                stats_category_counts[stats_name][stats_value] += 1
                # Passa à próxima coluna do registro.
                continue

            # Remove símbolos e separadores antes de converter o valor para número.
            stats_numeric_text = (
                stats_value.replace("$", "")
                .replace("€", "")
                .replace("£", "")
                .replace("%", "")
                .replace(",", "")
                .strip()
            )
            # Tenta converter o texto limpo para número decimal.
            try:
                # Faz a conversão usando Decimal.
                stats_decimal = Decimal(stats_numeric_text)
                # Confere se o resultado é finito antes de armazená-lo.
                if stats_decimal.is_finite():
                    # Guarda o número como float para os cálculos estatísticos.
                    stats_numeric_values[stats_name].append(float(stats_decimal))
                # Trata valores não finitos como falha de conversão.
                else:
                    # Conta o valor não finito para a coluna.
                    stats_parse_failures[stats_name] += 1
            # Captura textos e valores que não podem ser convertidos com segurança.
            except (InvalidOperation, ValueError, OverflowError):
                # Mantém o valor como categoria observada.
                stats_category_counts[stats_name][stats_value] += 1
                # Conta a falha na tentativa de conversão numérica.
                stats_parse_failures[stats_name] += 1

# 5. Classifica taxas definidas e colunas com pelo menos 90% de valores numéricos.
# Prepara a lista de nomes de colunas numéricas.
stats_numeric_columns = []
# Prepara a lista de nomes de colunas categóricas.
stats_categorical_columns = []
# Avalia uma a uma as colunas identificadas no arquivo.
for stats_name in stats_columns:
    # Exclui identificadores e datas das tabelas descritivas.
    if stats_name in stats_id_fields or stats_name in stats_date_fields:
        # Passa para a próxima coluna.
        continue
    # Conta os valores numéricos válidos da coluna.
    stats_numeric_n = len(stats_numeric_values[stats_name])
    # Soma os valores categóricos válidos da coluna.
    stats_category_n = sum(stats_category_counts[stats_name].values())
    # Calcula o total de valores observados que foram classificados.
    stats_observed_n = stats_numeric_n + stats_category_n
    # Classifica taxas ou colunas predominantemente numéricas como numéricas.
    if stats_name in stats_rate_fields or (
        stats_observed_n > 0 and stats_numeric_n / stats_observed_n >= 0.90
    ):
        # Adiciona a coluna à tabela numérica.
        stats_numeric_columns.append(stats_name)
    # Classifica as demais colunas como categóricas.
    else:
        # Adiciona a coluna à tabela categórica.
        stats_categorical_columns.append(stats_name)

# 6. Monta a tabela numérica com dispersão, quartis, assimetria e curtose excedente.
# Define os nomes das colunas da tabela numérica em Markdown.
stats_numeric_lines = [
    "| Variável numérica | N válido | Ausente (%) | Média | Desvio padrão | CV (%) | Assimetria (skew) | Curtose excedente (Fisher) | Mínimo | Q1 | Mediana | Q3 | Máximo |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"
]
# Monta as linhas de estatísticas em ordem alfabética.
for stats_name in sorted(stats_numeric_columns):
    # Ordena valores válidos para obter mínimo, máximo e quartis.
    stats_values = sorted(stats_numeric_values[stats_name])
    # Conta as observações numéricas válidas.
    stats_n = len(stats_values)
    # Soma células vazias, marcadores e falhas de conversão como ausentes.
    stats_missing_n = stats_blank_counts[stats_name] + stats_marker_counts[stats_name] + stats_parse_failures[stats_name]
    # Calcula a porcentagem ausente em relação ao total de anúncios.
    stats_missing_pct = stats_missing_n / stats_row_count * 100 if stats_row_count else 0

    # Calcula estatísticas quando há pelo menos um valor válido.
    if stats_n:
        # Calcula a média dos valores observados.
        stats_mean = statistics.mean(stats_values)
        # Obtém o menor valor após a ordenação.
        stats_minimum = stats_values[0]
        # Obtém o maior valor após a ordenação.
        stats_maximum = stats_values[-1]
        # Calcula medidas amostrais quando há mais de uma observação.
        if stats_n > 1:
            # Calcula o desvio padrão amostral.
            stats_std = statistics.stdev(stats_values)
            # Calcula primeiro quartil, mediana e terceiro quartil.
            stats_q1, stats_median, stats_q3 = statistics.quantiles(stats_values, n=4, method="inclusive")
            # Inicializa a assimetria como indisponível.
            stats_skew = None
            # Inicializa a curtose como indisponível.
            stats_kurtosis = None
            # Calcula a assimetria corrigida quando há observações suficientes e variação.
            if stats_n >= 3 and stats_std > 0:
                # Soma os desvios da média elevados à terceira potência.
                stats_centered_cubed_sum = sum((stats_value - stats_mean) ** 3 for stats_value in stats_values)
                # Aplica a fórmula de assimetria corrigida para amostra.
                stats_skew = (
                    stats_n * stats_centered_cubed_sum
                    / ((stats_n - 1) * (stats_n - 2) * (stats_std ** 3))
                )
            # Calcula a curtose excedente de Fisher, ajustada para a amostra.
            if stats_n >= 4 and stats_std > 0:
                # Soma os desvios da média elevados à quarta potência.
                stats_centered_fourth_sum = sum((stats_value - stats_mean) ** 4 for stats_value in stats_values)
                # Aplica a fórmula de curtose excedente corrigida para amostra.
                stats_kurtosis = (
                    stats_n * (stats_n + 1) * stats_centered_fourth_sum
                    / ((stats_n - 1) * (stats_n - 2) * (stats_n - 3) * (stats_std ** 4))
                    - 3 * (stats_n - 1) ** 2 / ((stats_n - 2) * (stats_n - 3))
                )
        # Trata uma única observação sem desvio ou quantis amostrais.
        else:
            # Marca o desvio padrão como indisponível.
            stats_std = None
            # Marca a assimetria como indisponível.
            stats_skew = None
            # Marca a curtose como indisponível.
            stats_kurtosis = None
            # Usa o único valor como todos os quartis.
            stats_q1 = stats_median = stats_q3 = stats_values[0]
    # Trata colunas sem valores numéricos válidos.
    else:
        # Marca estatísticas numéricas como indisponíveis para a coluna.
        stats_mean = stats_std = stats_skew = stats_kurtosis = stats_minimum = stats_q1 = stats_median = stats_q3 = stats_maximum = None

    # Calcula o CV amostral apenas para médias positivas e variáveis que não sejam coordenadas.
    if stats_std is not None and stats_mean is not None and stats_mean > 0 and stats_name not in {"latitude", "longitude"}:
        # Divide o desvio padrão amostral pela média e converte o resultado em percentual.
        stats_cv = stats_std / stats_mean * 100
    # Marca o CV como indisponível para média não positiva, coordenadas ou amostra insuficiente.
    else:
        # Usa valor ausente para evitar uma interpretação inadequada do coeficiente.
        stats_cv = None

    # Inicia a linha da tabela com nome, N válido e percentual ausente.
    stats_cells = [stats_name, str(stats_n), f"{stats_missing_pct:.2f}%".replace(".", ",")]
    # Percorre as estatísticas calculadas para formatar seus valores na tabela.
    for stats_value in [stats_mean, stats_std, stats_cv, stats_skew, stats_kurtosis, stats_minimum, stats_q1, stats_median, stats_q3, stats_maximum]:
        # Indica com travessão uma estatística não calculável.
        if stats_value is None:
            # Adiciona o marcador de valor indisponível.
            stats_cells.append("—")
        # Formata estatísticas numéricas com duas casas decimais.
        else:
            # Converte o ponto decimal para vírgula na apresentação.
            stats_cells.append(f"{stats_value:.2f}".replace(".", ","))
    # Constrói e adiciona a linha Markdown da variável.
    stats_numeric_lines.append("| " + " | ".join(stats_cells) + " |")

# 7. Monta a tabela categórica com quantidade de valores, moda e frequência.
# Define os nomes das colunas da tabela categórica em Markdown.
stats_categorical_lines = [
    "| Variável categórica | N válido | Ausente (%) | Valores únicos | Moda | Frequência da moda |",
    "|---|---:|---:|---:|---|---:|"
]
# Monta as linhas da tabela para as categorias em ordem alfabética.
for stats_name in sorted(stats_categorical_columns):
    # Combina categorias textuais e valores numéricos observados para resumir a coluna.
    stats_counts = stats_category_counts[stats_name].copy()
    # Inclui números válidos na contagem categórica para formar um resumo de todos os valores.
    for stats_number in stats_numeric_values[stats_name]:
        # Converte cada número em texto estável e soma uma ocorrência à categoria.
        stats_counts[f"{stats_number:.12g}"] += 1
    # Calcula o total de valores válidos da coluna.
    stats_n = sum(stats_counts.values())
    # Conta quantas categorias distintas foram observadas.
    stats_unique_n = len(stats_counts)
    # Soma células vazias e marcadores textuais de ausência.
    stats_missing_n = stats_blank_counts[stats_name] + stats_marker_counts[stats_name]
    # Calcula o percentual ausente em relação ao total de anúncios.
    stats_missing_pct = stats_missing_n / stats_row_count * 100 if stats_row_count else 0

    # Obtém a moda quando há pelo menos um valor válido.
    if stats_n:
        # Seleciona a categoria mais frequente e sua contagem.
        stats_mode_value, stats_mode_count = stats_counts.most_common(1)[0]
        # Oculta a moda quando há muitas categorias ou a variável não é segura para exibição.
        if stats_name in stats_safe_mode_fields and stats_unique_n <= 30:
            # Limpa separadores de tabela e limita o texto da categoria.
            stats_mode_label = stats_mode_value.replace("|", "/").replace("\n", " ")[:40]
        # Evita exibir moda em campos de alta cardinalidade.
        elif stats_unique_n > 30:
            # Registra que a moda foi ocultada devido à cardinalidade alta.
            stats_mode_label = "Oculta (alta cardinalidade)"
        # Oculta a moda dos demais campos.
        else:
            # Marca a moda como oculta.
            stats_mode_label = "Oculta"
    # Trata colunas sem valores válidos.
    else:
        # Indica que não há moda calculável.
        stats_mode_label = "—"
        # Define frequência zero para ausência de valores.
        stats_mode_count = 0

    # Prepara as células da linha categórica, incluindo a frequência da moda.
    stats_cells = [
        stats_name,
        str(stats_n),
        f"{stats_missing_pct:.2f}%".replace(".", ","),
        str(stats_unique_n),
        stats_mode_label,
        str(stats_mode_count)
    ]
    # Constrói e adiciona a linha Markdown da variável categórica.
    stats_categorical_lines.append("| " + " | ".join(stats_cells) + " |")

# 8. Exibe as tabelas e resume o tamanho da base analisada.
# Renderiza a tabela de estatísticas numéricas no notebook.
display(Markdown("#### Variáveis numéricas\n\n" + "\n".join(stats_numeric_lines)))
# Renderiza a tabela de estatísticas categóricas no notebook.
display(Markdown("#### Variáveis categóricas\n\n" + "\n".join(stats_categorical_lines)))
# Resume o arquivo e a quantidade de variáveis incluídas em cada tabela.
print(
    "Base:", stats_path,
    "| anúncios:", stats_row_count,
    "| variáveis numéricas:", len(stats_numeric_columns),
    "| variáveis categóricas:", len(stats_categorical_columns),
    "| IDs e datas fora das tabelas."
)

#### Variáveis numéricas

| Variável numérica | N válido | Ausente (%) | Média | Desvio padrão | CV (%) | Assimetria (skew) | Curtose excedente (Fisher) | Mínimo | Q1 | Mediana | Q3 | Máximo |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| accommodates | 20030 | 0,00% | 2,89 | 1,32 | 45,60 | 2,70 | 17,91 | 1,00 | 2,00 | 2,00 | 4,00 | 17,00 |
| availability_30 | 20030 | 0,00% | 4,91 | 7,94 | 161,91 | 1,67 | 1,71 | 0,00 | 0,00 | 0,00 | 7,00 | 30,00 |
| availability_365 | 20030 | 0,00% | 59,91 | 104,03 | 173,63 | 1,82 | 2,00 | 0,00 | 0,00 | 3,00 | 67,00 | 365,00 |
| availability_60 | 20030 | 0,00% | 11,41 | 17,50 | 153,35 | 1,44 | 0,77 | 0,00 | 0,00 | 0,00 | 18,00 | 60,00 |
| availability_90 | 20030 | 0,00% | 18,17 | 27,70 | 152,49 | 1,37 | 0,42 | 0,00 | 0,00 | 0,00 | 28,00 | 90,00 |
| bathrooms | 20020 | 0,05% | 1,14 | 0,79 | 69,78 | 98,64 | 12264,44 | 0,00 | 1,00 | 1,00 | 1,00 | 100,50 |
| bedrooms | 20022 | 0,04% | 1,45 | 0,89 | 61,29 | 1,86 | 9,03 | 0,00 | 1,00 | 1,00 | 2,00 | 12,00 |
| beds | 20023 | 0,03% | 1,85 | 1,39 | 75,15 | 4,79 | 57,92 | 0,00 | 1,00 | 1,00 | 2,00 | 32,00 |
| calculated_host_listings_count | 20030 | 0,00% | 4,73 | 22,92 | 484,13 | 7,85 | 63,73 | 1,00 | 1,00 | 1,00 | 1,00 | 208,00 |
| cleaning_fee | 16401 | 18,12% | 39,27 | 24,27 | 61,80 | 2,58 | 21,62 | 0,00 | 25,00 | 35,00 | 50,00 | 531,00 |
| extra_people | 20030 | 0,00% | 11,32 | 22,15 | 195,76 | 4,37 | 34,17 | 0,00 | 0,00 | 0,00 | 20,00 | 280,00 |
| guests_included | 20030 | 0,00% | 1,47 | 0,94 | 63,85 | 4,82 | 48,15 | 1,00 | 1,00 | 1,00 | 2,00 | 16,00 |
| host_acceptance_rate | 0 | 100,00% | — | — | — | — | — | — | — | — | — | — |
| host_listings_count | 20026 | 0,02% | 6,11 | 30,39 | 497,70 | 8,16 | 85,23 | 0,00 | 1,00 | 1,00 | 1,00 | 698,00 |
| host_response_rate | 10547 | 47,34% | 93,76 | 17,03 | 18,17 | -3,69 | 14,75 | 0,00 | 100,00 | 100,00 | 100,00 | 100,00 |
| host_total_listings_count | 20026 | 0,02% | 6,11 | 30,39 | 497,70 | 8,16 | 85,23 | 0,00 | 1,00 | 1,00 | 1,00 | 698,00 |
| latitude | 20030 | 0,00% | 52,37 | 0,02 | — | -0,20 | 2,12 | 52,29 | 52,36 | 52,36 | 52,38 | 52,42 |
| longitude | 20030 | 0,00% | 4,89 | 0,04 | — | 0,55 | 1,14 | 4,75 | 4,86 | 4,89 | 4,91 | 5,03 |
| maximum_nights | 20030 | 0,00% | 631,23 | 541,34 | 85,76 | 0,07 | 2,53 | 1,00 | 21,00 | 1125,00 | 1125,00 | 9999,00 |
| minimum_nights | 20030 | 0,00% | 3,33 | 12,54 | 376,65 | 56,36 | 4130,87 | 1,00 | 2,00 | 2,00 | 3,00 | 1001,00 |
| monthly_price | 1561 | 92,21% | 2594,99 | 1527,44 | 58,86 | 2,78 | 15,58 | 550,00 | 1650,00 | 2200,00 | 3000,00 | 20000,00 |
| number_of_reviews | 20030 | 0,00% | 21,56 | 43,24 | 200,55 | 5,42 | 41,50 | 0,00 | 3,00 | 8,00 | 22,00 | 695,00 |
| price | 20030 | 0,00% | 152,18 | 145,83 | 95,83 | 25,45 | 1168,15 | 0,00 | 96,00 | 125,00 | 175,00 | 8500,00 |
| review_scores_accuracy | 17381 | 13,23% | 9,68 | 0,69 | 7,08 | -3,93 | 26,90 | 2,00 | 9,00 | 10,00 | 10,00 | 10,00 |
| review_scores_checkin | 17369 | 13,29% | 9,74 | 0,65 | 6,64 | -3,98 | 25,81 | 2,00 | 10,00 | 10,00 | 10,00 | 10,00 |
| review_scores_cleanliness | 17383 | 13,22% | 9,47 | 0,87 | 9,17 | -2,84 | 13,87 | 2,00 | 9,00 | 10,00 | 10,00 | 10,00 |
| review_scores_communication | 17378 | 13,24% | 9,79 | 0,60 | 6,13 | -5,00 | 41,48 | 2,00 | 10,00 | 10,00 | 10,00 | 10,00 |
| review_scores_location | 17370 | 13,28% | 9,42 | 0,73 | 7,73 | -1,73 | 6,79 | 2,00 | 9,00 | 10,00 | 10,00 | 10,00 |
| review_scores_rating | 17391 | 13,18% | 94,85 | 6,65 | 7,01 | -3,40 | 22,03 | 20,00 | 93,00 | 97,00 | 100,00 | 100,00 |
| review_scores_value | 17371 | 13,28% | 9,18 | 0,81 | 8,78 | -2,00 | 10,73 | 2,00 | 9,00 | 9,00 | 10,00 | 10,00 |
| reviews_per_month | 17624 | 12,01% | 1,06 | 1,33 | 125,54 | 2,73 | 9,20 | 0,01 | 0,27 | 0,62 | 1,23 | 11,85 |
| security_deposit | 13864 | 30,78% | 221,88 | 340,21 | 153,33 | 6,56 | 63,69 | 0,00 | 0,00 | 150,00 | 295,00 | 4736,00 |
| square_feet | 406 | 97,97% | 520,49 | 544,56 | 104,62 | 1,69 | 6,30 | 0,00 | 0,00 | 484,00 | 807,00 | 4306,00 |
| weekly_price | 2843 | 85,81% | 821,05 | 458,84 | 55,88 | 3,04 | 17,09 | 100,00 | 550,00 | 700,00 | 950,00 | 6000,00 |

#### Variáveis categóricas

| Variável categórica | N válido | Ausente (%) | Valores únicos | Moda | Frequência da moda |
|---|---:|---:|---:|---|---:|
| access | 12227 | 38,96% | 9603 | Oculta (alta cardinalidade) | 397 |
| amenities | 20030 | 0,00% | 19034 | Oculta (alta cardinalidade) | 32 |
| bed_type | 20030 | 0,00% | 5 | Real Bed | 19829 |
| calendar_updated | 20030 | 0,00% | 68 | Oculta (alta cardinalidade) | 2996 |
| cancellation_policy | 20030 | 0,00% | 4 | strict_14_with_grace_period | 7874 |
| city | 20026 | 0,02% | 29 | Amsterdam | 19725 |
| country | 20030 | 0,00% | 1 | Netherlands | 20030 |
| country_code | 20030 | 0,00% | 1 | Oculta | 20030 |
| description | 19906 | 0,62% | 19648 | Oculta (alta cardinalidade) | 13 |
| experiences_offered | 20030 | 0,00% | 1 | Oculta | 20030 |
| has_availability | 20030 | 0,00% | 1 | t | 20030 |
| host_about | 11782 | 41,18% | 9429 | Oculta (alta cardinalidade) | 208 |
| host_has_profile_pic | 20026 | 0,02% | 2 | t | 20005 |
| host_identity_verified | 20026 | 0,02% | 2 | f | 11755 |
| host_is_superhost | 20026 | 0,02% | 2 | f | 16856 |
| host_location | 19991 | 0,19% | 546 | Oculta (alta cardinalidade) | 10161 |
| host_name | 20025 | 0,02% | 5914 | Oculta (alta cardinalidade) | 289 |
| host_neighbourhood | 14222 | 29,00% | 87 | Oculta (alta cardinalidade) | 3008 |
| host_picture_url | 20026 | 0,02% | 17240 | Oculta (alta cardinalidade) | 208 |
| host_response_time | 10547 | 47,34% | 4 | within an hour | 5113 |
| host_thumbnail_url | 20026 | 0,02% | 17240 | Oculta (alta cardinalidade) | 208 |
| host_url | 20030 | 0,00% | 17264 | Oculta (alta cardinalidade) | 208 |
| host_verifications | 20030 | 0,00% | 354 | Oculta (alta cardinalidade) | 3522 |
| house_rules | 12571 | 37,24% | 10679 | Oculta (alta cardinalidade) | 194 |
| instant_bookable | 20030 | 0,00% | 2 | f | 15230 |
| interaction | 11974 | 40,22% | 9632 | Oculta (alta cardinalidade) | 383 |
| is_business_travel_ready | 20030 | 0,00% | 1 | f | 20030 |
| is_location_exact | 20030 | 0,00% | 2 | t | 13982 |
| jurisdiction_names | 19358 | 3,35% | 1 | Oculta | 19358 |
| license | 9 | 99,96% | 2 | Oculta | 7 |
| listing_url | 20030 | 0,00% | 20030 | Oculta (alta cardinalidade) | 1 |
| market | 19988 | 0,21% | 4 | Amsterdam | 19981 |
| medium_url | 0 | 100,00% | 0 | — | 0 |
| name | 19992 | 0,19% | 19496 | Oculta (alta cardinalidade) | 39 |
| neighborhood_overview | 13257 | 33,81% | 12017 | Oculta (alta cardinalidade) | 47 |
| neighbourhood | 18377 | 8,25% | 44 | Oculta (alta cardinalidade) | 3924 |
| neighbourhood_cleansed | 20030 | 0,00% | 22 | De Baarsjes - Oud-West | 3515 |
| neighbourhood_group_cleansed | 0 | 100,00% | 0 | — | 0 |
| notes | 9031 | 54,91% | 7269 | Oculta (alta cardinalidade) | 233 |
| picture_url | 20030 | 0,00% | 19937 | Oculta (alta cardinalidade) | 3 |
| property_type | 20030 | 0,00% | 31 | Oculta (alta cardinalidade) | 15582 |
| require_guest_phone_verification | 20030 | 0,00% | 2 | f | 19707 |
| require_guest_profile_picture | 20030 | 0,00% | 2 | f | 19783 |
| requires_license | 20030 | 0,00% | 1 | Oculta | 20030 |
| room_type | 20030 | 0,00% | 3 | Entire home/apt | 15889 |
| smart_location | 20030 | 0,00% | 31 | Oculta (alta cardinalidade) | 19715 |
| space | 14579 | 27,21% | 14224 | Oculta (alta cardinalidade) | 26 |
| state | 19903 | 0,63% | 55 | Oculta (alta cardinalidade) | 15579 |
| street | 20030 | 0,00% | 103 | Oculta (alta cardinalidade) | 15361 |
| summary | 19510 | 2,60% | 19001 | Oculta (alta cardinalidade) | 32 |
| thumbnail_url | 0 | 100,00% | 0 | — | 0 |
| transit | 13635 | 31,93% | 12933 | Oculta (alta cardinalidade) | 20 |
| xl_picture_url | 0 | 100,00% | 0 | — | 0 |
| zipcode | 19164 | 4,32% | 6870 | Oculta (alta cardinalidade) | 403 |

Base: D:\repos\projetos\airbnb_price\data\raw\listings_details.csv | anúncios: 20030 | variáveis numéricas: 34 | variáveis categóricas: 54 | IDs e datas fora das tabelas.


A tabela numérica usa desvio padrão amostral, quartis por interpolação inclusiva, assimetria corrigida para amostra e curtose excedente de Fisher. O coeficiente de variação é calculado como desvio padrão amostral ÷ média × 100 e apresentado em percentual. Ele fica indisponível quando a média não é positiva, há poucas observações ou a variável é latitude/longitude, pois o zero das coordenadas é arbitrário. Em notas e taxas, a leitura do CV também deve respeitar a escala da variável. Os valores estão arredondados a duas casas. Para uma distribuição normal, a curtose excedente é 0. Nesta safra, `price` apresenta assimetria positiva elevada (25,45) e curtose excedente elevada (1.168,15), sinalizando cauda à direita e valores extremos a investigar na auditoria, sem determinar sua exclusão. A tabela categórica mostra a moda nas categorias de baixa cardinalidade; oculta o rótulo para texto livre e campos de alta cardinalidade, mantendo frequência e cardinalidade. As 88 variáveis exibidas incluem campos que serão excluídos pelo corte de ausências; a tabela é descritiva, não uma seleção de features.

#### Interpretação da verificação 02.2

`review_scores_rating` usa escala observada de 20–100; as seis notas detalhadas usam 2–10. As escalas devem permanecer distintas. As datas válidas e os campos booleanos seguem formatos consistentes; `availability_365` está no intervalo de 0 a 365.

A regra de 1,5 × IQR sinaliza muitos valores em variáveis assimétricas. Ela serve para triagem, não prova erro nem justifica exclusão. As tabelas descritivas mostram as distribuições completas; não filtram features. `host_response_rate` e `host_acceptance_rate` ficam fora da matriz por excederem o limite de 30% de ausências. Os valores de `bathrooms` e `maximum_nights` são examinados na subseção seguinte.

### 02.2.1 — Verificação de caudas em `bathrooms` e `maximum_nights`

A tabela abaixo investiga a assimetria de `bathrooms`, inspeciona os anúncios na cauda superior e resume a frequência de `maximum_nights`. Valores máximos altos podem ser configurações de estadia ou um limite operacional repetido; a frequência sozinha não comprova erro nem default.

In [9]:
# 1. Importa ferramentas para contar frequências e ler os registros completos.
# Importa Counter para contar frequências.
from collections import Counter
# Importa csv para ler os registros do snapshot.
import csv
# Importa Decimal e InvalidOperation para interpretar números com segurança.
from decimal import Decimal, InvalidOperation

# 2. Exibe as linhas de bathrooms e maximum_nights já calculadas na tabela descritiva.
for stats_line in stats_numeric_lines:
    # Seleciona as duas variáveis desta investigação.
    if stats_line.startswith("| bathrooms |") or stats_line.startswith("| maximum_nights |"):
        # Mostra média, desvio padrão, CV, assimetria, curtose e quantis da variável.
        print(stats_line)

# 3. Resume a frequência de cada valor observado em bathrooms e maximum_nights.
bathroom_frequency = Counter(stats_numeric_values["bathrooms"])
maximum_nights_frequency = Counter(stats_numeric_values["maximum_nights"])
# Calcula quantos registros de bathrooms estão na cauda alta.
bathroom_high_values = sorted(
    (value, count) for value, count in bathroom_frequency.items() if value >= 5
)
# Exibe as frequências de bathrooms iguais ou superiores a cinco.
print("\nFrequências de bathrooms >= 5:", bathroom_high_values)
# Exibe a frequência dos valores mais comuns de maximum_nights.
print("\nmaximum_nights — 15 valores mais frequentes:", maximum_nights_frequency.most_common(15))
# Ordena os valores de maximum_nights acima de 365 para mostrar as frequências da cauda.
maximum_nights_above_365 = sorted(
    (value, count) for value, count in maximum_nights_frequency.items() if value > 365
)
# Resume a quantidade de anúncios e valores distintos acima de 365 noites.
print(
    "maximum_nights > 365 — anúncios:",
    sum(count for value, count in maximum_nights_above_365),
    "| valores distintos:",
    len(maximum_nights_above_365),
    "| maiores valores e frequências:",
    maximum_nights_above_365[-15:]
)
# Calcula a participação do valor mais frequente no total observado.
maximum_nights_mode_value, maximum_nights_mode_count = maximum_nights_frequency.most_common(1)[0]
print(
    "Moda de maximum_nights:",
    maximum_nights_mode_value,
    "| frequência:",
    maximum_nights_mode_count,
    "| participação:",
    f"{maximum_nights_mode_count / len(stats_numeric_values['maximum_nights']) * 100:.2f}%"
)

# 4. Lista anúncios com dez ou mais banheiros para conferir compatibilidade dos atributos.
print("\nAnúncios com bathrooms >= 10:")
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio como um dicionário de campos do snapshot.
    stats_reader = csv.DictReader(csv_file)
    # Examina cada registro da base principal.
    for stats_row in stats_reader:
        # Lê o valor original de bathrooms.
        bathroom_text = (stats_row.get("bathrooms") or "").strip()
        # Tenta interpretar o valor para selecionar a cauda alta.
        try:
            # Converte o valor para Decimal sem aproximação binária.
            bathroom_value = Decimal(bathroom_text)
        # Ignora células vazias ou textos que não representam números.
        except (InvalidOperation, ValueError):
            # Passa ao próximo anúncio quando o campo não é numérico.
            continue
        # Exibe apenas os anúncios com dez ou mais banheiros.
        if bathroom_value >= 10:
            # Mostra campos associados para avaliar plausibilidade interna.
            print(
                "id:", stats_row.get("id"),
                "| bathrooms:", bathroom_text,
                "| bedrooms:", stats_row.get("bedrooms"),
                "| accommodates:", stats_row.get("accommodates"),
                "| room_type:", stats_row.get("room_type"),
                "| property_type:", stats_row.get("property_type"),
                "| price:", stats_row.get("price")
            )

# 5. Verifica inconsistências lógicas entre mínimo e máximo de noites.
maximum_nights_conflicts = []
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê novamente os registros para comparar os limites de estadia.
    stats_reader = csv.DictReader(csv_file)
    # Percorre anúncios que tenham ambos os limites preenchidos.
    for stats_row in stats_reader:
        # Lê os campos de estadia como texto.
        minimum_text = (stats_row.get("minimum_nights") or "").strip()
        maximum_text = (stats_row.get("maximum_nights") or "").strip()
        # Ignora registros sem um dos limites.
        if not minimum_text or not maximum_text:
            # Continua com o próximo anúncio.
            continue
        # Compara os valores depois de convertê-los para inteiro.
        try:
            # Converte os limites de estadia para números inteiros.
            minimum_value = int(Decimal(minimum_text))
            maximum_value = int(Decimal(maximum_text))
        # Ignora valores que não podem ser convertidos para inteiros.
        except (InvalidOperation, ValueError, OverflowError):
            # Continua com o próximo anúncio.
            continue
        # Registra casos em que a estadia mínima excede a máxima.
        if minimum_value > maximum_value:
            # Guarda o identificador e os dois limites para revisão.
            maximum_nights_conflicts.append(
                (stats_row.get("id"), minimum_value, maximum_value)
            )
# Informa quantos conflitos foram encontrados e lista até dez IDs.
print(
    "\nminimum_nights > maximum_nights:",
    len(maximum_nights_conflicts),
    "| exemplos:",
    maximum_nights_conflicts[:10]
)

# 6. Mede a sensibilidade do skew de bathrooms sem os três valores >= 10.
# Importa statistics para calcular média e desvio padrão amostral.
import statistics
# Mantém zeros e todos os demais valores para isolar o efeito da cauda muito alta.
bathroom_values_below_10 = [
    value for value in stats_numeric_values["bathrooms"] if value < 10
]
# Calcula o skew corrigido na amostra restante.
bathroom_mean_below_10 = statistics.mean(bathroom_values_below_10)
bathroom_std_below_10 = statistics.stdev(bathroom_values_below_10)
bathroom_centered_cubed_below_10 = sum(
    (value - bathroom_mean_below_10) ** 3 for value in bathroom_values_below_10
)
bathroom_skew_below_10 = (
    len(bathroom_values_below_10) * bathroom_centered_cubed_below_10
    / (
        (len(bathroom_values_below_10) - 1)
        * (len(bathroom_values_below_10) - 2)
        * (bathroom_std_below_10 ** 3)
    )
)
# Compara o skew observado com o skew após a exclusão apenas para diagnóstico.
print()
print(
    "bathrooms — skew original:",
    next(
        stats_line.split("|")[7].strip()
        for stats_line in stats_numeric_lines
        if stats_line.startswith("| bathrooms |")
    ),
    "| skew sem valores >= 10:",
    f"{bathroom_skew_below_10:.2f}",
    "| N após exclusão diagnóstica:",
    len(bathroom_values_below_10)
)

# 7. Inspeciona anúncios com maximum_nights acima do maior valor repetido de 1.125.
print()
print("Anúncios com maximum_nights > 1125:")
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio para associar o limite alto aos demais atributos.
    stats_reader = csv.DictReader(csv_file)
    # Percorre todos os anúncios do snapshot.
    for stats_row in stats_reader:
        # Lê o limite máximo de noites como texto.
        maximum_text = (stats_row.get("maximum_nights") or "").strip()
        # Ignora valores vazios ou inválidos.
        try:
            # Converte o limite máximo para inteiro.
            maximum_value = int(Decimal(maximum_text))
        # Pula o registro se o valor não puder ser convertido.
        except (InvalidOperation, ValueError, OverflowError):
            # Avança para o próximo anúncio.
            continue
        # Mostra apenas limites acima de 1.125 noites.
        if maximum_value > 1125:
            # Exibe os campos que ajudam a interpretar a configuração.
            print(
                "id:", stats_row.get("id"),
                "| maximum_nights:", maximum_value,
                "| minimum_nights:", stats_row.get("minimum_nights"),
                "| bedrooms:", stats_row.get("bedrooms"),
                "| accommodates:", stats_row.get("accommodates"),
                "| room_type:", stats_row.get("room_type"),
                "| property_type:", stats_row.get("property_type"),
                "| price:", stats_row.get("price")
            )


| bathrooms | 20020 | 0,05% | 1,14 | 0,79 | 69,78 | 98,64 | 12264,44 | 0,00 | 1,00 | 1,00 | 1,00 | 100,50 |
| maximum_nights | 20030 | 0,00% | 631,23 | 541,34 | 85,76 | 0,07 | 2,53 | 1,00 | 21,00 | 1125,00 | 1125,00 | 9999,00 |

Frequências de bathrooms >= 5: [(5.0, 2), (5.5, 1), (7.0, 1), (8.0, 3), (10.0, 1), (15.0, 1), (100.5, 1)]

maximum_nights — 15 valores mais frequentes: [(1125.0, 10229), (30.0, 1150), (14.0, 994), (7.0, 851), (365.0, 800), (60.0, 591), (10.0, 530), (21.0, 486), (1124.0, 425), (20.0, 333), (5.0, 324), (15.0, 287), (4.0, 279), (31.0, 220), (6.0, 207)]
maximum_nights > 365 — anúncios: 10783 | valores distintos: 29 | maiores valores e frequências: [(1082.0, 1), (1095.0, 1), (1096.0, 1), (1100.0, 10), (1103.0, 1), (1111.0, 2), (1112.0, 2), (1120.0, 18), (1122.0, 9), (1123.0, 23), (1124.0, 425), (1125.0, 10229), (1126.0, 1), (1825.0, 5), (9999.0, 1)]
Moda de maximum_nights: 1125.0 | frequência: 10229 | participação: 51.07%

Anúncios com bathrooms >= 10:
id: 15199899 

#### 02.2.2 — Interpretação da cauda de `bathrooms` e `maximum_nights`

**`bathrooms`.** A assimetria corrigida é 98,64 e o CV é 69,78%. Os três anúncios com 10, 15 e 100,5 banheiros têm um quarto e capacidade para quatro hóspedes. Excluindo provisoriamente esses três valores, o skew cai para 4,23 (20.017 observações), indicando que eles explicam a maior parte da cauda extrema. A regra para a base de modelagem será removê-los, mantendo os arquivos brutos intactos e registrando os IDs: 15199899, 16698560 e 17245735. Os demais valores altos não serão removidos automaticamente. Os 22 anúncios com zero banheiro serão mantidos, pois não há evidência de que zero seja um marcador de ausência.

**`maximum_nights`.** O valor 1.125 aparece em 10.229 anúncios (51,07%); 10.783 anúncios têm valor acima de 365. A mediana e o terceiro quartil são 1.125, e há um único valor 9.999 (anúncio 7447773; mínimo de 3 noites, dois quartos, capacidade para três hóspedes). Não há casos em que `minimum_nights` exceda `maximum_nights`. A documentação do Inside Airbnb descreve o campo como máximo de noites permitido no anúncio e observa que regras do calendário podem divergir; a Airbnb permite que anfitriões configurem a duração de estadias. Essas fontes não confirmam que 1.125 ou 9.999 sejam defaults da plataforma. A concentração em 1.125 é compatível com uma configuração padrão ou limite operacional, mas isso permanece uma hipótese. Manter os valores e anúncios na limpeza; na Etapa 03, avaliar `maximum_nights` como configuração discreta/categórica e evitar interpretar média ou desvio isoladamente.

Fontes de definição: [dicionário de dados do Inside Airbnb](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing) e [ajuda da Airbnb sobre noites mínimas e máximas](https://www.airbnb.com/help/article/880).

### 02.3 — Formatos e consistência de chaves

Verificar os valores brutos das taxas percentuais e comparar os IDs da base detalhada com `listings.csv`. Também comparar os nomes de bairros da base com `neighbourhoods.csv` e inspecionar o esquema do GeoJSON.

In [10]:
# Importa csv para ler as bases tabulares.
import csv
# Importa json para ler o arquivo de bairros em GeoJSON.
import json
# Importa Counter para contar frequências das taxas.
from collections import Counter

# Lista as taxas brutas que serão inspecionadas.
rate_fields = ["host_response_rate", "host_acceptance_rate"]
# Inicializa a contagem dos valores distintos de cada taxa.
rate_tokens = {name: Counter() for name in rate_fields}
# Inicializa contadores de valores ausentes por taxa.
rate_missing = Counter()

# Abre a base detalhada para conferir os valores brutos das taxas.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio como um dicionário de campos.
    reader = csv.DictReader(csv_file)
    # Percorre todos os anúncios.
    for row in reader:
        # Analisa cada uma das taxas selecionadas.
        for name in rate_fields:
            # Lê o valor e remove espaços das extremidades.
            value = (row.get(name) or "").strip()
            # Conta valores preenchidos.
            if value:
                # Atualiza a frequência do texto observado.
                rate_tokens[name][value] += 1
            # Conta valores vazios separadamente.
            else:
                # Soma a ausência para a taxa atual.
                rate_missing[name] += 1

# Inicializa o conjunto de IDs únicos da base resumida.
summary_ids = set()
# Inicializa a contagem de IDs repetidos na base resumida.
summary_duplicates = 0
# Inicializa a contagem de IDs ausentes na base resumida.
summary_missing_ids = 0
# Inicializa o total de linhas lidas da base resumida.
summary_rows = 0
# Abre listings.csv para comparar seus identificadores com a base detalhada.
with (raw_dir / "listings.csv").open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê a base resumida pelo nome das colunas.
    reader = csv.DictReader(csv_file)
    # Percorre as linhas da base resumida.
    for row in reader:
        # Atualiza o total de linhas lidas.
        summary_rows += 1
        # Lê o ID e remove espaços externos.
        listing_id = (row.get("id") or "").strip()
        # Verifica se o identificador está ausente.
        if not listing_id:
            # Conta o registro sem ID.
            summary_missing_ids += 1
        # Verifica se o ID já foi registrado.
        elif listing_id in summary_ids:
            # Conta o ID repetido.
            summary_duplicates += 1
        # Registra IDs preenchidos e ainda não vistos.
        else:
            # Adiciona o ID único ao conjunto.
            summary_ids.add(listing_id)

# Identifica anúncios que aparecem apenas na base detalhada.
detailed_only_ids = listing_ids - summary_ids
# Identifica anúncios que aparecem apenas na base resumida.
summary_only_ids = summary_ids - listing_ids

# Obtém bairros distintos da base principal.
master_neighbourhoods = set(category_counts["neighbourhood_cleansed"])
# Inicializa o conjunto de nomes encontrados no CSV auxiliar de bairros.
neighbourhood_csv_values = set()
# Inicializa a quantidade de linhas do CSV auxiliar.
neighbourhood_csv_rows = 0
# Abre o CSV auxiliar de bairros para comparar seus nomes.
with (raw_dir / "neighbourhoods.csv").open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê as linhas do arquivo pelo cabeçalho.
    reader = csv.DictReader(csv_file)
    # Percorre os bairros listados no arquivo auxiliar.
    for row in reader:
        # Atualiza o total de linhas auxiliares.
        neighbourhood_csv_rows += 1
        # Lê o nome do bairro e remove espaços externos.
        value = (row.get("neighbourhood") or "").strip()
        # Inclui apenas valores de bairro preenchidos.
        if value:
            # Registra o bairro no conjunto de nomes.
            neighbourhood_csv_values.add(value)

# Abre e interpreta o arquivo GeoJSON de bairros.
with (raw_dir / "neighbourhoods.geojson").open("r", encoding="utf-8") as geojson_file:
    # Carrega o conteúdo JSON em uma estrutura Python.
    neighbourhood_geojson = json.load(geojson_file)
# Obtém as feições geográficas ou uma lista vazia quando ausentes.
geojson_features = neighbourhood_geojson.get("features", [])
# Extrai os nomes dos campos de properties da primeira feição disponível.
geojson_property_keys = sorted(geojson_features[0].get("properties", {}).keys()) if geojson_features else []

# Introduz a contagem de valores brutos das taxas.
print("Taxas brutas:")
# Percorre cada taxa para mostrar ausências, cardinalidade e frequências.
for name in rate_fields:
    # Exibe o resumo da taxa atual.
    print(name, "| ausentes:", rate_missing[name], "| distintos:", len(rate_tokens[name]), "| frequências:", rate_tokens[name].most_common(12))

# Introduz a comparação entre base detalhada e base resumida.
print("\nlistings.csv:")
# Resume linhas, IDs únicos, ausentes e duplicados da base resumida.
print("Linhas:", summary_rows, "| IDs únicos:", len(summary_ids), "| IDs nulos:", summary_missing_ids, "| IDs duplicados:", summary_duplicates)
# Mostra quantos IDs são exclusivos de cada base.
print("IDs somente em listings_details.csv:", len(detailed_only_ids), "| IDs somente em listings.csv:", len(summary_only_ids))

# Introduz a comparação dos nomes de bairros.
print("\nBairros:")
# Resume a quantidade de nomes e linhas auxiliares encontradas.
print("Bairros em listings_details.csv:", len(master_neighbourhoods), "| em neighbourhoods.csv:", len(neighbourhood_csv_values), "| linhas auxiliares:", neighbourhood_csv_rows)
# Lista nomes presentes apenas na base detalhada.
print("Nomes somente na base detalhada:", sorted(master_neighbourhoods - neighbourhood_csv_values))
# Lista nomes presentes apenas no CSV auxiliar.
print("Nomes somente no arquivo auxiliar:", sorted(neighbourhood_csv_values - master_neighbourhoods))

# Introduz o resumo estrutural do GeoJSON.
print("\nGeoJSON:")
# Exibe quantidade de feições e propriedades encontradas.
print("Features:", len(geojson_features), "| campos em properties:", geojson_property_keys)

# Extrai os nomes de bairros das propriedades das feições GeoJSON.
geojson_neighbourhoods = {
    (feature.get("properties", {}).get("neighbourhood") or "").strip()
    for feature in geojson_features
}
# Remove entradas vazias do conjunto de nomes extraídos.
geojson_neighbourhoods.discard("")
# Lista bairros do GeoJSON ausentes na base principal.
print("Nomes somente no GeoJSON:", sorted(geojson_neighbourhoods - master_neighbourhoods))
# Lista bairros da base principal ausentes no GeoJSON.
print("Nomes somente nas bases detalhada/CSV:", sorted(master_neighbourhoods - geojson_neighbourhoods))

Taxas brutas:
host_response_rate | ausentes: 4 | distintos: 59 | frequências: [('N/A', 9479), ('100%', 7982), ('90%', 329), ('96%', 245), ('80%', 207), ('50%', 178), ('0%', 162), ('98%', 158), ('67%', 134), ('75%', 93), ('95%', 85), ('83%', 79)]
host_acceptance_rate | ausentes: 4 | distintos: 1 | frequências: [('N/A', 20026)]

listings.csv:
Linhas: 20030 | IDs únicos: 20030 | IDs nulos: 0 | IDs duplicados: 0
IDs somente em listings_details.csv: 0 | IDs somente em listings.csv: 0

Bairros:
Bairros em listings_details.csv: 22 | em neighbourhoods.csv: 22 | linhas auxiliares: 22
Nomes somente na base detalhada: []
Nomes somente no arquivo auxiliar: []

GeoJSON:
Features: 22 | campos em properties: ['neighbourhood', 'neighbourhood_group']
Nomes somente no GeoJSON: []
Nomes somente nas bases detalhada/CSV: []


#### Interpretação da verificação 02.3

`host_response_rate` registra 9.479 valores `N/A` e 10.547 percentuais; `host_acceptance_rate` registra `N/A` em todos os 20.026 valores não vazios. São marcadores de ausência na safra, não falhas de parsing numérico. O tratamento permanece pendente.

`listings.csv` contém os mesmos 20.030 IDs da base detalhada, sem duplicatas. Os 22 bairros coincidem entre a base, `neighbourhoods.csv` e as 22 features do GeoJSON.

### 02.4 — Calendário: chaves, datas e disponibilidade

A primeira varredura encontrou 365 linhas por anúncio e datas entre 06/12/2018 e 06/12/2019. Como o CSV não está agrupado por anúncio, a checagem abaixo valida a chave `listing_id + date` com um bitmap por anúncio. Também cruza `available` com preço ausente/presente. O preço não será usado como atributo do modelo.

In [11]:
# Importa csv para ler o calendário linha a linha.
import csv
# Importa Counter para resumir valores, linhas e frequências.
from collections import Counter
# Importa date e timedelta para construir e validar o período de datas.
from datetime import date, timedelta
# Importa Decimal e InvalidOperation para validar preços do calendário.
from decimal import Decimal, InvalidOperation

# Monta o caminho do calendário.
calendar_path = raw_dir / "calendar.csv"
# Declara as colunas necessárias para a auditoria.
calendar_required = {"listing_id", "date", "available", "price"}
# Define o primeiro dia do snapshot como origem do intervalo.
calendar_origin = date(2018, 12, 6)
# Define a quantidade de dias esperada no horizonte auditado.
calendar_horizon_days = 366
# Inicializa o mapeamento entre datas ISO e deslocamentos inteiros.
calendar_date_index = {}
# Cria um índice de cada dia esperado no intervalo.
for day_offset in range(calendar_horizon_days):
    # Associa a data ISO ao número de dias desde a origem.
    calendar_date_index[(calendar_origin + timedelta(days=day_offset)).isoformat()] = day_offset

# Inicializa variáveis para contar linhas, ausência e chaves.
calendar_dates = None
# Inicializa o total de linhas do calendário.
calendar_row_count = 0
# Prepara contagens de campos ausentes.
calendar_missing = Counter()
# Prepara o conjunto de anúncios encontrados no calendário.
calendar_listing_ids = set()
# Prepara o conjunto de IDs do calendário sem correspondência na base principal.
calendar_orphan_ids = set()
# Conta o número de linhas por anúncio.
calendar_rows_by_listing = Counter()
# Guarda o bitmap de datas encontradas para cada anúncio.
calendar_date_bitmap = {}
# Guarda pares de anúncio e data fora do intervalo para detectar duplicatas.
calendar_out_of_window_pairs = set()
# Conta quantas datas distintas existem por anúncio.
calendar_unique_date_counts = Counter()
# Conta os valores observados no campo available.
calendar_available_values = Counter()
# Conta a combinação entre disponibilidade e presença de preço.
calendar_availability_price = Counter()
# Inicializa a contagem de valores de disponibilidade inesperados.
calendar_unexpected_available = 0
# Inicializa o total de datas inválidas.
calendar_invalid_dates = 0
# Inicializa o total de datas fora do intervalo esperado.
calendar_outside_window = 0
# Inicializa a contagem de chaves anúncio-data repetidas.
calendar_duplicate_keys = 0
# Prepara a contagem de resultados do parsing de preço.
calendar_price_status = Counter()
# Guarda grupos de anúncios já encerrados na ordem original do arquivo.
calendar_groups_closed = set()
# Inicializa a contagem de grupos que reaparecem depois de encerrados.
calendar_group_reappears = 0
# Inicializa o ID do anúncio da linha anterior.
previous_listing_id = None
# Inicializa os limites mínimo e máximo de data observados.
calendar_min_date = None
calendar_max_date = None

# Abre o calendário em modo texto para processar muitas linhas com baixo uso de memória.
with calendar_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê o arquivo como linhas e preserva a ordem original.
    reader = csv.reader(csv_file)
    # Lê o cabeçalho ou uma lista vazia se não houver linhas.
    calendar_header = next(reader, [])
    # Identifica as colunas obrigatórias ausentes no cabeçalho.
    missing_calendar_columns = sorted(calendar_required - set(calendar_header))
    # Mapeia cada coluna disponível para sua posição na linha.
    calendar_index = {name: calendar_header.index(name) for name in calendar_required if name in calendar_header}

    # Percorre todas as linhas de dados do calendário.
    for row in reader:
        # Atualiza o total de linhas processadas.
        calendar_row_count += 1
        # Lê e limpa o identificador do anúncio.
        listing_id = row[calendar_index["listing_id"]].strip()
        # Lê e limpa a data associada à linha.
        date_text = row[calendar_index["date"]].strip()
        # Lê disponibilidade e padroniza o texto para minúsculas.
        available_text = row[calendar_index["available"]].strip().lower()
        # Lê e limpa o preço do calendário.
        price_text = row[calendar_index["price"]].strip()

        # Percorre as colunas necessárias para contar valores vazios.
        for name, value in [
            ("listing_id", listing_id),
            ("date", date_text),
            ("available", available_text),
            ("price", price_text)
        ]:
            # Verifica se o valor da coluna está vazio.
            if not value:
                # Soma a ausência para a coluna correspondente.
                calendar_missing[name] += 1

        # Processa o identificador quando estiver preenchido.
        if listing_id:
            # Registra o anúncio no conjunto de IDs encontrados.
            calendar_listing_ids.add(listing_id)
            # Atualiza a contagem de linhas desse anúncio.
            calendar_rows_by_listing[listing_id] += 1
            # Confere se o anúncio está presente na base principal.
            if listing_id not in listing_ids:
                # Registra o ID sem correspondência.
                calendar_orphan_ids.add(listing_id)

        # Conta o valor da disponibilidade, incluindo vazios.
        calendar_available_values[available_text or "(vazio)"] += 1
        # Detecta valores de disponibilidade fora das representações booleanas conhecidas.
        if available_text and available_text not in {"t", "f", "true", "false", "yes", "no"}:
            # Incrementa a contagem de valores inesperados.
            calendar_unexpected_available += 1
        # Conta se o preço está presente para cada valor de disponibilidade.
        calendar_availability_price[(available_text or "(vazio)", "presente" if price_text else "ausente")] += 1

        # Processa os limites e a posição de linhas com data preenchida.
        if date_text:
            # Atualiza a menor data observada.
            if calendar_min_date is None or date_text < calendar_min_date:
                # Guarda a nova data mínima.
                calendar_min_date = date_text
            # Atualiza a maior data observada.
            if calendar_max_date is None or date_text > calendar_max_date:
                # Guarda a nova data máxima.
                calendar_max_date = date_text

            # Obtém o deslocamento da data no intervalo esperado.
            day_offset = calendar_date_index.get(date_text)
            # Identifica datas que não pertencem ao intervalo indexado.
            if day_offset is None:
                # Conta a data como fora da janela.
                calendar_outside_window += 1
                # Conta a data como não reconhecida no intervalo.
                calendar_invalid_dates += 1
                # Detecta repetições fora do intervalo quando há ID disponível.
                if listing_id:
                    # Forma a chave de anúncio e data para checagem.
                    key = (listing_id, date_text)
                    # Verifica se a mesma chave já foi encontrada fora da janela.
                    if key in calendar_out_of_window_pairs:
                        # Conta uma chave duplicada.
                        calendar_duplicate_keys += 1
                    # Registra uma nova combinação de anúncio e data.
                    else:
                        # Adiciona o par ao conjunto de controle.
                        calendar_out_of_window_pairs.add(key)
            # Registra datas esperadas para anúncios com ID válido.
            elif listing_id:
                # Obtém o bitmap já criado para o anúncio, se houver.
                bitmap = calendar_date_bitmap.get(listing_id)
                # Cria um bitmap vazio ao encontrar o anúncio pela primeira vez.
                if bitmap is None:
                    # Aloca um marcador por dia do horizonte.
                    bitmap = bytearray(calendar_horizon_days)
                    # Guarda o bitmap para reuso nas próximas datas do anúncio.
                    calendar_date_bitmap[listing_id] = bitmap
                # Verifica se essa data já foi registrada para o anúncio.
                if bitmap[day_offset]:
                    # Conta a chave duplicada de anúncio e data.
                    calendar_duplicate_keys += 1
                # Marca a data como encontrada quando ainda não havia sido registrada.
                else:
                    # Atualiza o marcador do dia no bitmap.
                    bitmap[day_offset] = 1

        # Valida o preço quando o campo está preenchido.
        if price_text:
            # Remove símbolos monetários e separadores antes da conversão.
            normalized_price = price_text.replace("$", "").replace("€", "").replace("£", "").replace(",", "").strip()
            # Tenta interpretar o preço limpo como Decimal.
            try:
                # Converte o preço para valor numérico.
                parsed_price = Decimal(normalized_price)
                # Conta o valor como preço numérico.
                calendar_price_status["numérico"] += 1
                # Conta preços iguais a zero.
                if parsed_price == 0:
                    # Incrementa o contador de preços zero.
                    calendar_price_status["zero"] += 1
                # Conta preços negativos.
                elif parsed_price < 0:
                    # Incrementa o contador de preços negativos.
                    calendar_price_status["negativo"] += 1
            # Captura texto de preço que não pode ser convertido.
            except InvalidOperation:
                # Conta preço não numérico.
                calendar_price_status["não numérico"] += 1
        # Registra preço ausente.
        else:
            # Incrementa o contador de preços ausentes.
            calendar_price_status["ausente"] += 1

        # Detecta a mudança de grupo de anúncio na ordem do arquivo.
        if listing_id != previous_listing_id:
            # Fecha o grupo anterior quando existe um anúncio anterior.
            if previous_listing_id is not None:
                # Marca o anúncio anterior como grupo já encerrado.
                calendar_groups_closed.add(previous_listing_id)
            # Verifica se o novo ID já teve grupo encerrado anteriormente.
            if listing_id in calendar_groups_closed:
                # Conta o grupo que reaparece fora da sequência contínua.
                calendar_group_reappears += 1
            # Atualiza o ID anterior para a linha seguinte.
            previous_listing_id = listing_id

# Conta datas únicas de cada anúncio somando os marcadores do bitmap.
for listing_id, bitmap in calendar_date_bitmap.items():
    # Soma os dias registrados e atualiza a distribuição de datas por anúncio.
    calendar_unique_date_counts[sum(bitmap)] += 1
# Resume a distribuição da quantidade de linhas por anúncio.
calendar_count_distribution = Counter(calendar_rows_by_listing.values())

# Informa o arquivo de calendário auditado.
print("Arquivo:", calendar_path)
# Exibe colunas necessárias que não foram encontradas.
print("Colunas ausentes:", missing_calendar_columns)
# Resume linhas, IDs referenciados e IDs sem correspondência.
print("Linhas:", calendar_row_count, "| anúncios referenciados:", len(calendar_listing_ids), "| IDs sem correspondência:", len(calendar_orphan_ids))
# Exibe intervalo observado e datas fora da janela.
print("Intervalo de datas:", calendar_min_date, "/", calendar_max_date, "| fora da janela observada:", calendar_outside_window)
# Mostra a distribuição do número de linhas por anúncio.
print("Linhas por anúncio:", sorted(calendar_count_distribution.items()))
# Mostra a distribuição de datas únicas por anúncio.
print("Datas únicas por anúncio:", sorted(calendar_unique_date_counts.items()))
# Exibe a quantidade de chaves anúncio-data repetidas.
print("Chaves listing_id + date duplicadas:", calendar_duplicate_keys)
# Exibe grupos de anúncio que reapareceram após outro grupo.
print("Grupos de anúncio que reaparecem na ordem do arquivo:", calendar_group_reappears)
# Resume valores do campo available e valores inesperados.
print("available:", dict(calendar_available_values), "| valores fora do domínio booleano:", calendar_unexpected_available)
# Resume presença de preço por estado de disponibilidade.
print("available × preço:", dict(calendar_availability_price))
# Exibe ausências nas colunas obrigatórias.
print("Campos obrigatórios ausentes:", dict(calendar_missing))
# Exibe os resultados da validação numérica dos preços do calendário.
print("Parsing do preço do calendário:", dict(calendar_price_status))

Arquivo: D:\repos\projetos\airbnb_price\data\raw\calendar.csv
Colunas ausentes: []
Linhas: 7310950 | anúncios referenciados: 20030 | IDs sem correspondência: 0
Intervalo de datas: 2018-12-06 / 2019-12-06 | fora da janela observada: 0
Linhas por anúncio: [(365, 20030)]
Datas únicas por anúncio: [(365, 20030)]
Chaves listing_id + date duplicadas: 0
Grupos de anúncio que reaparecem na ordem do arquivo: 54509
available: {'f': 6110879, 't': 1200071} | valores fora do domínio booleano: 0
available × preço: {('f', 'ausente'): 6110879, ('t', 'presente'): 1200071}
Campos obrigatórios ausentes: {'price': 6110879}
Parsing do preço do calendário: {'ausente': 6110879, 'numérico': 1200071}


### 02.5 — Avaliações e granularidade

Comparar `reviews.csv` e `reviews_details.csv` por estrutura, chaves, datas e vínculo com anúncios. A saída mostra apenas contagens e nomes de colunas; comentários e nomes de usuários não serão exibidos. Também comparar a contagem por anúncio com `number_of_reviews`, sem assumir que as bases cobrem o mesmo período.

In [12]:
# Importa csv para percorrer arquivos de avaliações.
import csv
# Importa Counter para contar valores, chaves e divergências.
from collections import Counter
# Importa date para validar datas de avaliações.
from datetime import date
# Importa Decimal e InvalidOperation para ler a contagem de avaliações da base principal.
from decimal import Decimal, InvalidOperation

# Lista as bases de avaliações que serão comparadas.
review_files = ["reviews.csv", "reviews_details.csv"]
# Prepara resumos por arquivo.
review_stats = {}
# Prepara conjuntos de IDs de avaliação por arquivo.
review_id_sets = {}
# Prepara contagens de avaliações por anúncio e arquivo.
review_listing_counts = {}
# Prepara contagens de combinações anúncio-data por arquivo.
review_pair_counts = {}
# Prepara as contagens de avaliações declaradas na base principal.
expected_review_counts = {}

# Abre a base detalhada para guardar number_of_reviews por anúncio.
with listings_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio pelas colunas do cabeçalho.
    for listing in csv.DictReader(csv_file):
        # Lê e limpa o ID do anúncio.
        listing_id = (listing.get("id") or "").strip()
        # Lê e limpa a contagem de avaliações declarada.
        raw_count = (listing.get("number_of_reviews") or "").strip()
        # Só converte quando ID e contagem estão preenchidos.
        if listing_id and raw_count:
            # Tenta converter a contagem textual para inteiro.
            try:
                # Guarda o número esperado de avaliações pelo ID.
                expected_review_counts[listing_id] = int(Decimal(raw_count))
            # Ignora contagens que não são números válidos.
            except (InvalidOperation, ValueError):
                # Continua a leitura sem interromper a auditoria.
                pass

# Analisa uma base de avaliações por vez.
for review_file_name in review_files:
    # Monta o caminho do arquivo de avaliações atual.
    review_path = raw_dir / review_file_name
    # Inicializa a quantidade de linhas do arquivo.
    row_count = 0
    # Prepara contagens de ausências por coluna.
    missing_counts = Counter()
    # Inicializa a contagem de datas inválidas.
    invalid_dates = 0
    # Inicializa a contagem de registros sem vínculo a anúncio.
    orphan_rows = 0
    # Prepara o conjunto de IDs de avaliações já vistos.
    ids_seen = set()
    # Inicializa a contagem de IDs de avaliações repetidos.
    duplicate_review_ids = 0
    # Prepara contagens de avaliações por anúncio.
    listing_counts = Counter()
    # Prepara contagens da chave anúncio-data.
    pair_counts = Counter()
    # Abre o arquivo atual em modo texto.
    with review_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
        # Lê os registros como dicionários.
        reader = csv.DictReader(csv_file)
        # Guarda os nomes das colunas disponíveis.
        review_columns = reader.fieldnames or []
        # Percorre todas as avaliações do arquivo.
        for review in reader:
            # Atualiza o total de linhas lidas.
            row_count += 1
            # Lê e limpa o ID da avaliação, se houver.
            review_id = (review.get("id") or "").strip()
            # Lê e limpa o ID do anúncio associado.
            listing_id = (review.get("listing_id") or "").strip()
            # Lê e limpa a data da avaliação.
            review_date = (review.get("date") or "").strip()

            # Verifica IDs de avaliação quando essa coluna existe.
            if "id" in review_columns:
                # Conta IDs de avaliação ausentes.
                if not review_id:
                    # Soma ausência na coluna id.
                    missing_counts["id"] += 1
                # Detecta repetição de um ID já visto.
                elif review_id in ids_seen:
                    # Incrementa a contagem de IDs duplicados.
                    duplicate_review_ids += 1
                # Registra um ID de avaliação novo.
                else:
                    # Adiciona o ID ao conjunto de IDs vistos.
                    ids_seen.add(review_id)

            # Conta ausência de listing_id quando o campo existe.
            if not listing_id and "listing_id" in review_columns:
                # Soma a ausência da chave do anúncio.
                missing_counts["listing_id"] += 1
            # Processa vínculos com anúncio quando o ID está preenchido.
            elif listing_id:
                # Conta a avaliação para o anúncio atual.
                listing_counts[listing_id] += 1
                # Verifica se o anúncio está na base principal.
                if listing_id not in listing_ids:
                    # Conta a linha como sem vínculo com a base principal.
                    orphan_rows += 1

            # Conta ausência da data de avaliação.
            if not review_date:
                # Soma a ausência na coluna de data.
                missing_counts["date"] += 1
            # Valida datas preenchidas.
            else:
                # Tenta interpretar a data em formato ISO.
                try:
                    # Valida os primeiros dez caracteres da data.
                    date.fromisoformat(review_date[:10])
                # Captura valores com formato de data inválido.
                except ValueError:
                    # Incrementa o total de datas inválidas.
                    invalid_dates += 1

            # Conta a chave anúncio-data quando ambas as partes existem.
            if listing_id and review_date:
                # Atualiza a frequência dessa combinação.
                pair_counts[(listing_id, review_date)] += 1
            # Conta comentários vazios sem exibir seu conteúdo.
            if "comments" in review_columns and not (review.get("comments") or "").strip():
                # Soma ausência de comentário.
                missing_counts["comments"] += 1

    # Salva resumo estrutural e de qualidade do arquivo atual.
    review_stats[review_file_name] = {
        "rows": row_count,
        "columns": review_columns,
        "missing": missing_counts,
        "invalid_dates": invalid_dates,
        "orphan_rows": orphan_rows,
        "unique_ids": len(ids_seen) if "id" in review_columns else None,
        "duplicate_ids": duplicate_review_ids if "id" in review_columns else None,
        "ids_available": "id" in review_columns
    }
    # Guarda os IDs de avaliação do arquivo atual para comparação entre arquivos.
    review_id_sets[review_file_name] = ids_seen
    # Guarda contagens por anúncio do arquivo atual.
    review_listing_counts[review_file_name] = listing_counts
    # Guarda contagens por chave anúncio-data do arquivo atual.
    review_pair_counts[review_file_name] = pair_counts

    # Identifica anúncios presentes na base principal e no arquivo atual.
    comparable_listings = set(expected_review_counts) & set(listing_counts)
    # Conta quantos anúncios comparáveis têm a mesma contagem de avaliações.
    exact_count_matches = sum(
        listing_counts[listing_id] == expected_review_counts[listing_id]
        for listing_id in comparable_listings
    )
    # Conta anúncios com contagem positiva na base principal e sem linhas no arquivo.
    listings_without_rows = sum(
        expected_review_counts[listing_id] > 0 and listing_id not in listing_counts
        for listing_id in expected_review_counts
    )
    # Exibe estrutura resumida e qualidade sem mostrar identificadores pessoais ou comentários.
    print(
        review_file_name,
        "| linhas:", row_count,
        "| colunas (sem campos de identificação pessoal/conteúdo):",
        [name for name in review_columns if name not in {"reviewer_name", "reviewer_id", "comments"}],
        "| IDs de avaliação:",
        (f"{len(ids_seen)} únicos, {duplicate_review_ids} repetidos" if "id" in review_columns else "coluna inexistente"),
        "| ausências:", dict(missing_counts),
        "| datas inválidas:", invalid_dates,
        "| linhas sem vínculo com anúncio:", orphan_rows
    )
    # Exibe a comparação com number_of_reviews da base principal.
    print(
        "  comparação com number_of_reviews:",
        "anúncios comparáveis:", len(comparable_listings),
        "| contagens iguais:", exact_count_matches,
        "| diferentes:", len(comparable_listings) - exact_count_matches,
        "| anúncios sem registro apesar de contagem positiva:", listings_without_rows
    )

# Obtém a contagem anúncio-data do primeiro arquivo.
first_pairs = review_pair_counts[review_files[0]]
# Obtém a contagem anúncio-data do segundo arquivo.
second_pairs = review_pair_counts[review_files[1]]
# Compara os arquivos quando ambos têm pares anúncio-data.
if first_pairs and second_pairs:
    # Conta linhas pareadas considerando a menor frequência em cada arquivo.
    common_pair_rows = sum(min(count, second_pairs.get(key, 0)) for key, count in first_pairs.items())
    # Conta linhas excedentes apenas no primeiro arquivo.
    first_only_rows = sum(max(0, count - second_pairs.get(key, 0)) for key, count in first_pairs.items())
    # Conta linhas excedentes apenas no segundo arquivo.
    second_only_rows = sum(max(0, count - first_pairs.get(key, 0)) for key, count in second_pairs.items())
    # Conta pares com frequências diferentes entre os arquivos.
    pair_count_differences = sum(first_pairs.get(key, 0) != second_pairs.get(key, 0) for key in set(first_pairs) | set(second_pairs))
    # Exibe linhas em comum e diferenças encontradas.
    print(
        "Comparação por listing_id + date:",
        "linhas pareadas:", common_pair_rows,
        "| somente em reviews.csv:", first_only_rows,
        "| somente em reviews_details.csv:", second_only_rows,
        "| combinações com contagem diferente:", pair_count_differences
    )

# Compara IDs de avaliação quando os dois arquivos oferecem essa coluna.
if all(review_stats[name]["ids_available"] for name in review_files):
    # Calcula a interseção dos IDs de avaliação entre os arquivos.
    review_id_intersection = review_id_sets[review_files[0]] & review_id_sets[review_files[1]]
    # Exibe IDs compartilhados e exclusivos em cada arquivo.
    print(
        "IDs compartilhados entre os arquivos:",
        len(review_id_intersection),
        "| exclusivos em reviews.csv:", len(review_id_sets[review_files[0]] - review_id_sets[review_files[1]]),
        "| exclusivos em reviews_details.csv:", len(review_id_sets[review_files[1]] - review_id_sets[review_files[0]])
    )

reviews.csv | linhas: 431830 | colunas (sem campos de identificação pessoal/conteúdo): ['listing_id', 'date'] | IDs de avaliação: coluna inexistente | ausências: {} | datas inválidas: 0 | linhas sem vínculo com anúncio: 0
  comparação com number_of_reviews: anúncios comparáveis: 17624 | contagens iguais: 17513 | diferentes: 111 | anúncios sem registro apesar de contagem positiva: 18
reviews_details.csv | linhas: 431830 | colunas (sem campos de identificação pessoal/conteúdo): ['listing_id', 'id', 'date'] | IDs de avaliação: 431830 únicos, 0 repetidos | ausências: {'comments': 630} | datas inválidas: 0 | linhas sem vínculo com anúncio: 0
  comparação com number_of_reviews: anúncios comparáveis: 17624 | contagens iguais: 17513 | diferentes: 111 | anúncios sem registro apesar de contagem positiva: 18
Comparação por listing_id + date: linhas pareadas: 431830 | somente em reviews.csv: 0 | somente em reviews_details.csv: 0 | combinações com contagem diferente: 0


#### Interpretação das verificações 02.4–02.5

O calendário contém 7.310.950 linhas: 365 datas únicas por anúncio, sem chaves duplicadas, IDs órfãos ou datas fora do período verificado. `available='f'` coincide com preço ausente e `available='t'` com preço preenchido; isso descreve disponibilidade no calendário e não confirma reserva ou venda. Os preços futuros do calendário ficam fora dos preditores.

As duas bases de avaliações têm 431.830 linhas e coincidem integralmente pela contagem da chave `listing_id + date`. `reviews_details.csv` tem IDs de avaliação únicos e 630 comentários ausentes; não foram encontrados IDs de anúncio órfãos nem datas inválidas. Em 111 dos 17.624 anúncios comparáveis, a contagem difere de `number_of_reviews`; outros 18 anúncios com contagem positiva não têm linhas nessas bases. A divergência deve ser preservada como ressalva, sem presumir a causa. `number_of_reviews` é histórico e não representa avaliações dos últimos 12 meses.

### 02.6 — Plano de tratamento e proteção da partição de teste

Usar `listings_details.csv` do snapshot de 06–07/12/2018, uma linha por `id` e `price` como alvo. A partição 70/15/15 foi criada no início da Etapa 02, antes das auditorias. O teste bruto está salvo em `data/processed/df_test_raw.csv` e não será limpo nem usado para decisões nesta etapa.

Para a base de desenvolvimento, manter features candidatas com menos de 30% de ausências e aplicar as regras já justificadas: Amsterdam pelos bairros oficiais, preço positivo, exclusão dos três erros de `bathrooms >= 10` e remoção de linhas incompletas. Limpar somente treino e validação; não juntar linhas diárias do calendário. Valores altos de preço, `minimum_nights` e `maximum_nights` não serão removidos apenas por IQR.

In [13]:
# 1. Confere a partição criada no início do Data Quality, sem sortear novamente.
split_id_counts = {
    "treino": len(train_listing_ids),
    "validação": len(validation_listing_ids),
    "teste": len(test_listing_ids)
}
# Recalcula sobreposição e cobertura usando os conjuntos já congelados.
split_overlap = (
    bool(train_listing_ids & validation_listing_ids)
    or bool(train_listing_ids & test_listing_ids)
    or bool(validation_listing_ids & test_listing_ids)
)
split_coverage_ok = (
    train_listing_ids | validation_listing_ids | test_listing_ids
) == set(all_listing_ids)

# 2. Interrompe a auditoria se a partição inicial não continuar íntegra.
if split_overlap or not split_coverage_ok:
    # Evita continuar com IDs repetidos ou sem partição.
    raise ValueError("A partição inicial perdeu disjunção ou cobertura.")

# 3. Confirma que o teste segue salvo e com o tamanho original esperado.
test_file_exists = df_test_raw_path.exists()
test_file_bytes = df_test_raw_path.stat().st_size if test_file_exists else 0
if len(df_test) != len(test_listing_ids) or not test_file_exists:
    # Impede que a etapa continue se o holdout bruto estiver incompleto.
    raise ValueError("O teste bruto não está íntegro ou não foi salvo.")

# 4. Exibe a partição congelada e o registro do teste bruto.
print("Partição definida em 02.0:", split_id_counts)
print("Sobreposição:", split_overlap, "| cobertura completa:", split_coverage_ok)
print(
    "Teste bruto preservado:",
    df_test.shape,
    "| arquivo:",
    df_test_raw_path,
    "| bytes:",
    test_file_bytes
)

Partição definida em 02.0: {'treino': 14021, 'validação': 3004, 'teste': 3005}
Sobreposição: False | cobertura completa: True
Teste bruto preservado: (3005, 96) | arquivo: D:\repos\projetos\airbnb_price\data\processed\df_test_raw.csv | bytes: 11634013


#### Interpretação da partição

A partição original cobre os 20.030 anúncios: 14.021 treino, 3.004 validação e 3.005 teste. Os conjuntos são disjuntos e usam a semente 42. O `df_test` bruto, com 96 colunas, está salvo em `data/processed/df_test_raw.csv`; a limpeza desta etapa será limitada a treino e validação. O teste será preparado somente depois de congelar o modelo e as regras da avaliação.

### 02.7 — Resumo da auditoria inicial

- A base detalhada do snapshot tem 20.030 anúncios, IDs únicos, 96 colunas e 54 campos com ausências.
- A análise confirmou a escala das notas, os limites de `availability_365`, os formatos de datas e booleanos, e a granularidade das bases auxiliares.
- O calendário e as avaliações foram auditados sem interpretar indisponibilidade como reserva ou usar preços futuros como preditores.
- A limpeza, as decisões sobre extremos e a tabela em memória são registradas nas verificações 02.8–02.10; o gate final fica ao término da Etapa 02.

### 02.8 — Escopo geográfico e seleção preliminar de features

O escopo usa os nomes de bairros geocodificados do anúncio, comparados com `neighbourhoods.csv` e `neighbourhoods.geojson`. Os campos livres `city` e `market` têm ruído: valores vazios ou diferentes de Amsterdam só serão excluídos se também não corresponderem a um bairro oficial. A lista de features inclui as 17 candidatas anteriores e `maximum_nights`, solicitado nesta revisão; sua concentração em 1.125 será preservada para a Etapa 03 decidir a codificação.

In [14]:
# 1. Importa ferramentas para contar valores de localização.
# Importa csv para percorrer as linhas do snapshot.
import csv
# Importa json para ler o GeoJSON oficial dos bairros.
import json
# Importa Counter para resumir os valores geográficos.
from collections import Counter
# Importa Decimal e InvalidOperation para validar coordenadas.
from decimal import Decimal, InvalidOperation

# 2. Define os campos geográficos disponíveis para inspeção.
location_fields = [
    field for field in ["city", "market", "state", "country", "neighbourhood_cleansed"]
    if field in stats_columns
]
# Prepara um contador para os valores de cada campo geográfico.
location_counts = {field: Counter() for field in location_fields}
# Inicializa os intervalos observados das coordenadas.
latitude_values = []
longitude_values = []
# Inicializa contadores de coordenadas vazias ou inválidas.
coordinate_missing = Counter()

# 3. Percorre a base para contar localizações e validar coordenadas.
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio como um dicionário.
    stats_reader = csv.DictReader(csv_file)
    # Analisa todos os registros do snapshot.
    for stats_row in stats_reader:
        # Conta os valores de cidade, mercado, país, estado e bairro.
        for location_field in location_fields:
            # Normaliza espaços e marca valores vazios de forma explícita.
            location_value = (stats_row.get(location_field) or "").strip()
            # Atualiza a frequência do valor observado.
            location_counts[location_field][location_value or "<VAZIO>"] += 1
        # Tenta converter latitude e longitude para validar seus intervalos.
        for coordinate_field, coordinate_values in [
            ("latitude", latitude_values),
            ("longitude", longitude_values)
        ]:
            # Lê a coordenada como texto.
            coordinate_text = (stats_row.get(coordinate_field) or "").strip()
            # Registra ausência quando a coordenada não está preenchida.
            if not coordinate_text:
                # Soma uma ausência para o campo geográfico correspondente.
                coordinate_missing[coordinate_field] += 1
                # Passa à próxima coordenada.
                continue
            # Converte o texto para número decimal.
            try:
                # Valida que a coordenada representa um número finito.
                coordinate_value = Decimal(coordinate_text)
                # Guarda coordenadas válidas para resumir a faixa.
                if coordinate_value.is_finite():
                    # Converte para float para manter o mesmo tipo das estatísticas.
                    coordinate_values.append(float(coordinate_value))
                # Registra valores numéricos não finitos como inválidos.
                else:
                    # Soma uma falha de coordenada.
                    coordinate_missing[coordinate_field] += 1
            # Conta falhas de conversão sem interromper a auditoria.
            except (InvalidOperation, ValueError, OverflowError):
                # Soma uma falha de coordenada.
                coordinate_missing[coordinate_field] += 1

# 4. Exibe as frequências de localização em ordem de maior ocorrência.
for location_field in location_fields:
    # Mostra todos os valores de cidade/mercado/país/estado ou bairro.
    print(location_field, "|", location_counts[location_field].most_common())

# 5. Inspeciona mercados diferentes de Amsterdam e confere o bairro geocodificado.
non_amsterdam_markets = []
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê anúncios para associar o campo market ao bairro e às coordenadas.
    stats_reader = csv.DictReader(csv_file)
    # Percorre cada anúncio que não está marcado explicitamente como Amsterdam.
    for stats_row in stats_reader:
        # Normaliza o campo market para comparação.
        market_value = (stats_row.get("market") or "").strip()
        # Guarda market vazio ou diferente do nome canônico da cidade.
        if market_value.casefold() != "amsterdam":
            # Registra campos que ajudam a verificar se o anúncio está dentro do mapa.
            non_amsterdam_markets.append((
                stats_row.get("id"),
                market_value or "<VAZIO>",
                stats_row.get("neighbourhood_cleansed"),
                stats_row.get("latitude"),
                stats_row.get("longitude")
            ))
# Mostra quantidade e exemplos de market fora do valor canônico.
print()
print("Market vazio ou diferente de Amsterdam:", len(non_amsterdam_markets))
print("Exemplos com bairro e coordenadas:", non_amsterdam_markets[:60])

# 6. Confirma os nomes de bairros com o CSV auxiliar e o GeoJSON de Amsterdam.
neighbourhood_csv_values = set()
with (raw_dir / "neighbourhoods.csv").open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê os nomes oficiais de bairros da lista auxiliar.
    neighbourhood_reader = csv.DictReader(csv_file)
    # Percorre cada bairro cadastrado.
    for neighbourhood_row in neighbourhood_reader:
        # Guarda o nome de bairro preenchido.
        neighbourhood_name = (neighbourhood_row.get("neighbourhood") or "").strip()
        if neighbourhood_name:
            # Adiciona o bairro à lista de referências.
            neighbourhood_csv_values.add(neighbourhood_name)
with (raw_dir / "neighbourhoods.geojson").open("r", encoding="utf-8") as geojson_file:
    # Carrega os polígonos oficiais de bairros.
    neighbourhood_geojson = json.load(geojson_file)
# Extrai os nomes dos bairros dos polígonos do GeoJSON.
geojson_neighbourhoods = {
    (feature.get("properties", {}).get("neighbourhood") or "").strip()
    for feature in neighbourhood_geojson.get("features", [])
}
# Remove propriedades sem nome.
geojson_neighbourhoods.discard("")
# Usa o conjunto do GeoJSON como referência geográfica dos bairros.
amsterdam_neighbourhoods = geojson_neighbourhoods
# Confere se as referências CSV e GeoJSON têm os mesmos nomes.
print()
print(
    "Bairros de referência | CSV:", len(neighbourhood_csv_values),
    "| GeoJSON:", len(geojson_neighbourhoods),
    "| divergências:",
    sorted(neighbourhood_csv_values ^ geojson_neighbourhoods)
)

# 7. Resume as faixas de coordenadas antes de aplicar o escopo geográfico.
print(
    "\nLatitude | válidas:", len(latitude_values),
    "| mínimo/máximo:", min(latitude_values), "/", max(latitude_values),
    "| vazias/inválidas:", coordinate_missing["latitude"]
)
print(
    "Longitude | válidas:", len(longitude_values),
    "| mínimo/máximo:", min(longitude_values), "/", max(longitude_values),
    "| vazias/inválidas:", coordinate_missing["longitude"]
)

# 8. Conta os anúncios que não correspondem a um bairro de Amsterdam.
outside_amsterdam_ids = []
with stats_path.open("r", encoding="utf-8-sig", newline="") as csv_file:
    # Lê cada anúncio para conferir seu bairro geocodificado.
    stats_reader = csv.DictReader(csv_file)
    # Mantém como fora do escopo bairros vazios ou não mapeados no GeoJSON.
    for stats_row in stats_reader:
        # Lê o bairro geocodificado do anúncio.
        neighbourhood_name = (stats_row.get("neighbourhood_cleansed") or "").strip()
        if neighbourhood_name not in amsterdam_neighbourhoods:
            # Registra o ID para auditoria antes da exclusão.
            outside_amsterdam_ids.append(stats_row.get("id"))
# Resume correspondências e anúncios fora dos bairros oficiais.
print(
    "Anúncios mapeados em bairros de Amsterdam:",
    stats_row_count - len(outside_amsterdam_ids),
    "| fora do escopo geográfico:",
    len(outside_amsterdam_ids),
    "| IDs:",
    outside_amsterdam_ids[:20]
)
# Conta quantos markets não canônicos apontam para bairros oficiais.
noncanonical_market_valid_neighbourhoods = sum(
    market_record[2] in amsterdam_neighbourhoods
    for market_record in non_amsterdam_markets
)
print(
    "Market vazio/diferente de Amsterdam com bairro oficial:",
    noncanonical_market_valid_neighbourhoods,
    "de",
    len(non_amsterdam_markets)
)

# 9. Mede as ausências nas features candidatas e aplica o limite estrito de 30%.
candidate_features = [
    "room_type", "property_type", "accommodates", "bedrooms", "bathrooms",
    "neighbourhood_cleansed", "latitude", "longitude", "host_is_superhost",
    "host_since", "host_listings_count", "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "minimum_nights", "maximum_nights",
    "availability_365", "instant_bookable"
]
selected_features = []
print("\nFeatures candidatas e percentual ausente:")
for candidate_name in candidate_features:
    # Soma células vazias e marcadores textuais de ausência.
    candidate_missing = stats_blank_counts[candidate_name] + stats_marker_counts[candidate_name]
    # Soma falhas de parsing somente em campos numéricos.
    if candidate_name in stats_numeric_columns:
        # Conta números inválidos como ausentes para features numéricas.
        candidate_missing += stats_parse_failures[candidate_name]
    # Calcula o percentual ausente em relação ao snapshot completo.
    candidate_missing_pct = candidate_missing / stats_row_count * 100
    # Mantém somente campos existentes com menos de 30% de ausências.
    if candidate_name in stats_columns and candidate_missing_pct < 30:
        # Inclui a feature aprovada no conjunto preliminar.
        selected_features.append(candidate_name)
    # Exibe a taxa de ausências e o resultado do corte.
    print(
        candidate_name,
        "| ausente:", f"{candidate_missing_pct:.2f}%",
        "| mantida:", candidate_name in selected_features
    )
# Resume as features aprovadas pelo corte preliminar.
print("\nFeatures selecionadas pelo corte <30%:", selected_features)
print("Quantidade de features selecionadas:", len(selected_features))

city | [('Amsterdam', 19725), ('Amsterdam-Zuidoost', 193), ('Amsterdam Zuid-Oost', 35), ('Badhoevedorp', 9), ('Diemen', 8), ('De Pijp', 7), ('Amstelveen', 7), ('Watergraafsmeer', 6), ('Duivendrecht', 6), ('Jordaan', 5), ('<VAZIO>', 4), ('Halfweg', 3), ('Amsterdam (Zunderdorp)', 3), ('Zeeburg', 2), ('Amsterdam Zuidoost', 2), ('Ámsterdam', 1), ('Jordaan, Amsterdam', 1), ('Amsterdão', 1), ('Nederland', 1), ('Амстердам', 1), ('Haarlem', 1), ('Amsterdam West', 1), ('Lijnden', 1), ('Amsterdam, Noord-Holland, NL', 1), ('1015 SN Amsterdam', 1), ('Zeeburgerkade', 1), ('Weesp', 1), ('Buitenveldert', 1), ('Ouderkerk aan de Amstel', 1), ('Zaandam', 1)]
market | [('Amsterdam', 19981), ('<VAZIO>', 42), ('Other (International)', 5), ('Barcelona', 1), ('Vilnius', 1)]
state | [('Noord-Holland', 15579), ('North Holland', 2951), ('NH', 1278), ('<VAZIO>', 127), ('Noord Holland', 15), ('Netherlands', 15), ('Amsterdam', 7), ('Badhoevedorp', 7), ('XH', 2), ('NL', 2), ('Bilderdijkstraat', 2), ('Noordholland',

### 02.9 — Limpeza apenas da base de desenvolvimento

Aplicar a limpeza somente aos anúncios de treino e validação, usando `id` e a partição congelada em 02.0. Manter apenas `id`, `price` e as 18 features abaixo de 30% de ausências; filtrar bairros oficiais de Amsterdam, excluir `price <= 0`, os três erros registrados de `bathrooms >= 10` e linhas incompletas. O teste não entra nesta transformação: `df_test` e `data/processed/df_test_raw.csv` permanecem brutos para a avaliação futura.

In [15]:
# 1. Importa pandas para preparar a base de desenvolvimento em memória.
import pandas as pd

# 2. Define as colunas necessárias para a modelagem.
model_columns = ["id", "price"] + selected_features

# 3. Lê as colunas selecionadas e vincula os IDs à partição inicial.
df_source_selected = pd.read_csv(
    listings_path,
    usecols=model_columns,
    dtype={"id": "string"}
)
df_source_selected["particao"] = df_source_selected["id"].map(partition_by_id)

# 4. Confere a atribuição dos IDs antes de excluir o teste do processamento.
unassigned_source_ids = int(df_source_selected["particao"].isna().sum())
if unassigned_source_ids:
    # Interrompe a limpeza caso algum anúncio não pertença às partições congeladas.
    raise ValueError(f"IDs sem partição: {unassigned_source_ids}")

# 5. Retém somente treino e validação; o teste permanece na cópia bruta separada.
development_mask = df_source_selected["particao"].isin(["treino", "validação"])
df_stage02_source = df_source_selected.loc[development_mask].copy()
del df_source_selected
source_row_count = len(df_stage02_source)
development_partition_counts = (
    df_stage02_source["particao"]
    .value_counts()
    .reindex(["treino", "validação"], fill_value=0)
    .to_dict()
)

# 6. Converte o preço textual para valor numérico, preservando a moeda do snapshot.
df_stage02_source["price"] = (
    df_stage02_source["price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace("€", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_stage02_source["price"] = pd.to_numeric(
    df_stage02_source["price"],
    errors="coerce"
)

# 7. Define as features quantitativas e converte seus valores.
numeric_features = [
    "accommodates", "bedrooms", "bathrooms", "latitude", "longitude",
    "host_listings_count", "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "minimum_nights", "maximum_nights",
    "availability_365"
]
for numeric_feature in numeric_features:
    # Converte apenas as features numéricas que passaram pelo corte de ausências.
    if numeric_feature in selected_features:
        # Trata textos não numéricos como valores ausentes para a limpeza.
        df_stage02_source[numeric_feature] = pd.to_numeric(
            df_stage02_source[numeric_feature],
            errors="coerce"
        )

# 8. Converte host_since para data, mantendo ausências como NaT.
df_stage02_source["host_since"] = pd.to_datetime(
    df_stage02_source["host_since"],
    errors="coerce"
)

# 9. Mantém somente bairros oficiais de Amsterdam na base de desenvolvimento.
amsterdam_mask = df_stage02_source["neighbourhood_cleansed"].isin(
    amsterdam_neighbourhoods
)
outside_amsterdam_count = int((~amsterdam_mask).sum())
df_amsterdam = df_stage02_source.loc[amsterdam_mask].copy()

# 10. Registra e exclui preços não positivos e alvos ausentes.
zero_price_ids = (
    df_amsterdam.loc[df_amsterdam["price"] == 0, "id"]
    .astype("string")
    .tolist()
)
nonpositive_price_ids = (
    df_amsterdam.loc[df_amsterdam["price"] <= 0, "id"]
    .astype("string")
    .tolist()
)
missing_price_count = int(df_amsterdam["price"].isna().sum())
df_positive_price = df_amsterdam.loc[df_amsterdam["price"] > 0].copy()

# 11. Remove os três erros registrados de bathrooms sem alterar os dados brutos.
bathroom_error_ids = (
    df_positive_price.loc[df_positive_price["bathrooms"] >= 10, "id"]
    .astype("string")
    .tolist()
)
df_plausible_bathrooms = df_positive_price.loc[
    ~(df_positive_price["bathrooms"] >= 10)
].copy()

# 12. Inspeciona o anúncio de preço extremo somente se estiver no desenvolvimento.
high_price_partition = partition_by_id.get("11040717")
if high_price_partition in {"treino", "validação"}:
    # Localiza o anúncio dentro da base que pode ser limpa nesta etapa.
    high_price_candidate = df_plausible_bathrooms.loc[
        df_plausible_bathrooms["id"] == "11040717"
    ]
    if not high_price_candidate.empty:
        # Lista features ausentes antes de aplicar casos completos.
        high_price_missing_mask = high_price_candidate[selected_features].isna().iloc[0]
        print(
            "Anúncio 11040717 no desenvolvimento | ausentes:",
            high_price_missing_mask[high_price_missing_mask].index.tolist()
        )
else:
    # Evita consultar atributos do anúncio se ele pertencer ao holdout de teste.
    print("Anúncio 11040717 está no teste; preservado sem inspeção nesta etapa.")

# 13. Remove linhas sem alvo ou feature selecionada.
complete_case_columns = ["price"] + selected_features
complete_case_mask = df_plausible_bathrooms[complete_case_columns].notna().all(axis=1)
missing_feature_ids = (
    df_plausible_bathrooms.loc[~complete_case_mask, "id"]
    .astype("string")
    .tolist()
)
df_modeling_pre_split = df_plausible_bathrooms.loc[complete_case_mask].copy()
df_modeling_pre_split = df_modeling_pre_split[model_columns + ["particao"]]

# 14. Resume as regras aplicadas somente à base de desenvolvimento.
print("Linhas de treino e validação antes da limpeza:", source_row_count)
print("Partições antes da limpeza:", development_partition_counts)
print("Fora dos bairros oficiais de Amsterdam:", outside_amsterdam_count)
print(
    "Preço zero:", len(zero_price_ids),
    "| IDs:", zero_price_ids,
    "| preços <= 0:", len(nonpositive_price_ids),
    "| alvo ausente/inválido:", missing_price_count
)
print(
    "Erros de bathrooms >= 10:",
    len(bathroom_error_ids),
    "| IDs:",
    bathroom_error_ids
)
print(
    "Linhas incompletas removidas:",
    len(missing_feature_ids),
    "| exemplos de IDs:",
    missing_feature_ids[:10]
)
print(
    "Base de desenvolvimento após a limpeza:",
    df_modeling_pre_split.shape
)
print(
    "Partições retidas:",
    df_modeling_pre_split["particao"].value_counts().to_dict()
)
print("Features da tabela:", selected_features)

Anúncio 11040717 no desenvolvimento | ausentes: ['reviews_per_month', 'review_scores_rating']
Linhas de treino e validação antes da limpeza: 17025
Partições antes da limpeza: {'treino': 14021, 'validação': 3004}
Fora dos bairros oficiais de Amsterdam: 0
Preço zero: 2 | IDs: ['20597844', '21423332'] | preços <= 0: 2 | alvo ausente/inválido: 0
Erros de bathrooms >= 10: 3 | IDs: ['15199899', '16698560', '17245735']
Linhas incompletas removidas: 2214 | exemplos de IDs: ['28871', '62801', '65230', '291752', '308028', '432610', '515088', '517092', '626872', '652660']
Base de desenvolvimento após a limpeza: (14806, 21)
Partições retidas: {'treino': 12187, 'validação': 2619}
Features da tabela: ['room_type', 'property_type', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'latitude', 'longitude', 'host_is_superhost', 'host_since', 'host_listings_count', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'minimum_nights', 'maximum_nights', 'availability_365', '

#### 02.9.1 — Interpretação e registro da limpeza do desenvolvimento

- **Escopo geográfico:** na auditoria do snapshot completo, os 20.030 anúncios correspondem aos 22 bairros oficiais. Na base de desenvolvimento, nenhum anúncio foi removido por geografia; os campos livres `city` e `market` não definem o escopo.
- **Features:** 18 features passaram pelo corte estrito de menos de 30% de ausências, incluindo `maximum_nights`.
- **Limpeza aplicada somente a treino e validação:** dos 17.025 anúncios dessas partições, foram excluídos 2 com `price=0` (IDs `20597844`, `21423332`), 3 com `bathrooms >= 10` (IDs `15199899`, `16698560`, `17245735`) e 2.214 linhas incompletas. Restaram 14.806 anúncios: 12.187 de treino e 2.619 de validação.
- **Teste reservado:** os 3.005 anúncios de teste, com 96 colunas, estão em `df_test` e em `data/processed/df_test_raw.csv` como valores textuais originais. Não receberam filtros nem conversões nesta etapa. A limpeza será aplicada durante a avaliação, depois que as decisões do modelo forem congeladas.
- **Preço extremo:** o anúncio `11040717` está no desenvolvimento e foi excluído somente por valores ausentes em `reviews_per_month` e `review_scores_rating`; não foi classificado como erro de preço. O bruto original permanece preservado.
- **Fallback de bathrooms:** não alterar os dados brutos. Se evidência posterior confirmar que algum valor `>=10` é legítimo, reconstruir a linha a partir do snapshot e tratar a incerteza na preparação.
- **maximum_nights:** manter 1.125 e 9.999; a documentação consultada não confirma que sejam erro ou valor padrão. Avaliar o campo na Etapa 03. Manter também `minimum_nights=999` e 1.001; não há conflito com o máximo.

### 02.9.2 — Agregação e auditoria das avaliações por anúncio

Ler somente o identificador do anúncio e a data de reviews.csv. Restringir as métricas aos anúncios de treino e validação, agregar primeiro por anúncio e usar o corte do snapshot em 06/12/2018. Comparar a contagem da fonte com number_of_reviews e inspecionar cobertura antes da junção. Esta célula não altera df_modeling_pre_split, df_modeling_final nem df_test.

In [18]:
# 1. Importa pandas para agregar os eventos de avaliação por anúncio.
import pandas as pd

# 2. Define a data de corte e os IDs somente da base de desenvolvimento.
review_snapshot_date = pd.Timestamp("2018-12-06")
development_ids_for_reviews = set(
    df_modeling_pre_split["id"].astype("string")
)

# 3. Lê apenas o identificador do anúncio e a data, sem importar comentários ou dados pessoais.
reviews_path = raw_dir / "reviews.csv"
reviews_source = pd.read_csv(
    reviews_path,
    usecols=["listing_id", "date"],
    dtype={"listing_id": "string"}
)

# 4. Filtra primeiro os IDs de treino e validação, sem calcular métricas para o teste.
reviews_development = reviews_source.loc[
    reviews_source["listing_id"].isin(development_ids_for_reviews)
].copy()
del reviews_source

# 5. Valida datas somente nos eventos associados à base de desenvolvimento.
reviews_development["date"] = pd.to_datetime(
    reviews_development["date"],
    errors="coerce"
)
invalid_review_dates = int(reviews_development["date"].isna().sum())
events_after_snapshot = int(
    (
        reviews_development["date"].notna()
        & reviews_development["date"].gt(review_snapshot_date)
    ).sum()
)
reviews_development = reviews_development.loc[
    reviews_development["date"].notna()
    & reviews_development["date"].le(review_snapshot_date)
].copy()

# 6. Resume o total de eventos e a data da última avaliação por anúncio.
review_history_by_listing = (
    reviews_development
    .groupby("listing_id")
    .agg(
        review_rows_in_source=("date", "size"),
        last_review_date_aux=("date", "max")
    )
)

# 7. Conta eventos na janela móvel de 365 dias que termina no snapshot.
recent_review_start = review_snapshot_date - pd.Timedelta(days=365)
recent_reviews = reviews_development.loc[
    reviews_development["date"].gt(recent_review_start)
    & reviews_development["date"].le(review_snapshot_date)
]
reviews_last_365d_by_listing = (
    recent_reviews
    .groupby("listing_id")
    .size()
    .rename("reviews_last_365d")
)
review_history_by_listing = review_history_by_listing.join(
    reviews_last_365d_by_listing,
    how="left"
)

# 8. Calcula os dias desde a avaliação mais recente e prepara uma linha por anúncio.
review_history_by_listing["days_since_last_review"] = (
    review_snapshot_date
    - review_history_by_listing["last_review_date_aux"]
).dt.days
review_features_candidate = review_history_by_listing.reset_index()

# 9. Compara a contagem de eventos da fonte com number_of_reviews no desenvolvimento.
review_count_audit = df_modeling_pre_split[
    ["id", "number_of_reviews"]
].copy()
review_count_audit["review_rows_in_source"] = (
    review_count_audit["id"]
    .map(review_history_by_listing["review_rows_in_source"])
    .fillna(0)
    .astype("int64")
)
review_count_audit["review_count_mismatch"] = (
    review_count_audit["review_rows_in_source"]
    != review_count_audit["number_of_reviews"]
)
positive_count_without_source_rows = (
    (review_count_audit["number_of_reviews"] > 0)
    & (review_count_audit["review_rows_in_source"] == 0)
)

# 10. Exibe cobertura, divergências e dimensões sem consultar o teste.
print("Base de desenvolvimento antes da junção:", df_modeling_pre_split.shape)
print("IDs de desenvolvimento considerados:", len(development_ids_for_reviews))
print("Eventos de review associados ao desenvolvimento:", len(reviews_development))
print("Anúncios com histórico na fonte:", len(review_history_by_listing))
print("Datas inválidas no desenvolvimento:", invalid_review_dates)
print("Eventos posteriores ao snapshot no desenvolvimento:", events_after_snapshot)
print(
    "Contagens divergentes no desenvolvimento:",
    int(review_count_audit["review_count_mismatch"].sum())
)
print(
    "Contagem positiva sem eventos na fonte:",
    int(positive_count_without_source_rows.sum())
)
print("Tabela agregada candidata:", review_features_candidate.shape)
print(
    "IDs duplicados na agregação:",
    int(review_features_candidate["listing_id"].duplicated().sum())
)
print(
    "Sem avaliações nos 365 dias:",
    int(review_features_candidate["reviews_last_365d"].isna().sum())
)
print(
    "Sem data da última avaliação:",
    int(review_features_candidate["days_since_last_review"].isna().sum())
)
print(
    "Contagem de reviews nos 365 dias | mínimo:",
    int(review_features_candidate["reviews_last_365d"].fillna(0).min()),
    "| mediana:",
    float(review_features_candidate["reviews_last_365d"].fillna(0).median()),
    "| máximo:",
    int(review_features_candidate["reviews_last_365d"].fillna(0).max())
)
print("Colunas candidatas:", review_features_candidate.columns.tolist())
print("Data de corte:", review_snapshot_date.date())


Base de desenvolvimento antes da junção: (14806, 21)
IDs de desenvolvimento considerados: 14806
Eventos de review associados ao desenvolvimento: 371840
Anúncios com histórico na fonte: 14806
Datas inválidas no desenvolvimento: 0
Eventos posteriores ao snapshot no desenvolvimento: 0
Contagens divergentes no desenvolvimento: 91
Contagem positiva sem eventos na fonte: 0
Tabela agregada candidata: (14806, 5)
IDs duplicados na agregação: 0
Sem avaliações nos 365 dias: 3557
Sem data da última avaliação: 0
Contagem de reviews nos 365 dias | mínimo: 0 | mediana: 4.0 | máximo: 166
Colunas candidatas: ['listing_id', 'review_rows_in_source', 'last_review_date_aux', 'reviews_last_365d', 'days_since_last_review']
Data de corte: 2018-12-06


### 02.9.3 — Junção das métricas agregadas de avaliações

Criar uma tabela auxiliar com uma linha por anúncio e juntar as métricas por id com validação um-para-um. A falta de eventos dentro da janela de 365 dias vira contagem zero porque todos os anúncios de desenvolvimento têm histórico na fonte. Manter number_of_reviews original e registrar divergências da fonte; não remover linhas nem transformar o teste.

In [19]:
# 1. Guarda uma cópia da base de desenvolvimento para conferir que a junção não a altera.
df_modeling_pre_split_before_review_join = df_modeling_pre_split.copy(deep=True)

# 2. Seleciona somente as duas métricas que entrarão na base de modelagem.
review_features_for_join = review_features_candidate[
    ["listing_id", "reviews_last_365d", "days_since_last_review"]
].copy()

# 3. Converte ausência de evento na janela em zero e define tipos numéricos inteiros.
review_features_for_join["reviews_last_365d"] = (
    review_features_for_join["reviews_last_365d"]
    .fillna(0)
    .astype("int64")
)
review_features_for_join["days_since_last_review"] = (
    review_features_for_join["days_since_last_review"]
    .astype("int64")
)

# 4. Junta as métricas agregadas pelo identificador, validando cardinalidade um-para-um.
df_modeling_enriched = df_modeling_pre_split.merge(
    review_features_for_join,
    how="left",
    left_on="id",
    right_on="listing_id",
    validate="one_to_one",
    indicator=True
)

# 5. Verifica cobertura, quantidade de linhas, IDs, duplicatas e partições.
review_join_unmatched = int(
    (df_modeling_enriched["_merge"] != "both").sum()
)
df_modeling_enriched.drop(
    columns=["listing_id", "_merge"],
    inplace=True
)
review_join_rows_preserved = (
    len(df_modeling_enriched) == len(df_modeling_pre_split)
)
review_join_ids_preserved = (
    df_modeling_enriched["id"].tolist()
    == df_modeling_pre_split["id"].tolist()
)
review_join_duplicate_ids = int(
    df_modeling_enriched["id"].duplicated().sum()
)
review_join_partitions_preserved = (
    df_modeling_enriched["particao"].value_counts().to_dict()
    == df_modeling_pre_split["particao"].value_counts().to_dict()
)
development_source_unchanged = df_modeling_pre_split.equals(
    df_modeling_pre_split_before_review_join
)
review_join_missing = (
    df_modeling_enriched[
        ["reviews_last_365d", "days_since_last_review"]
    ]
    .isna()
    .sum()
    .to_dict()
)

# 6. Interrompe a validação se a junção não preservar os registros do desenvolvimento.
if (
    review_join_unmatched
    or not review_join_rows_preserved
    or not review_join_ids_preserved
    or review_join_duplicate_ids
    or not review_join_partitions_preserved
    or not development_source_unchanged
    or any(review_join_missing.values())
):
    raise ValueError("A junção das métricas de avaliações falhou nas verificações.")

# 7. Exibe o resultado e a distribuição das métricas agregadas.
print("Base enriquecida:", df_modeling_enriched.shape)
print("Linhas de treino/validação preservadas:", review_join_rows_preserved)
print("IDs e ordem preservados:", review_join_ids_preserved)
print("IDs duplicados após a junção:", review_join_duplicate_ids)
print("Anúncios sem correspondência:", review_join_unmatched)
print("Partições preservadas:", review_join_partitions_preserved)
print("Base anterior à junção inalterada:", development_source_unchanged)
print("Ausências nas novas colunas:", review_join_missing)
print(
    "reviews_last_365d | mínimo:",
    int(df_modeling_enriched["reviews_last_365d"].min()),
    "| mediana:",
    float(df_modeling_enriched["reviews_last_365d"].median()),
    "| máximo:",
    int(df_modeling_enriched["reviews_last_365d"].max())
)
print(
    "days_since_last_review | mínimo:",
    int(df_modeling_enriched["days_since_last_review"].min()),
    "| mediana:",
    float(df_modeling_enriched["days_since_last_review"].median()),
    "| máximo:",
    int(df_modeling_enriched["days_since_last_review"].max())
)

Base enriquecida: (14806, 23)
Linhas de treino/validação preservadas: True
IDs e ordem preservados: True
IDs duplicados após a junção: 0
Anúncios sem correspondência: 0
Partições preservadas: True
Base anterior à junção inalterada: True
Ausências nas novas colunas: {'reviews_last_365d': 0, 'days_since_last_review': 0}
reviews_last_365d | mínimo: 0 | mediana: 4.0 | máximo: 166
days_since_last_review | mínimo: 0 | mediana: 101.0 | máximo: 2584


### 02.10 — Base de desenvolvimento enriquecida e validação do teste reservado

Manter `df_modeling_final` em memória com treino e validação limpos, `id`, `price`, as 18 features da base principal, duas métricas agregadas de `reviews.csv` e a partição. As métricas de avaliações usam apenas eventos até 06/12/2018 e não substituem `number_of_reviews`. A partição de teste não entra nessa tabela: `df_test` permanece sem conversões ou filtros e está salvo em `data/processed/df_test_raw.csv`. Conferir integridade da base de desenvolvimento, cobertura das partições e integridade do arquivo reservado para a avaliação futura.

In [20]:
# 1. Importa pandas para validar a tabela de desenvolvimento.
import pandas as pd

# 2. Define as features finais e organiza a tabela com as métricas agregadas.
review_feature_columns = [
    "reviews_last_365d",
    "days_since_last_review"
]
final_feature_columns = selected_features + review_feature_columns
df_modeling_final = df_modeling_enriched[
    ["id", "price"] + final_feature_columns + ["particao"]
].copy()

# 3. Calcula verificações de qualidade somente sobre treino e validação.
final_data_columns = ["price"] + final_feature_columns
final_missing_cells = int(df_modeling_final[final_data_columns].isna().sum().sum())
final_nonpositive_prices = int((df_modeling_final["price"] <= 0).sum())
final_outside_neighbourhoods = int(
    (~df_modeling_final["neighbourhood_cleansed"].isin(amsterdam_neighbourhoods)).sum()
)
final_bathroom_errors = int((df_modeling_final["bathrooms"] >= 10).sum())
final_duplicate_ids = int(df_modeling_final["id"].duplicated().sum())
final_unassigned_partitions = int(df_modeling_final["particao"].isna().sum())
unexpected_development_partitions = sorted(
    set(df_modeling_final["particao"].dropna()) - {"treino", "validação"}
)
final_partition_counts = (
    df_modeling_final["particao"]
    .value_counts()
    .reindex(["treino", "validação"], fill_value=0)
    .to_dict()
)

# 4. Confere a partição congelada e a preservação do teste sem limpar seus valores.
split_overlap = (
    bool(train_listing_ids & validation_listing_ids)
    or bool(train_listing_ids & test_listing_ids)
    or bool(validation_listing_ids & test_listing_ids)
)
split_coverage_ok = (
    train_listing_ids | validation_listing_ids | test_listing_ids
) == set(all_listing_ids)
test_ids_match_partition = set(df_test["id"].tolist()) == test_listing_ids
test_schema_unchanged = df_test.columns.tolist() == df_test_raw_columns
test_values_remain_text = all(str(dtype) == "string" for dtype in df_test.dtypes)
test_has_no_partition_column = "particao" not in df_test.columns
test_file_exists = df_test_raw_path.exists()
test_file_bytes = df_test_raw_path.stat().st_size if test_file_exists else 0
final_test_partition_counts = {
    "treino": len(train_listing_ids),
    "validação": len(validation_listing_ids),
    "teste": len(test_listing_ids)
}

# 5. Interrompe se a base de desenvolvimento ou o teste reservado falhar nos critérios.
if (
    final_duplicate_ids
    or final_missing_cells
    or final_nonpositive_prices
    or final_outside_neighbourhoods
    or final_bathroom_errors
    or final_unassigned_partitions
    or unexpected_development_partitions
    or split_overlap
    or not split_coverage_ok
    or not test_ids_match_partition
    or not test_schema_unchanged
    or not test_values_remain_text
    or not test_has_no_partition_column
    or not test_file_exists
):
    # Evita declarar a etapa pronta se houver falha de integridade ou limpeza.
    raise ValueError("A validação da base de desenvolvimento ou do teste falhou.")

# 6. Exibe estrutura, partições e integridade sem calcular estatísticas do teste.
print("Base de desenvolvimento em memória:", df_modeling_final.shape)
print("Colunas:", df_modeling_final.columns.tolist())
print("Features preditoras:", final_feature_columns)
print("Quantidade de features preditoras:", len(final_feature_columns))
print("IDs duplicados:", final_duplicate_ids)
print("Células ausentes nas features e no alvo:", final_missing_cells)
print("Preços não positivos:", final_nonpositive_prices)
print("Anúncios fora dos bairros oficiais:", final_outside_neighbourhoods)
print("Bathrooms >= 10:", final_bathroom_errors)
print("IDs sem partição:", final_unassigned_partitions)
print("Partições da base de desenvolvimento:", final_partition_counts)
print(
    "Divergências entre number_of_reviews e os eventos da fonte:",
    int(review_count_audit["review_count_mismatch"].sum())
)
print(
    "Contagem positiva sem eventos na fonte:",
    int(positive_count_without_source_rows.sum())
)
print(
    "Partição original | treino:", final_test_partition_counts["treino"],
    "| validação:", final_test_partition_counts["validação"],
    "| teste:", final_test_partition_counts["teste"],
    "| sobreposição:", split_overlap,
    "| cobertura completa:", split_coverage_ok
)
print(
    "df_test preservado | formato:", df_test.shape,
    "| IDs corretos:", test_ids_match_partition,
    "| esquema original:", test_schema_unchanged,
    "| valores ainda textuais:", test_values_remain_text,
    "| coluna de partição adicionada:", not test_has_no_partition_column
)
print(
    "Arquivo bruto do teste:", df_test_raw_path,
    "| existe:", test_file_exists,
    "| bytes:", test_file_bytes
)
print(
    "Preço na base de desenvolvimento | mínimo:", df_modeling_final["price"].min(),
    "| mediana:", df_modeling_final["price"].median(),
    "| máximo:", df_modeling_final["price"].max()
)
print("Versão do pandas:", pd.__version__)
print("Tipos de dados da base de desenvolvimento:")
print(df_modeling_final.dtypes.to_string())

# 7. Mostra uma amostra da base limpa sem exportá-la para outro arquivo.
display(df_modeling_final.head())

Base de desenvolvimento em memória: (14806, 23)
Colunas: ['id', 'price', 'room_type', 'property_type', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'latitude', 'longitude', 'host_is_superhost', 'host_since', 'host_listings_count', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'minimum_nights', 'maximum_nights', 'availability_365', 'instant_bookable', 'reviews_last_365d', 'days_since_last_review', 'particao']
Features preditoras: ['room_type', 'property_type', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'latitude', 'longitude', 'host_is_superhost', 'host_since', 'host_listings_count', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'minimum_nights', 'maximum_nights', 'availability_365', 'instant_bookable', 'reviews_last_365d', 'days_since_last_review']
Quantidade de features preditoras: 20
IDs duplicados: 0
Células ausentes nas features e no alvo: 0
Preços não positivos: 0
Anúncios fora dos bairros oficia

,id,price,room_type,property_type,accommodates,bedrooms,bathrooms,neighbourhood_cleansed,latitude,longitude,...,number_of_reviews,reviews_per_month,review_scores_rating,minimum_nights,maximum_nights,availability_365,instant_bookable,reviews_last_365d,days_since_last_review,particao
0,2818,59.0,Private room,Apartment,2,1.0,1.5,Oostelijk Havengebied - Indische Buurt,52.365755,4.941419,...,248,2.10,97.0,3,15,44,t,30,8,treino
1,3209,160.0,Entire home/apt,Apartment,5,2.0,1.0,Westerpark,52.390225,4.873924,...,42,1.03,96.0,4,20,47,f,8,99,treino
2,20168,80.0,Entire home/apt,Townhouse,2,1.0,1.0,Centrum-Oost,52.365087,4.893541,...,233,2.18,87.0,1,1000,198,f,50,6,treino
3,28658,65.0,Private room,Apartment,2,1.0,1.0,Bos en Lommer,52.375342,4.857289,...,434,4.16,93.0,3,15,295,f,59,17,treino
4,29051,55.0,Private room,Apartment,1,1.0,1.0,Centrum-West,52.367725,4.891512,...,383,4.07,95.0,2,730,188,f,82,1,treino


### 02.10.1 — Resultado final e gate da Etapa 02

#### Resultado

- `df_modeling_final` está disponível em memória com **14.806 linhas e 23 colunas**: `id`, `price`, 20 features preditoras e a partição; contém somente treino e validação limpos.
- As 20 features são as 18 selecionadas de `listings_details.csv` mais `reviews_last_365d` e `days_since_last_review`, agregadas de `reviews.csv` por anúncio.
- A base não tem IDs duplicados, ausências nas features/alvo, preços não positivos, bairros fora do escopo, `bathrooms >= 10` ou IDs sem partição.
- A partição original cobre 20.030 IDs sem sobreposição: **14.021 treino, 3.004 validação e 3.005 teste**. Após a limpeza, a base de desenvolvimento contém **12.187 treino e 2.619 validação**.
- A agregação de avaliações usou a janela encerrada em **06/12/2018**. Foram associados 371.840 eventos aos anúncios de desenvolvimento; os 14.806 anúncios têm histórico na fonte. Não houve datas inválidas, eventos após o snapshot ou IDs sem correspondência na junção.
- `reviews_last_365d` varia de 0 a 166, com mediana 4; **3.557** anúncios não tiveram eventos na janela e receberam zero nessa contagem. `days_since_last_review` varia de 0 a 2.584 dias, com mediana 101. Nenhuma das duas colunas tem ausências.
- A contagem de eventos em `reviews.csv` diverge de `number_of_reviews` em **91** anúncios de desenvolvimento. Nenhum anúncio com `number_of_reviews` positivo ficou sem eventos na fonte. A coluna original `number_of_reviews` foi preservada; divergência não foi tratada como prova de erro.
- O preço na base de desenvolvimento varia de €8 a €8.500, com mediana de €125. Não foi exportado CSV da base de desenvolvimento; os arquivos brutos permanecem intactos.
- `df_test` permanece bruto com **3.005 linhas e 96 colunas**, salvo em `data/processed/df_test_raw.csv` (11.634.013 bytes), sem limpeza, conversão, coluna de partição ou integração de avaliações.

#### Decisões e pendências

- Integrar as métricas de avaliações por `id` após agregação, com validação um-para-um e sem multiplicar anúncios. O total, IDs, ordem e partições da base de desenvolvimento foram preservados.
- Manter `number_of_reviews` da base principal e documentar as 91 divergências com a fonte auxiliar.
- Não juntar `listings.csv` nem `neighbourhoods.csv`, pois não acrescentam colunas preenchidas úteis à base atual; `reviews_details.csv` duplica os eventos e inclui comentários/dados pessoais. `calendar.csv` permanece excluído conforme decisão do usuário.
- Manter preços extremos sem evidência de erro. O ID `11040717` ficou fora somente por duas features ausentes.
- `maximum_nights=1.125` ocorre em muitos anúncios, mas nenhuma fonte consultada confirma que seja valor padrão. Os valores e anúncios são mantidos.
- A separação do teste foi iniciada após auditorias anteriores do snapshot completo; não se afirma independência histórica total das decisões de Data Quality.

#### Estado do projeto

- **Status:** integração de `reviews.csv`, base de desenvolvimento e preservação do teste bruto executadas e inspecionadas; gate revisado da Etapa 02 aprovado pelo usuário ao solicitar a implementação da Etapa 03.
- **Última etapa aprovada:** Etapa 02.
- **Base de desenvolvimento:** `df_modeling_final`, somente em memória.
- **Teste bruto reservado:** `df_test` no kernel e `data/processed/df_test_raw.csv` em disco.
- **Pendência:** executar a Etapa 03 conforme o plano autorizado, mantendo o teste bruto reservado.

**Gate da Etapa 02:** aprovado pelo usuário ao autorizar a implementação da Etapa 03.```

## Etapa 03 — Feature Engineering

Objetivo: criar atributos interpretáveis para estimar o preço anunciado a partir da base limpa de Amsterdam. O gate revisado da Etapa 02 foi aprovado quando o usuário autorizou esta implementação. A entrada é `df_modeling_final` (treino e validação); `df_test` e seu arquivo bruto permanecem reservados. As transformações desta etapa não usam `price` para construir preditores nem consultam preços ou eventos futuros.

### 03.1 — Verificação do contrato da base e prevenção de vazamento

Conferir colunas, dimensões, partições e integridade da base de desenvolvimento antes de criar atributos. Fazer uma cópia independente para a Etapa 03. O teste reservado será verificado apenas pela existência do arquivo, sem leitura de valores.

In [21]:
# 1. Importa as bibliotecas usadas nesta etapa e registra suas versões.
import pandas as pd
import numpy as np

# 2. Confere se a base aprovada da Etapa 02 continua disponível no kernel.
if "df_modeling_final" not in globals():
    raise RuntimeError("df_modeling_final não está no kernel; recarregue a Etapa 02 pelo MCP Jupyter.")

# 3. Define o contrato observado na execução final da Etapa 02.
expected_stage03_columns = [
    "id", "price", "room_type", "property_type", "accommodates",
    "bedrooms", "bathrooms", "neighbourhood_cleansed", "latitude",
    "longitude", "host_is_superhost", "host_since", "host_listings_count",
    "number_of_reviews", "reviews_per_month", "review_scores_rating",
    "minimum_nights", "maximum_nights", "availability_365",
    "instant_bookable", "reviews_last_365d", "days_since_last_review",
    "particao"
]
stage03_source_features = [
    column for column in expected_stage03_columns
    if column not in ["id", "price", "particao"]
]
stage03_partition_counts = df_modeling_final["particao"].value_counts().to_dict()

# 4. Interrompe se a base, a partição ou o arquivo reservado divergirem do contrato.
if (
    df_modeling_final.shape != (14806, 23)
    or df_modeling_final.columns.tolist() != expected_stage03_columns
    or stage03_partition_counts != {"treino": 12187, "validação": 2619}
    or not df_modeling_final["id"].is_unique
    or df_modeling_final[["price"] + stage03_source_features].isna().any().any()
    or (df_modeling_final["price"] <= 0).any()
    or "df_test_raw_path" not in globals()
    or not df_test_raw_path.exists()
):
    raise ValueError("A base de entrada da Etapa 03 não atende ao contrato aprovado.")

# 5. Guarda a entrada intacta e cria a cópia de trabalho para novas features.
df_modeling_stage02_reference = df_modeling_final.copy(deep=True)
df_modeling_features = df_modeling_final.copy(deep=True)
stage03_seed = 42

# 6. Mostra apenas o contrato de desenvolvimento e metadados do teste reservado.
print("Entrada da Etapa 03:", df_modeling_final.shape)
print("Partições:", stage03_partition_counts)
print("Features de entrada:", len(stage03_source_features))
print("IDs duplicados:", int(df_modeling_final["id"].duplicated().sum()))
print("Preço ausente/não positivo:", int(df_modeling_final["price"].isna().sum()), int((df_modeling_final["price"] <= 0).sum()))
print("Cópia de trabalho independente:", df_modeling_features is not df_modeling_final)
print("Arquivo de teste bruto presente:", df_test_raw_path.exists())
print("Semente:", stage03_seed)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

Entrada da Etapa 03: (14806, 23)
Partições: {'treino': 12187, 'validação': 2619}
Features de entrada: 20
IDs duplicados: 0
Preço ausente/não positivo: 0 0
Cópia de trabalho independente: True
Arquivo de teste bruto presente: True
Semente: 42
pandas: 2.3.3 | numpy: 2.2.6


### 03.2 — Verificação da capacidade e configuração do imóvel

Criar sinais de quarto não separado, quartos e banheiros por hóspede e uma transformação logarítmica da capacidade. Conferir denominadores e valores negativos antes do cálculo. `is_studio` significa apenas `bedrooms == 0`: o nome é uma convenção analítica, não comprova o tipo real do imóvel.

In [22]:
# 1. Audita os domínios usados nas features de capacidade.
invalid_accommodates = int(df_modeling_features["accommodates"].le(0).sum())
invalid_bedrooms = int(df_modeling_features["bedrooms"].lt(0).sum())
invalid_bathrooms = int(df_modeling_features["bathrooms"].lt(0).sum())
if invalid_accommodates or invalid_bedrooms or invalid_bathrooms:
    raise ValueError("Há valores não positivos ou negativos nas colunas de capacidade.")

# 2. Cria o indicador de anúncio sem quarto separado.
df_modeling_features["is_studio"] = (
    df_modeling_features["bedrooms"].eq(0).astype("int8")
)

# 3. Calcula razões somente porque a capacidade positiva foi confirmada.
df_modeling_features["bedrooms_per_guest"] = (
    df_modeling_features["bedrooms"] / df_modeling_features["accommodates"]
)
df_modeling_features["bathrooms_per_guest"] = (
    df_modeling_features["bathrooms"] / df_modeling_features["accommodates"]
)

# 4. Preserva a capacidade original e acrescenta uma escala logarítmica.
df_modeling_features["accommodates_log1p"] = np.log1p(
    df_modeling_features["accommodates"]
)

# 5. Confere ausências, infinitos e distribuição das novas colunas.
capacity_features = [
    "is_studio", "bedrooms_per_guest", "bathrooms_per_guest",
    "accommodates_log1p"
]
capacity_missing = df_modeling_features[capacity_features].isna().sum().to_dict()
capacity_infinite = int(
    np.isinf(df_modeling_features[capacity_features].to_numpy(dtype=float)).sum()
)
if any(capacity_missing.values()) or capacity_infinite:
    raise ValueError("As novas features de capacidade geraram ausências ou infinitos.")
print("Capacidade <= 0:", invalid_accommodates)
print("Quartos negativos:", invalid_bedrooms, "| banheiros negativos:", invalid_bathrooms)
print("Anúncios com bedrooms=0:", int(df_modeling_features["is_studio"].sum()))
print("Ausências nas novas features:", capacity_missing)
print("Infinitos nas novas features:", capacity_infinite)
print(
    df_modeling_features[capacity_features]
    .agg(["min", "median", "max"])
    .to_string()
)
display(
    df_modeling_features[
        ["accommodates", "bedrooms", "bathrooms"] + capacity_features
    ].head(5)
)

Capacidade <= 0: 0
Quartos negativos: 0 | banheiros negativos: 0
Anúncios com bedrooms=0: 806
Ausências nas novas features: {'is_studio': 0, 'bedrooms_per_guest': 0, 'bathrooms_per_guest': 0, 'accommodates_log1p': 0}
Infinitos nas novas features: 0
        is_studio  bedrooms_per_guest  bathrooms_per_guest  accommodates_log1p
min           0.0                 0.0                  0.0            0.693147
median        0.0                 0.5                  0.5            1.098612
max           1.0                 4.0                  2.0            2.890372


,accommodates,bedrooms,bathrooms,is_studio,bedrooms_per_guest,bathrooms_per_guest,accommodates_log1p
0,2,1.0,1.5,0,0.5,0.75,1.098612
1,5,2.0,1.0,0,0.4,0.20,1.791759
2,2,1.0,1.0,0,0.5,0.50,1.098612
3,2,1.0,1.0,0,0.5,0.50,1.098612
4,1,1.0,1.0,0,1.0,1.00,0.693147


**Leitura de 03.2.** Nenhum anúncio de desenvolvimento tem `accommodates <= 0`, quartos negativos ou banheiros negativos. As quatro features foram criadas sem ausências ou infinitos. Há 806 anúncios com `bedrooms=0`; o indicador não deve ser lido como confirmação de que todos sejam estúdios.

### 03.3 — Verificação de anfitrião e reputação

Usar a data fixa do snapshot para a experiência do anfitrião. Aplicar `log1p` apenas a contagens não negativas e converter os booleanos `t/f` após validar seu domínio. As métricas derivadas de avaliações representam histórico observado, não reservas.

In [23]:
# 1. Fixa a data de referência e audita valores inválidos antes da engenharia.
stage03_snapshot_date = pd.Timestamp("2018-12-06")
future_host_dates = int(df_modeling_features["host_since"].gt(stage03_snapshot_date).sum())
missing_host_dates = int(df_modeling_features["host_since"].isna().sum())
negative_host_portfolios = int(df_modeling_features["host_listings_count"].lt(0).sum())
negative_review_counts = int(df_modeling_features["number_of_reviews"].lt(0).sum())
negative_review_rates = int(df_modeling_features["reviews_per_month"].lt(0).sum())
superhost_values = set(df_modeling_features["host_is_superhost"].dropna().unique())
instant_values = set(df_modeling_features["instant_bookable"].dropna().unique())
if (
    future_host_dates or missing_host_dates or negative_host_portfolios
    or negative_review_counts or negative_review_rates
    or superhost_values != {"t", "f"}
    or instant_values != {"t", "f"}
):
    raise ValueError("Datas, contagens ou booleanos inválidos para 03.3.")

# 2. Calcula experiência do anfitrião usando somente o snapshot.
df_modeling_features["host_tenure_years"] = (
    (stage03_snapshot_date - df_modeling_features["host_since"]).dt.days / 365.25
)

# 3. Cria sinais de tamanho do portfólio e volume de avaliações.
df_modeling_features["host_listings_log1p"] = np.log1p(
    df_modeling_features["host_listings_count"]
)
df_modeling_features["host_has_multiple_listings"] = (
    df_modeling_features["host_listings_count"].gt(1).astype("int8")
)
df_modeling_features["number_of_reviews_log1p"] = np.log1p(
    df_modeling_features["number_of_reviews"]
)
df_modeling_features["reviews_per_month_log1p"] = np.log1p(
    df_modeling_features["reviews_per_month"]
)

# 4. Padroniza os dois campos booleanos sem substituir as colunas originais.
df_modeling_features["is_superhost"] = (
    df_modeling_features["host_is_superhost"]
    .map({"t": 1, "f": 0})
    .astype("int8")
)
df_modeling_features["is_instant_bookable"] = (
    df_modeling_features["instant_bookable"]
    .map({"t": 1, "f": 0})
    .astype("int8")
)

# 5. Confere faixas, ausências e infinitos nas novas colunas.
host_reputation_features = [
    "host_tenure_years", "host_listings_log1p",
    "host_has_multiple_listings", "number_of_reviews_log1p",
    "reviews_per_month_log1p", "is_superhost", "is_instant_bookable"
]
host_reputation_missing = (
    df_modeling_features[host_reputation_features].isna().sum().to_dict()
)
host_reputation_infinite = int(
    np.isinf(
        df_modeling_features[host_reputation_features].to_numpy(dtype=float)
    ).sum()
)
if any(host_reputation_missing.values()) or host_reputation_infinite:
    raise ValueError("As features de anfitrião/reputação geraram ausências ou infinitos.")
print("Datas de host futuras/ausentes:", future_host_dates, missing_host_dates)
print("Contagens negativas:", negative_host_portfolios, negative_review_counts, negative_review_rates)
print("Domínios booleanos:", sorted(superhost_values), sorted(instant_values))
print("Anfitriões com múltiplos anúncios:", int(df_modeling_features["host_has_multiple_listings"].sum()))
print("Superhosts:", int(df_modeling_features["is_superhost"].sum()))
print("Reserva instantânea:", int(df_modeling_features["is_instant_bookable"].sum()))
print("Ausências/infinitos:", host_reputation_missing, host_reputation_infinite)
print(
    df_modeling_features[host_reputation_features]
    .agg(["min", "median", "max"])
    .to_string()
)

Datas de host futuras/ausentes: 0 0
Contagens negativas: 0 0 0
Domínios booleanos: ['f', 't'] ['f', 't']
Anfitriões com múltiplos anúncios: 3499
Superhosts: 2626
Reserva instantânea: 3538
Ausências/infinitos: {'host_tenure_years': 0, 'host_listings_log1p': 0, 'host_has_multiple_listings': 0, 'number_of_reviews_log1p': 0, 'reviews_per_month_log1p': 0, 'is_superhost': 0, 'is_instant_bookable': 0} 0
        host_tenure_years  host_listings_log1p  host_has_multiple_listings  number_of_reviews_log1p  reviews_per_month_log1p  is_superhost  is_instant_bookable
min              0.030116             0.000000                         0.0                 0.693147                 0.009950           0.0                  0.0
median           4.145106             0.693147                         0.0                 2.484907                 0.488580           0.0                  0.0
max             10.198494             6.549651                         1.0                 6.545350                 2.55

**Leitura de 03.3.** As datas de `host_since` não ultrapassam 06/12/2018; contagens e taxas usadas nos logaritmos são não negativas. Os booleanos têm apenas `t/f`. As sete features foram criadas sem ausências ou infinitos. `host_has_multiple_listings` identifica 3.499 anúncios, sem classificá-los automaticamente como negócios profissionais.

### 03.4 — Verificação da localização e das interações

Medir no treino a cardinalidade de combinações de bairro, tipo de quarto e tipo de propriedade. Confirmar o formato e os limites do GeoJSON e escolher uma referência espacial reprodutível. Uma distância ao centro mediano dos anúncios de treino é apenas uma medida de centralidade da amostra, não uma distância ao centro oficial da cidade.

In [24]:
# 1. Importa o leitor padrão de GeoJSON e separa somente as linhas de treino.
import json
stage03_train_mask = df_modeling_features["particao"].eq("treino")
stage03_validation_mask = df_modeling_features["particao"].eq("validação")

# 2. Monta combinações candidatas sem incorporá-las ainda ao DataFrame.
room_neighbourhood_candidate = (
    df_modeling_features["room_type"].astype("string")
    + " | "
    + df_modeling_features["neighbourhood_cleansed"].astype("string")
)
property_room_candidate = (
    df_modeling_features["property_type"].astype("string")
    + " | "
    + df_modeling_features["room_type"].astype("string")
)
room_neighbourhood_counts = room_neighbourhood_candidate.loc[
    stage03_train_mask
].value_counts()
property_room_counts = property_room_candidate.loc[
    stage03_train_mask
].value_counts()
stage03_rare_threshold = 20

# 3. Lê a estrutura da fonte geográfica e identifica seus limites de coordenadas.
neighbourhoods_geojson_path = raw_dir / "neighbourhoods.geojson"
if not neighbourhoods_geojson_path.exists():
    raise FileNotFoundError(neighbourhoods_geojson_path)
with neighbourhoods_geojson_path.open(encoding="utf-8") as geojson_file:
    neighbourhoods_geojson = json.load(geojson_file)
geojson_features = neighbourhoods_geojson["features"]
geojson_geometry_types = pd.Series(
    [feature["geometry"]["type"] for feature in geojson_features]
).value_counts().to_dict()
coordinate_stack = [
    feature["geometry"]["coordinates"] for feature in geojson_features
]
geo_longitudes = []
geo_latitudes = []
while coordinate_stack:
    coordinate_item = coordinate_stack.pop()
    if (
        len(coordinate_item) >= 2
        and isinstance(coordinate_item[0], (int, float))
        and isinstance(coordinate_item[1], (int, float))
    ):
        geo_longitudes.append(coordinate_item[0])
        geo_latitudes.append(coordinate_item[1])
    else:
        coordinate_stack.extend(coordinate_item)

# 4. Estima o centro de referência apenas pela mediana das coordenadas do treino.
training_center_latitude = float(
    df_modeling_features.loc[stage03_train_mask, "latitude"].median()
)
training_center_longitude = float(
    df_modeling_features.loc[stage03_train_mask, "longitude"].median()
)
if (
    not min(geo_latitudes) <= training_center_latitude <= max(geo_latitudes)
    or not min(geo_longitudes) <= training_center_longitude <= max(geo_longitudes)
):
    raise ValueError("O centro mediano do treino está fora dos limites do GeoJSON.")

# 5. Registra somente as frequências do treino e a referência geográfica.
print("Combinações quarto × bairro no treino:", len(room_neighbourhood_counts))
print(
    "Combinações quarto × bairro com menos de 20 anúncios:",
    int(room_neighbourhood_counts.lt(stage03_rare_threshold).sum()),
    "| linhas nessas combinações:",
    int(room_neighbourhood_counts[room_neighbourhood_counts.lt(stage03_rare_threshold)].sum())
)
print("Combinações propriedade × quarto no treino:", len(property_room_counts))
print(
    "Combinações propriedade × quarto com menos de 20 anúncios:",
    int(property_room_counts.lt(stage03_rare_threshold).sum()),
    "| linhas nessas combinações:",
    int(property_room_counts[property_room_counts.lt(stage03_rare_threshold)].sum())
)
print("GeoJSON | tipo:", neighbourhoods_geojson["type"], "| geometrias:", len(geojson_features), "| tipos:", geojson_geometry_types)
print("GeoJSON | CRS declarado:", neighbourhoods_geojson.get("crs"))
print("Limites | longitude:", min(geo_longitudes), max(geo_longitudes))
print("Limites | latitude:", min(geo_latitudes), max(geo_latitudes))
print("Centro mediano do treino | latitude:", training_center_latitude, "| longitude:", training_center_longitude)

Combinações quarto × bairro no treino: 60
Combinações quarto × bairro com menos de 20 anúncios: 16 | linhas nessas combinações: 43
Combinações propriedade × quarto no treino: 53
Combinações propriedade × quarto com menos de 20 anúncios: 33 | linhas nessas combinações: 158
GeoJSON | tipo: FeatureCollection | geometrias: 22 | tipos: {'MultiPolygon': 22}
GeoJSON | CRS declarado: None
Limites | longitude: 4.754837 5.079162
Limites | latitude: 52.278174 52.430679
Centro mediano do treino | latitude: 52.36471983211245 | longitude: 4.88606668034429


**Leitura de 03.4 (auditoria).** No treino há 60 combinações de quarto e bairro; 16 têm menos de 20 anúncios e abrangem 43 linhas. Há 53 combinações de propriedade e quarto; 33 têm menos de 20 anúncios e abrangem 158 linhas. Ambas podem ser mantidas como candidatas, com categorias raras acompanhadas nas etapas de seleção e preparo. O GeoJSON tem 22 geometrias, mas não declara CRS; por isso a distância abaixo usa a mediana de latitude/longitude do treino como referência empírica, sem chamá-la de centro oficial.

In [25]:
# 1. Confere o domínio global das coordenadas antes da distância geográfica.
invalid_latitude = int(
    (~df_modeling_features["latitude"].between(-90, 90)).sum()
)
invalid_longitude = int(
    (~df_modeling_features["longitude"].between(-180, 180)).sum()
)
outside_geojson_bbox = int(
    (
        ~df_modeling_features["latitude"].between(
            min(geo_latitudes), max(geo_latitudes)
        )
        | ~df_modeling_features["longitude"].between(
            min(geo_longitudes), max(geo_longitudes)
        )
    ).sum()
)
if invalid_latitude or invalid_longitude:
    raise ValueError("Há coordenadas fora do domínio de latitude/longitude.")

# 2. Inclui as interações categóricas e de reputação como candidatas.
df_modeling_features["room_type_neighbourhood"] = (
    room_neighbourhood_candidate
)
df_modeling_features["property_type_room_type"] = (
    property_room_candidate
)
df_modeling_features["rating_x_superhost"] = (
    df_modeling_features["review_scores_rating"]
    * df_modeling_features["is_superhost"]
)

# 3. Calcula a distância geodésica aproximada ao centro mediano do treino.
earth_radius_km = 6371.0088
latitude_radians = np.radians(
    df_modeling_features["latitude"].to_numpy(dtype=float)
)
longitude_radians = np.radians(
    df_modeling_features["longitude"].to_numpy(dtype=float)
)
center_latitude_radians = np.radians(training_center_latitude)
center_longitude_radians = np.radians(training_center_longitude)
latitude_delta = latitude_radians - center_latitude_radians
longitude_delta = longitude_radians - center_longitude_radians
haversine_a = (
    np.sin(latitude_delta / 2) ** 2
    + np.cos(center_latitude_radians)
    * np.cos(latitude_radians)
    * np.sin(longitude_delta / 2) ** 2
)
df_modeling_features["distance_to_training_center_km"] = (
    2 * earth_radius_km * np.arcsin(np.sqrt(np.clip(haversine_a, 0, 1)))
)

# 4. Verifica as novas features sem usar o alvo para decidir cortes.
location_interaction_features = [
    "room_type_neighbourhood", "property_type_room_type",
    "rating_x_superhost", "distance_to_training_center_km"
]
location_missing = (
    df_modeling_features[location_interaction_features].isna().sum().to_dict()
)
location_infinite = int(
    np.isinf(
        df_modeling_features[
            ["rating_x_superhost", "distance_to_training_center_km"]
        ].to_numpy(dtype=float)
    ).sum()
)
if any(location_missing.values()) or location_infinite:
    raise ValueError("As features de localização/interação geraram valores inválidos.")
print("Latitude/longitude inválidas:", invalid_latitude, invalid_longitude)
print("Coordenadas fora da envoltória do GeoJSON:", outside_geojson_bbox)
print("Ausências/infinitos:", location_missing, location_infinite)
print(
    "Distância ao centro mediano do treino | mínimo, mediana, máximo (km):",
    df_modeling_features.loc[
        stage03_train_mask, "distance_to_training_center_km"
    ].agg(["min", "median", "max"]).to_dict()
)
print(
    "Categorias no treino:",
    df_modeling_features.loc[
        stage03_train_mask, "room_type_neighbourhood"
    ].nunique(),
    df_modeling_features.loc[
        stage03_train_mask, "property_type_room_type"
    ].nunique()
)
display(
    df_modeling_features[
        ["neighbourhood_cleansed", "room_type", "property_type"]
        + location_interaction_features
    ].head(5)
)

Latitude/longitude inválidas: 0 0
Coordenadas fora da envoltória do GeoJSON: 0
Ausências/infinitos: {'room_type_neighbourhood': 0, 'property_type_room_type': 0, 'rating_x_superhost': 0, 'distance_to_training_center_km': 0} 0
Distância ao centro mediano do treino | mínimo, mediana, máximo (km): {'min': 0.02811529777663727, 'median': 2.1689507871733866, 'max': 11.162968461897055}
Categorias no treino: 60 53


,neighbourhood_cleansed,room_type,property_type,room_type_neighbourhood,property_type_room_type,rating_x_superhost,distance_to_training_center_km
0,Oostelijk Havengebied - Indische Buurt,Private room,Apartment,Private room | Oostelijk Havengebied - Indisch...,Apartment | Private room,97.0,3.760120
1,Westerpark,Entire home/apt,Apartment,Entire home/apt | Westerpark,Apartment | Entire home/apt,0.0,2.953400
2,Centrum-Oost,Entire home/apt,Townhouse,Entire home/apt | Centrum-Oost,Townhouse | Entire home/apt,0.0,0.509140
3,Bos en Lommer,Private room,Apartment,Private room | Bos en Lommer,Apartment | Private room,0.0,2.283019
4,Centrum-West,Private room,Apartment,Private room | Centrum-West,Apartment | Private room,95.0,0.498371


**Leitura de 03.4.** As quatro features de localização/interação não geraram ausências ou infinitos; nenhuma coordenada ficou fora da envoltória do GeoJSON. `distance_to_training_center_km` usa um centro calculado apenas com o treino (52,364720; 4,886067) e é uma aproximação de centralidade. As interações categóricas seguem como candidatas; sua utilidade preditiva e o tratamento de categorias raras serão avaliados mais tarde.

### 03.5 — Verificação das restrições de estadia e disponibilidade

Aplicar `log1p` a mínimos e máximos de noites depois de validar valores não negativos. Criar indicadores para limites superiores a 30/365 noites, para o valor frequente 1.125 em `maximum_nights` e para disponibilidade baixa ou integral. Esses indicadores descrevem regras e disponibilidade anunciada, sem medir noites vendidas.

In [26]:
# 1. Confere os domínios das regras de estadia e da disponibilidade.
negative_minimum_nights = int(df_modeling_features["minimum_nights"].lt(0).sum())
negative_maximum_nights = int(df_modeling_features["maximum_nights"].lt(0).sum())
minimum_above_maximum = int(
    df_modeling_features["minimum_nights"].gt(
        df_modeling_features["maximum_nights"]
    ).sum()
)
invalid_availability = int(
    (~df_modeling_features["availability_365"].between(0, 365)).sum()
)
if (
    negative_minimum_nights or negative_maximum_nights
    or minimum_above_maximum or invalid_availability
):
    raise ValueError("Valores inválidos em noites mínimas/máximas ou disponibilidade.")

# 2. Mantém os valores originais e adiciona transformações logarítmicas.
df_modeling_features["minimum_nights_log1p"] = np.log1p(
    df_modeling_features["minimum_nights"]
)
df_modeling_features["maximum_nights_log1p"] = np.log1p(
    df_modeling_features["maximum_nights"]
)

# 3. Cria flags de restrição de estadia, sem classificar 1.125 como default.
df_modeling_features["minimum_nights_over_30"] = (
    df_modeling_features["minimum_nights"].gt(30).astype("int8")
)
df_modeling_features["maximum_nights_over_365"] = (
    df_modeling_features["maximum_nights"].gt(365).astype("int8")
)
df_modeling_features["maximum_nights_equals_1125"] = (
    df_modeling_features["maximum_nights"].eq(1125).astype("int8")
)

# 4. Cria flags de disponibilidade, sem inferir ocupação ou reservas.
df_modeling_features["availability_full_year"] = (
    df_modeling_features["availability_365"].eq(365).astype("int8")
)
df_modeling_features["availability_low"] = (
    df_modeling_features["availability_365"].le(30).astype("int8")
)

# 5. Confere integridade e frequências dos indicadores no desenvolvimento.
stay_availability_features = [
    "minimum_nights_log1p", "maximum_nights_log1p",
    "minimum_nights_over_30", "maximum_nights_over_365",
    "maximum_nights_equals_1125", "availability_full_year",
    "availability_low"
]
stay_missing = (
    df_modeling_features[stay_availability_features].isna().sum().to_dict()
)
stay_infinite = int(
    np.isinf(
        df_modeling_features[stay_availability_features].to_numpy(dtype=float)
    ).sum()
)
if any(stay_missing.values()) or stay_infinite:
    raise ValueError("As features de estadia/disponibilidade geraram valores inválidos.")
print("Regras inválidas:", negative_minimum_nights, negative_maximum_nights, minimum_above_maximum)
print("Disponibilidade inválida:", invalid_availability)
print("Ausências/infinitos:", stay_missing, stay_infinite)
print(
    "Contagem das flags:",
    df_modeling_features[
        stay_availability_features[2:]
    ].sum().astype(int).to_dict()
)
print(
    df_modeling_features[
        ["minimum_nights_log1p", "maximum_nights_log1p"]
    ].agg(["min", "median", "max"]).to_string()
)

Regras inválidas: 0 0 0
Disponibilidade inválida: 0
Ausências/infinitos: {'minimum_nights_log1p': 0, 'maximum_nights_log1p': 0, 'minimum_nights_over_30': 0, 'maximum_nights_over_365': 0, 'maximum_nights_equals_1125': 0, 'availability_full_year': 0, 'availability_low': 0} 0
Contagem das flags: {'minimum_nights_over_30': 47, 'maximum_nights_over_365': 7845, 'maximum_nights_equals_1125': 7398, 'availability_full_year': 40, 'availability_low': 9884}
        minimum_nights_log1p  maximum_nights_log1p
min                 0.693147              0.693147
median              1.098612              7.025983
max                 6.909753              9.210340


**Leitura de 03.5.** As sete features de estadia/disponibilidade foram criadas sem regras inválidas, ausências ou infinitos. Na base de desenvolvimento, 7.398 anúncios têm `maximum_nights=1125`; a flag descreve essa configuração frequente sem tratá-la como erro ou default comprovado. Os 9.884 casos de `availability_low` indicam até 30 noites disponíveis, sem informar quantas foram reservadas.

### 03.6 — Verificação de redundância, dicionário e integridade

Medir associações entre preditores somente no treino e sem consultar `price`. A correlação serve para identificar redundância e orientar a comparação futura; não será usada aqui para declarar ganho de MAPE ou excluir automaticamente uma feature.

In [27]:
# 1. Seleciona apenas preditores numéricos do treino, excluindo o alvo.
numeric_train_features = (
    df_modeling_features.loc[stage03_train_mask]
    .select_dtypes(include="number")
    .drop(columns=["price"], errors="ignore")
)
spearman_matrix = numeric_train_features.corr(method="spearman")
upper_triangle = np.triu(
    np.ones(spearman_matrix.shape, dtype=bool), k=1
)
top_correlations = (
    spearman_matrix
    .where(upper_triangle)
    .stack()
    .reset_index()
)
top_correlations.columns = [
    "feature_a", "feature_b", "spearman"
]
top_correlations["abs_spearman"] = (
    top_correlations["spearman"].abs()
)
top_correlations = top_correlations.sort_values(
    "abs_spearman", ascending=False
)

# 2. Mede associações entre categorias originais com V de Cramér no treino.
categorical_pairs = [
    ("room_type", "property_type"),
    ("room_type", "neighbourhood_cleansed"),
    ("property_type", "neighbourhood_cleansed")
]
categorical_associations = []
for feature_a, feature_b in categorical_pairs:
    contingency = pd.crosstab(
        df_modeling_features.loc[stage03_train_mask, feature_a],
        df_modeling_features.loc[stage03_train_mask, feature_b]
    )
    observed = contingency.to_numpy(dtype=float)
    expected = np.outer(
        observed.sum(axis=1), observed.sum(axis=0)
    ) / observed.sum()
    chi_square = np.where(
        expected > 0,
        (observed - expected) ** 2 / expected,
        0
    ).sum()
    denominator = observed.sum() * min(
        observed.shape[0] - 1, observed.shape[1] - 1
    )
    cramers_v = np.sqrt(
        chi_square / denominator
    ) if denominator > 0 else np.nan
    categorical_associations.append({
        "feature_a": feature_a,
        "feature_b": feature_b,
        "cramers_v": cramers_v,
        "categories_a": observed.shape[0],
        "categories_b": observed.shape[1]
    })
categorical_associations = pd.DataFrame(
    categorical_associations
)

# 3. Exibe pares mais associados sem usar a validação ou o preço.
print("Preditores numéricos analisados no treino:", numeric_train_features.shape[1])
print(
    "Pares numéricos com |Spearman| >= 0,90:",
    int(top_correlations["abs_spearman"].ge(0.90).sum())
)
print("Maiores correlações numéricas no treino:")
display(top_correlations.head(15))
print("Associações entre categóricas originais no treino:")
display(categorical_associations)

Preditores numéricos analisados no treino: 34
Pares numéricos com |Spearman| >= 0,90: 14
Maiores correlações numéricas no treino:


,feature_a,feature_b,spearman,abs_spearman
16,accommodates,accommodates_log1p,1.000000,1.000000
224,reviews_per_month,reviews_per_month_log1p,1.000000,1.000000
278,minimum_nights,minimum_nights_log1p,1.000000,1.000000
168,host_listings_count,host_listings_log1p,1.000000,1.000000
302,maximum_nights,maximum_nights_log1p,1.000000,1.000000
197,number_of_reviews,number_of_reviews_log1p,1.000000,1.000000
507,is_superhost,rating_x_superhost,0.993921,0.993921
456,host_listings_log1p,host_has_multiple_listings,0.988371,0.988371
169,host_listings_count,host_has_multiple_listings,0.988371,0.988371
555,maximum_nights_over_365,maximum_nights_equals_1125,0.941937,0.941937


Associações entre categóricas originais no treino:


,feature_a,feature_b,cramers_v,categories_a,categories_b
0,room_type,property_type,0.252459,3,27
1,room_type,neighbourhood_cleansed,0.151033,3,22
2,property_type,neighbourhood_cleansed,0.132302,27,22


**Leitura inicial de 03.6.** Seis pares têm Spearman igual a 1 porque `log1p` preserva a ordenação dos valores originais. `rating_x_superhost` e as flags de `maximum_nights` também mostram alta associação. Para examinar dependência linear, calcular Pearson no treino; a decisão de manter combinações redundantes para cada algoritmo ficará para seleção e validação posteriores.

In [28]:
# 1. Calcula correlação linear somente entre preditores numéricos do treino.
pearson_matrix = numeric_train_features.corr(method="pearson")
pearson_pairs = (
    pearson_matrix
    .where(np.triu(np.ones(pearson_matrix.shape, dtype=bool), k=1))
    .stack()
    .reset_index()
)
pearson_pairs.columns = ["feature_a", "feature_b", "pearson"]
pearson_pairs["abs_pearson"] = pearson_pairs["pearson"].abs()
pearson_pairs = pearson_pairs.sort_values(
    "abs_pearson", ascending=False
)

# 2. Mostra dependências lineares fortes para orientar o dicionário.
print(
    "Pares numéricos com |Pearson| >= 0,90:",
    int(pearson_pairs["abs_pearson"].ge(0.90).sum())
)
print("Maiores correlações lineares no treino:")
display(pearson_pairs.head(15))

Pares numéricos com |Pearson| >= 0,90: 8
Maiores correlações lineares no treino:


,feature_a,feature_b,pearson,abs_pearson
507,is_superhost,rating_x_superhost,0.999638,0.999638
304,maximum_nights,maximum_nights_over_365,0.980041,0.980041
16,accommodates,accommodates_log1p,0.964585,0.964585
224,reviews_per_month,reviews_per_month_log1p,0.950472,0.950472
302,maximum_nights,maximum_nights_log1p,0.947084,0.947084
555,maximum_nights_over_365,maximum_nights_equals_1125,0.941937,0.941937
305,maximum_nights,maximum_nights_equals_1125,0.922716,0.922716
547,maximum_nights_log1p,maximum_nights_over_365,0.920506,0.920506
548,maximum_nights_log1p,maximum_nights_equals_1125,0.867550,0.867550
214,reviews_per_month,reviews_last_365d,0.829812,0.829812


#### 03.6.1 — Leitura da redundância e dicionário de features (parte 1)

No treino, 8 pares numéricos têm |Pearson| ≥ 0,90. O maior é `is_superhost` × `rating_x_superhost` (0,9996). Correlação alta entre preditores pode prejudicar a interpretação de modelos lineares; nenhuma coluna foi descartada por esse diagnóstico isolado. Todas as decisões abaixo significam **manter como candidata em `df_modeling_features`**, com comparação posterior usando validação. As duas métricas agregadas de `reviews.csv` já eram colunas da Etapa 02 e não foram recalculadas.

| Feature | Tipo | Origem | Regra | Hipótese de negócio | Limitação | Decisão |
|---|---|---|---|---|---|---|
| `is_studio` | int8 | `bedrooms` | 1 se quartos = 0 | Falta de quarto separado pode alterar preço. | Zero quarto não comprova imóvel do tipo estúdio. | Candidata |
| `bedrooms_per_guest` | float | `bedrooms`, `accommodates` | quartos / capacidade positiva | Mais quartos por hóspede podem sinalizar conforto. | Relação estrutural com capacidade e quartos. | Candidata |
| `bathrooms_per_guest` | float | `bathrooms`, `accommodates` | banheiros / capacidade positiva | Mais banheiros por hóspede podem agregar valor. | Zero banheiro foi mantido como valor observado. | Candidata |
| `accommodates_log1p` | float | `accommodates` | ln(1 + capacidade) | Permite efeito não linear da lotação. | Pearson 0,9646 com a coluna original. | Comparar depois |
| `host_tenure_years` | float | `host_since` | dias até 06/12/2018 / 365,25 | Experiência do anfitrião pode afetar o posicionamento. | Data fixa do snapshot; não mede qualidade causal. | Candidata |
| `host_listings_log1p` | float | `host_listings_count` | ln(1 + anúncios) | Atenua a cauda de portfólios grandes. | Mesma informação ordenada da contagem original. | Comparar depois |
| `host_has_multiple_listings` | int8 | `host_listings_count` | 1 se anúncios > 1 | Diferencia anfitriões com portfólio maior. | Não comprova operação profissional. | Comparar depois |
| `number_of_reviews_log1p` | float | `number_of_reviews` | ln(1 + total de avaliações) | Resume volume histórico de reputação. | Total não equivale a reservas; 91 divergências com a fonte de eventos. | Comparar depois |
| `reviews_per_month_log1p` | float | `reviews_per_month` | ln(1 + avaliações/mês) | Atenua cauda do ritmo observado de avaliações. | Não representa ocupação; Pearson 0,9505 com original. | Comparar depois |
| `is_superhost` | int8 | `host_is_superhost` | `t` → 1; `f` → 0 | Status pode capturar confiança percebida. | Forte sobreposição com a interação de nota. | Candidata |
| `is_instant_bookable` | int8 | `instant_bookable` | `t` → 1; `f` → 0 | Reserva imediata pode influenciar preço. | É uma configuração, não medida de conversão. | Candidata |

#### 03.6.2 — Dicionário de features (parte 2)

| Feature | Tipo | Origem | Regra | Hipótese de negócio | Limitação | Decisão |
|---|---|---|---|---|---|---|
| `room_type_neighbourhood` | string | `room_type`, `neighbourhood_cleansed` | concatenação quarto + bairro | Um quarto pode ter posição de preço diferente por bairro. | 16 de 60 combinações têm <20 anúncios no treino. | Candidata |
| `property_type_room_type` | string | `property_type`, `room_type` | concatenação propriedade + quarto | A configuração conjunta pode diferenciar imóveis. | 33 de 53 combinações têm <20 anúncios no treino. | Comparar depois |
| `rating_x_superhost` | float | `review_scores_rating`, `is_superhost` | nota × indicador | Permite efeito distinto da nota para superhosts. | Pearson 0,9996 com `is_superhost`; alto risco de redundância linear. | Comparar depois |
| `distance_to_training_center_km` | float | `latitude`, `longitude`; treino | Haversine até a mediana das coordenadas do treino | Centralidade aproximada pode afetar preço. | Referência empírica, não centro oficial; guardar as duas medianas. | Candidata |
| `minimum_nights_log1p` | float | `minimum_nights` | ln(1 + mínimo de noites) | Atenua restrições muito altas. | Mesma ordenação da coluna original. | Comparar depois |
| `maximum_nights_log1p` | float | `maximum_nights` | ln(1 + máximo de noites) | Atenua a massa de limites altos. | Pearson 0,9471 com original; valores altos mantidos. | Comparar depois |
| `minimum_nights_over_30` | int8 | `minimum_nights` | 1 se mínimo > 30 | Identifica estadias mínimas longas. | Corte analítico, não classificação legal. | Candidata |
| `maximum_nights_over_365` | int8 | `maximum_nights` | 1 se máximo > 365 | Distingue limites superiores a um ano. | Pearson 0,9800 com máximo original. | Comparar depois |
| `maximum_nights_equals_1125` | int8 | `maximum_nights` | 1 se máximo = 1.125 | Isola a configuração frequente. | Não foi comprovado que 1.125 seja default. | Comparar depois |
| `availability_full_year` | int8 | `availability_365` | 1 se disponibilidade = 365 | Distingue calendário completamente aberto. | Não mede ocupação ou noites vendidas. | Candidata |
| `availability_low` | int8 | `availability_365` | 1 se disponibilidade ≤ 30 | Distingue calendário pouco aberto. | Corte operacional; indisponível não significa reservado. | Candidata |

**Regra geral:** `id`, `price` e `particao` não são preditores. As transformações são determinísticas, exceto o centro espacial, estimado somente no treino. Codificação de categorias e eventual descarte por desempenho ficam para as etapas previstas no plano.

#### 03.6.3 — Verificação final da base de features

Confirmar que as 22 novas colunas correspondem ao dicionário, que as 23 colunas originais continuam intactas, que IDs e partições não mudaram e que o teste bruto não foi preparado. Criar a lista explícita de preditores sem `id`, `price` ou `particao`.

In [29]:
# 1. Lista explicitamente as 22 colunas criadas nesta etapa.
stage03_engineered_features = [
    "is_studio", "bedrooms_per_guest", "bathrooms_per_guest",
    "accommodates_log1p", "host_tenure_years", "host_listings_log1p",
    "host_has_multiple_listings", "number_of_reviews_log1p",
    "reviews_per_month_log1p", "is_superhost", "is_instant_bookable",
    "room_type_neighbourhood", "property_type_room_type",
    "rating_x_superhost", "distance_to_training_center_km",
    "minimum_nights_log1p", "maximum_nights_log1p",
    "minimum_nights_over_30", "maximum_nights_over_365",
    "maximum_nights_equals_1125", "availability_full_year",
    "availability_low"
]
stage03_actual_new_columns = [
    column for column in df_modeling_features.columns
    if column not in expected_stage03_columns
]
stage03_predictor_columns = (
    stage03_source_features + stage03_engineered_features
)

# 2. Confere se a fonte, os IDs e as partições foram preservados.
stage03_original_unchanged = df_modeling_final.equals(
    df_modeling_stage02_reference
)
stage03_source_columns_unchanged = (
    df_modeling_features[expected_stage03_columns].equals(
        df_modeling_final
    )
)
stage03_ids_preserved = (
    df_modeling_features["id"].tolist()
    == df_modeling_final["id"].tolist()
)
stage03_partitions_preserved = (
    df_modeling_features["particao"].value_counts().to_dict()
    == stage03_partition_counts
)

# 3. Audita ausências e infinitos sem usar o alvo para selecionar features.
stage03_new_missing = int(
    df_modeling_features[stage03_engineered_features].isna().sum().sum()
)
stage03_numeric_new = (
    df_modeling_features[stage03_engineered_features]
    .select_dtypes(include="number")
)
stage03_new_infinite = int(
    np.isinf(stage03_numeric_new.to_numpy(dtype=float)).sum()
)
stage03_forbidden_predictors = sorted(
    set(stage03_predictor_columns) & {"id", "price", "particao"}
)

# 4. Confere somente os metadados do teste reservado.
stage03_test_metadata_preserved = (
    df_test.shape == (3005, 96)
    and df_test.columns.tolist() == df_test_raw_columns
    and all(str(dtype) == "string" for dtype in df_test.dtypes)
    and df_test_raw_path.exists()
    and df_test_raw_path.stat().st_size == 11634013
)

# 5. Interrompe se o contrato da Etapa 03 não for cumprido.
if (
    df_modeling_features.shape != (14806, 45)
    or stage03_actual_new_columns != stage03_engineered_features
    or not stage03_original_unchanged
    or not stage03_source_columns_unchanged
    or not stage03_ids_preserved
    or not stage03_partitions_preserved
    or stage03_new_missing
    or stage03_new_infinite
    or stage03_forbidden_predictors
    or not stage03_test_metadata_preserved
):
    raise ValueError("A base de features falhou na verificação final.")

# 6. Resume o contrato da saída sem exportar a base ou avaliar modelo.
print("Base de features em memória:", df_modeling_features.shape)
print("Features originais:", len(stage03_source_features))
print("Features criadas:", len(stage03_engineered_features))
print("Preditores disponíveis:", len(stage03_predictor_columns))
print("Nomes das novas features:", stage03_actual_new_columns)
print("Entrada da Etapa 02 inalterada:", stage03_original_unchanged)
print("Colunas originais/IDs/partições preservados:", stage03_source_columns_unchanged, stage03_ids_preserved, stage03_partitions_preserved)
print("Ausências/infinitos novos:", stage03_new_missing, stage03_new_infinite)
print("Preditores proibidos:", stage03_forbidden_predictors)
print("Metadados do teste bruto preservados:", stage03_test_metadata_preserved)
print("Partições na saída:", df_modeling_features["particao"].value_counts().to_dict())

Base de features em memória: (14806, 45)
Features originais: 20
Features criadas: 22
Preditores disponíveis: 42
Nomes das novas features: ['is_studio', 'bedrooms_per_guest', 'bathrooms_per_guest', 'accommodates_log1p', 'host_tenure_years', 'host_listings_log1p', 'host_has_multiple_listings', 'number_of_reviews_log1p', 'reviews_per_month_log1p', 'is_superhost', 'is_instant_bookable', 'room_type_neighbourhood', 'property_type_room_type', 'rating_x_superhost', 'distance_to_training_center_km', 'minimum_nights_log1p', 'maximum_nights_log1p', 'minimum_nights_over_30', 'maximum_nights_over_365', 'maximum_nights_equals_1125', 'availability_full_year', 'availability_low']
Entrada da Etapa 02 inalterada: True
Colunas originais/IDs/partições preservados: True True True
Ausências/infinitos novos: 0 0
Preditores proibidos: []
Metadados do teste bruto preservados: True
Partições na saída: {'treino': 12187, 'validação': 2619}


### 03.6.4 — Resultado e gate da Etapa 03

#### Resultado

- `df_modeling_features` foi criado **somente em memória**, com **14.806 linhas e 45 colunas**: 20 preditoras de entrada, 22 features criadas nesta etapa, `id`, `price` e `particao`. São **12.187 anúncios de treino** e **2.619 de validação**.
- As 22 novas colunas cobrem capacidade, anfitrião, reputação, localização, restrições de estadia e disponibilidade. O dicionário acima registra fórmula, tipo, origem, hipótese, limitação e decisão de cada uma.
- `df_modeling_final` e suas 23 colunas permanecem intactos. IDs, ordem, colunas originais e partições foram preservados na saída. As novas features têm **0 ausências e 0 infinitos**; `id`, `price` e `particao` não entram na lista de preditores.
- A distância espacial usa Haversine e o centro mediano calculado **somente no treino** (latitude 52,3647198321; longitude 4,8860666803). É uma referência empírica da amostra, não o centro oficial de Amsterdam. O GeoJSON contém 22 geometrias e não declara CRS; foi usado apenas para conferir a faixa das coordenadas, sem calcular o centro.
- No treino, há 60 combinações `room_type_neighbourhood` e 53 `property_type_room_type`; respectivamente 16 e 33 categorias têm menos de 20 anúncios. Categorias raras exigirão tratamento no preparo e comparação posterior.
- A auditoria sem alvo encontrou **14 pares com |Spearman| ≥ 0,90** e **8 pares com |Pearson| ≥ 0,90**. O maior Pearson foi 0,9996 entre `is_superhost` e `rating_x_superhost`. Logs, flags e originais foram mantidos como alternativas para comparação futura; nenhuma melhora de MAPE foi afirmada.
- O teste bruto continua reservado: metadados de `df_test` e de `data/processed/df_test_raw.csv` preservados. Nenhum preço do calendário, receita derivada do alvo, evento posterior ao snapshot ou valor de negócio do teste foi usado nesta etapa.

#### Decisões e limitações

- `is_studio` significa `bedrooms=0`, sem comprovar que o anúncio seja um estúdio. `availability_365` e suas flags descrevem disponibilidade, sem medir ocupação ou reservas.
- `maximum_nights=1125` foi mantido como configuração observada, sem classificá-lo como default. Flags com cortes 30/365 são hipóteses analíticas.
- As métricas de `reviews.csv` já integradas na Etapa 02 foram preservadas. As 91 divergências de contagem permanecem documentadas; a Etapa 03 não reconciliou nem substituiu `number_of_reviews`.
- Transformações, interações e codificações com correlação alta ou categorias raras serão comparadas depois na validação. A seleção de features, os encoders e o treinamento pertencem às etapas seguintes.

#### Estado do projeto

- **Status:** Etapa 03 implementada e verificada no notebook; gate aguarda aprovação.
- **Última etapa aprovada:** Etapa 02.
- **Decisões:** 22 novas features mantidas como candidatas em `df_modeling_features`; referência espacial ajustada apenas no treino; teste bruto reservado.
- **Pendências:** aprovação do dicionário e das regras contra vazamento; nenhuma avaliação de MAPE foi realizada.

**Você aprova a Etapa 03 para avançarmos à Etapa 04?**